# SAM3D Objects -- LiDAR-Guided Metric Reconstruction

**Full pipeline** for per-instance 3D car reconstruction:

1. **Data loading** — nuScenes calibration, camera image, nearest LiDAR sweep
2. **In-mask LiDAR filtering** — depth-window + MAD robust filter isolates car-surface points
3. **MoGe depth inference** — monocular relative-depth pointmap
4. **Affine calibration** — global (scene-wide) and local (per-object, MAD-robust) `Z_lidar = a * Z_moge + b`
5. **SAM3D inference (Option C)** — metric pointmap fed directly into SAM3D, so the model sees metric depth
6. **Visibility-Aware Refinement (VAR)** — post-hoc correction using only z-buffer-visible mesh surface
   - VAR1: front-depth + XY translation
   - **VAR2 (default)**: VAR1 + yaw correction
   - VAR3 (diagnostic): VAR2 + clamped scale
   - VAR4 (**experimental**): expanded LiDAR cluster → ego-facing OBB face alignment
7. **OBB evaluation** — gravity-aligned oriented bounding boxes compared across strategies
8. **Diagnostics** — Plotly 3D LiDAR scene, BEV footprints, metrics table

> **Appendix** (at the end) contains the legacy hybrid post-hoc anchor as an optional baseline.


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
import numpy as np
import cv2
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from scipy.spatial import cKDTree # For efficient nearest neighbor search

# Set DPI (dots per inch) for better resolution in plots
matplotlib.rcParams['figure.dpi'] = 120

# Enable inline plotting in Jupyter notebooks
%matplotlib inline

print('Imports OK')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Step 1: set FRAME_IDX and OBJ_CLASS, run cells up through §3 to see detections.
# Step 2: set OBJ_IDXS, then run the remaining cells.

DATA_ROOT          = '/media/lleba/ECP_Nuscenes_01/output2/ecp2nuscenes'
SAM3D_REPO         = Path('/home/lleba/Thesis/Development/SAM3D/sam-3d-objects')
CKPT_CONFIG        = SAM3D_REPO / 'checkpoints' / 'hf' / 'pipeline.yaml'

NUSCENES_SCENE_IDX = 10
FRAME_IDX          = 580

# Object class to reconstruct — choose one:
#   'car'  'motorcycle'  'bus'  'construction_vehicle'
#   'trailer'  'truck'  'bicycle and cyclist'
OBJ_CLASS = 'car'

OBJ_IDXS  = 'all'   #   use a list  [0, 1, 2]  or 'all' for every detection
                   
SAM3_MASK_DIR      = Path('/media/lleba/ECP_Nuscenes_01/SAM3_Visualizations/Strassbourg')
SAM3_SCENE_PREFIX  = f'scene_{NUSCENES_SCENE_IDX}_cam_0_sam3_outputs__'

MIN_PROB     = 0.6      # Minimum SAM3 mask probability to consider for depth estimation
MIN_MASK_PX  = 100      # Minimum SAM3 mask size in pixels to attempt depth estimation

# Minimum in-mask LiDAR hits to trust the depth anchor
MIN_LIDAR_HITS = 3

# Per-class maximum plausible depth span in meters (depth-window filter)
CLASS_DEPTH_SPAN = {
    'car':                  3.0,
    'motorcycle':           2.5,
    'bus':                 13.0,
    'construction_vehicle': 8.0,
    'trailer':             13.0,
    'truck':                8.0,
    'bicycle and cyclist':  2.0,
}
MAX_DEPTH_SPAN = CLASS_DEPTH_SPAN[OBJ_CLASS]

# Robust MAD (median absolute deviation) outlier filter (applied after depth-window filter, before local affine fit)
MAD_SIGMA_THRESH = 3.0   # reject points whose |residual| > MAD_SIGMA_THRESH * sigma_MAD
MIN_MOGE_SPAN   = 0.05   # min MoGe depth range (normalised) to attempt fit
MIN_LIDAR_SPAN  = 0.30   # min LiDAR depth span (m) to attempt fit

# Per-class prior dimensions [W, H, L] in meters
CLASS_PRIOR_DIMS = {
    'car':                  np.array([1.9, 1.5,  4.5]),
    'motorcycle':           np.array([0.8, 1.5,  2.2]),
    'bus':                  np.array([2.9, 3.5, 12.0]),
    'construction_vehicle': np.array([2.8, 3.0,  6.0]),
    'trailer':              np.array([2.9, 3.5, 12.0]),
    'truck':                np.array([2.5, 3.0,  7.0]),
    'bicycle and cyclist':  np.array([0.6, 1.7,  1.8]),
}
OBJ_PRIOR_DIMS = CLASS_PRIOR_DIMS[OBJ_CLASS]

# Config dict passed to process_object
config = dict(
    NUSCENES_SCENE_IDX = NUSCENES_SCENE_IDX,
    FRAME_IDX          = FRAME_IDX,
    OBJ_CLASS          = OBJ_CLASS,
    MIN_LIDAR_HITS     = MIN_LIDAR_HITS,
    MAX_DEPTH_SPAN     = MAX_DEPTH_SPAN,
    MAD_SIGMA_THRESH   = MAD_SIGMA_THRESH,
    MIN_MOGE_SPAN      = MIN_MOGE_SPAN,
    MIN_LIDAR_SPAN     = MIN_LIDAR_SPAN,
    OBJ_PRIOR_DIMS     = OBJ_PRIOR_DIMS,
)

print(f'Config: scene={NUSCENES_SCENE_IDX}  frame={FRAME_IDX}  '
      f'class={OBJ_CLASS}  obj_idxs={OBJ_IDXS}')


## Setup: SAM3D model + helper functions

Load the SAM3D-Objects model once per session. Defines `mesh_to_r3`, `project`, `draw_mesh_on_image`, and `compute_metrics`.
All downstream sections depend on these.


In [ ]:
# Load SAM3D-Objects model (run this cell once per notebook session)
import torch
sys.path.insert(0, str(SAM3D_REPO / 'notebook'))
from inference import Inference

inference_model = Inference(str(CKPT_CONFIG), compile=False)
print('SAM3D-Objects model ready.')

In [ ]:
def mesh_to_r3(output):
    """object-local → R3 (OpenCV) camera space. Returns (verts (V,3), faces (F,3))."""
    from pytorch3d.renderer import look_at_view_transform
    from pytorch3d.transforms import Transform3d, quaternion_to_matrix
    from sam3d_objects.data.dataset.tdfy.transforms_3d import compose_transform

    def _sq(t):
        while t.dim() > 2: t = t.squeeze(0)
        return t

    quat  = _sq(output['rotation'].detach().cpu().float())
    trans = _sq(output['translation'].detach().cpu().float())
    scale = _sq(output['scale'].detach().cpu().float())
    if quat.dim() == 1:
        quat = quat.unsqueeze(0)

    l2c = compose_transform(
        scale=scale, rotation=quaternion_to_matrix(quat), translation=trans,
    )
    R_r3_to_p3d, _ = look_at_view_transform(
        eye=torch.tensor([[0., 0., -1.]]),
        at=torch.tensor([[0., 0.,  0.]]),
        up=torch.tensor([[0., -1., 0.]]),
    )
    t_p3d_to_r3 = Transform3d().rotate(R_r3_to_p3d).inverse()

    mesh_res    = output['mesh'][0]
    verts_local = mesh_res.vertices.float().cpu()
    faces       = mesh_res.faces.long().cpu().numpy().astype(np.int32)
    xyz_p3d     = l2c.transform_points(verts_local.unsqueeze(0)).squeeze(0)
    verts_r3    = (
        t_p3d_to_r3.transform_points(xyz_p3d.unsqueeze(0))
        .squeeze(0).numpy().astype(np.float32)
    )
    return verts_r3, faces


def project(pts, K):
    """(N,3) R3 → (N,2) pixel. Points behind camera → (-9999, -9999)."""
    valid = pts[:, 2] > 1e-6
    u = np.where(valid, K[0,0]*pts[:,0]/np.maximum(pts[:,2],1e-6)+K[0,2], -9999.)
    v = np.where(valid, K[1,1]*pts[:,1]/np.maximum(pts[:,2],1e-6)+K[1,2], -9999.)
    return np.stack([u, v], axis=1)


def draw_mesh_on_image(base_rgb, verts_r3, K, color=(255, 100, 0)):
    """Project mesh vertices onto image as colored dots."""
    out = base_rgb.copy()
    pts2d = project(verts_r3, K)
    u = np.round(pts2d[:, 0]).astype(int)
    v = np.round(pts2d[:, 1]).astype(int)
    valid = (u >= 0) & (u < out.shape[1]) & (v >= 0) & (v < out.shape[0])
    out[v[valid], u[valid]] = color
    return out


print('Helper functions defined: mesh_to_r3, project, draw_mesh_on_image')

In [ ]:
# ── Metrics helper ──────────────────────────────────────────────────────────
def compute_metrics(verts_r3, binary_mask, K, Z_lidar_front, label=''):
    Hm, Wm = binary_mask.shape
    fx_k = float(K[0,0]); fy_k = float(K[1,1])
    cx_k = float(K[0,2]); cy_k = float(K[1,2])

    # Project all vertices
    valid = verts_r3[:,2] > 1e-6
    u_p = np.where(valid, fx_k*verts_r3[:,0]/np.maximum(verts_r3[:,2],1e-6)+cx_k, -9999.)
    v_p = np.where(valid, fy_k*verts_r3[:,1]/np.maximum(verts_r3[:,2],1e-6)+cy_k, -9999.)

    valid_proj = valid & (u_p > -9000)
    v2d = np.stack([u_p[valid_proj], v_p[valid_proj]], axis=1)

    if len(v2d) == 0:
        return {'label':label,'iou':0.,'coverage':0.,'Z_front':float('nan'),
                'Z_front_err':float('nan'),'dims':np.zeros(3),'ctr':np.zeros(3),
                'ctr_ego':np.zeros(3),'dim_err':float('nan'),'n_in_mask':0}

    bx1,by1 = float(v2d[:,0].min()), float(v2d[:,1].min())
    bx2,by2 = float(v2d[:,0].max()), float(v2d[:,1].max())

    ys_m2, xs_m2 = np.where(binary_mask)
    mx1,my1 = float(xs_m2.min()), float(ys_m2.min())
    mx2,my2 = float(xs_m2.max()), float(ys_m2.max())

    ix1,iy1 = max(bx1,mx1), max(by1,my1)
    ix2,iy2 = min(bx2,mx2), min(by2,my2)
    inter = max(0.,ix2-ix1)*max(0.,iy2-iy1)
    a_mesh = (bx2-bx1)*(by2-by1); a_mask = (mx2-mx1)*(my2-my1)
    iou = inter/(a_mesh+a_mask-inter) if (a_mesh+a_mask-inter)>0 else 0.

    u_vi = np.round(u_p).astype(int); v_vi = np.round(v_p).astype(int)
    in_img2 = valid & (u_vi>=0)&(u_vi<Wm)&(v_vi>=0)&(v_vi<Hm)
    in_msk  = np.zeros(len(verts_r3), dtype=bool)
    in_msk[in_img2] = binary_mask[v_vi[in_img2], u_vi[in_img2]]
    n_in_mask = int(in_msk.sum())
    coverage  = n_in_mask / max(1, int(valid_proj.sum()))

    ctr   = verts_r3.mean(axis=0)
    dims  = verts_r3.max(axis=0) - verts_r3.min(axis=0)
    Z_front = float(verts_r3[:,2].min())
    Z_front_err = Z_front - Z_lidar_front if Z_lidar_front is not None else float('nan')
    dim_err = float(np.abs(dims - OBJ_PRIOR_DIMS).mean())
    ctr_ego = R_cam2ego @ ctr + t_cam2ego

    return {'label':label,'iou':iou,'coverage':coverage,'Z_front':Z_front,
            'Z_front_err':Z_front_err,'dims':dims,'ctr':ctr,'ctr_ego':ctr_ego,
            'dim_err':dim_err,'n_in_mask':n_in_mask,
            'bbox_mesh':[bx1,by1,bx2,by2]}


# ── Hybrid anchor helper ─────────────────────────────────────────────────────
def apply_hybrid_anchor(verts_r3, binary_mask, K_proj, u_in, v_in, Z_in, search_radius=8):
    Hm, Wm = binary_mask.shape
    fx_p=float(K_proj[0,0]); fy_p=float(K_proj[1,1])
    cx_p=float(K_proj[0,2]); cy_p=float(K_proj[1,2])

    valid_z = verts_r3[:,2] > 1e-6
    u_v = np.where(valid_z, fx_p*verts_r3[:,0]/np.maximum(verts_r3[:,2],1e-6)+cx_p, -1.)
    v_v = np.where(valid_z, fy_p*verts_r3[:,1]/np.maximum(verts_r3[:,2],1e-6)+cy_p, -1.)
    u_vi2=np.round(u_v).astype(int); v_vi2=np.round(v_v).astype(int)
    in_img_v=valid_z&(u_vi2>=0)&(u_vi2<Wm)&(v_vi2>=0)&(v_vi2<Hm)
    mhit=np.zeros(len(verts_r3),dtype=bool)
    mhit[in_img_v]=binary_mask[v_vi2[in_img_v],u_vi2[in_img_v]]
    visible_idx=np.where(mhit)[0]

    Z_front_anch=float(Z_in.min()); clean=Z_in<=Z_front_anch*1.5
    Z_c=Z_in[clean]; u_c=u_in[clean]; v_c=v_in[clean]

    scale=1.0; n_matched=0
    if len(visible_idx)>=10 and len(Z_c)>=3:
        lpix=np.stack([u_c,v_c],axis=1).astype(np.float32)
        vpix=np.stack([u_v[visible_idx],v_v[visible_idx]],axis=1).astype(np.float32)
        tree=cKDTree(lpix); dists,nn_idx=tree.query(vpix,k=1,workers=-1)
        matched=dists<=search_radius
        if matched.sum()<3: matched=dists<=search_radius*3
        if matched.sum()>=3:
            zm=verts_r3[visible_idx[matched],2]; zl=Z_c[nn_idx[matched]]
            ratios=zl/np.maximum(zm,1e-6); scale=float(np.median(ratios)); n_matched=int(matched.sum())

    vs=verts_r3*scale
    Z_med=float(np.median(vs[:,2])); near=vs[:,2]<=Z_med
    near_ctr=(vs[near].mean(axis=0) if near.sum()>=10 else vs.mean(axis=0))
    Xl=(u_c-cx_p)*Z_c/fx_p; Yl=(v_c-cy_p)*Z_c/fy_p
    lidar_ctr=np.array([Xl.mean(),Yl.mean(),Z_c.mean()],dtype=np.float32)
    t=lidar_ctr-near_ctr
    return (vs+t).astype(np.float32), scale, t, len(visible_idx), n_matched



print("compute_metrics, apply_hybrid_anchor defined.")


In [ ]:
# ── §15 core: z-buffer visibility extraction + VAR refinement ────────────────

def _extract_visible_verts(verts_r3, binary_mask, K,
                            dilation_px=50, zbuf_tol=0.30):
    """
    Extract front-facing mesh vertices via projection + z-buffer.

    Steps
    -----
    1. Project all vertices into image, drop behind-camera / out-of-bounds.
    2. Dilate the binary mask.
    3. Keep only vertices whose pixel falls inside the dilated mask.
    4. Build a per-pixel depth buffer (nearest depth wins).
    5. Keep vertices within zbuf_tol of their pixel's nearest depth
       → removes back-face / occluded vertices.

    Returns
    -------
    vis_mask : (V,) bool  — final visible vertices
    dbg      : dict       — intermediate arrays for debug plots
    """
    H, W = binary_mask.shape
    fx = float(K[0,0]); fy = float(K[1,1])
    cx = float(K[0,2]); cy = float(K[1,2])
    v = verts_r3.astype(np.float64)

    # Project
    vz   = v[:,2] > 1e-6
    u_f  = np.where(vz, fx*v[:,0]/np.maximum(v[:,2],1e-6)+cx, -9999.)
    v_f  = np.where(vz, fy*v[:,1]/np.maximum(v[:,2],1e-6)+cy, -9999.)
    u_vi = np.round(u_f).astype(int)
    v_vi = np.round(v_f).astype(int)
    in_img = vz & (u_f > -9000) & (u_vi >= 0) & (u_vi < W) & (v_vi >= 0) & (v_vi < H)
    idx = np.where(in_img)[0]

    # Dilated mask
    kern  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilation_px*2+1, dilation_px*2+1))
    mdil  = cv2.dilate(binary_mask.astype(np.uint8), kern).astype(bool)
    in_dil = np.zeros(len(v), bool)
    if len(idx):
        in_dil[idx] = mdil[v_vi[idx], u_vi[idx]]

    # Per-pixel depth buffer (min depth per pixel across all in-image verts)
    depth_buf = np.full((H, W), np.inf, dtype=np.float64)
    if len(idx):
        np.minimum.at(depth_buf, (v_vi[idx], u_vi[idx]), v[idx, 2])

    # Z-buffer visibility: vert depth ≈ nearest depth at its pixel
    zbuf_vis = np.zeros(len(v), bool)
    if len(idx):
        zbuf_vis[idx] = (
            np.abs(v[idx, 2] - depth_buf[v_vi[idx], u_vi[idx]]) <= zbuf_tol
        )

    vis_mask = in_dil & zbuf_vis
    dbg = dict(u_f=u_f, v_f=v_f, u_vi=u_vi, v_vi=v_vi,
               in_img=in_img, in_dil=in_dil, zbuf_vis=zbuf_vis,
               depth_buf=depth_buf, vis_mask=vis_mask)
    return vis_mask, dbg


# ── Updated distance helper (uses z-buffer + trimmed Chamfer) ─────────────────
def compute_lidar_vis_mesh_dist(verts_r3, binary_mask, K,
                                u_lf, v_lf, Z_lf, R_c2e, t_c2e,
                                dilation_px=50, zbuf_tol=0.30, trim_frac=0.80):
    # Trimmed mean distance: filtered LiDAR → nearest z-buffer-visible mesh vert.
    if len(Z_lf) == 0:
        return float('nan'), 0
    vis_mask, _ = _extract_visible_verts(verts_r3, binary_mask, K, dilation_px, zbuf_tol)
    n_vis = int(vis_mask.sum())
    if n_vis < 3:
        return float('nan'), 0
    R = R_c2e.astype(np.float64); t = t_c2e.astype(np.float64)
    fx_ = float(K[0,0]); fy_ = float(K[1,1])
    cx_ = float(K[0,2]); cy_ = float(K[1,2])
    vis_ego = (R @ verts_r3.astype(np.float64)[vis_mask].T).T + t
    Ulf = u_lf.astype(np.float64); Vlf = v_lf.astype(np.float64); Zlf = Z_lf.astype(np.float64)
    li_ego = (R @ np.stack([(Ulf-cx_)*Zlf/fx_, (Vlf-cy_)*Zlf/fy_, Zlf], 1).T).T + t
    dists, _ = cKDTree(vis_ego).query(li_ego)
    n_keep = max(3, int(len(dists) * trim_frac))
    return float(np.mean(np.sort(dists)[:n_keep])), n_vis


# ── Main VAR function ──────────────────────────────────────────────────────────
def visibility_aware_refinement(
    verts_r3, binary_mask, K,
    u_lf, v_lf, Z_lf,
    R_c2e, t_c2e,
    current_obb_yaw,
    dilation_px=50,
    zbuf_tol=0.30,
    max_trans_xy_m=1.0,
    max_trans_z_m=1.50,
    max_yaw_deg=10.0,
    scale_min=0.95, scale_max=1.05,
    min_vis_verts=15,
    min_lidar_yaw=6,
    yaw_eigenratio_min=2.0,
    trim_frac=0.80,
    do_yaw=True,
    do_scale=False,
    debug=True,
):
    # Visibility-aware post-hoc refinement.  Returns (verts_corrected_f32, info_dict).
    H_, W_ = binary_mask.shape
    fx_ = float(K[0,0]); fy_ = float(K[1,1])
    cx_ = float(K[0,2]); cy_ = float(K[1,2])
    R = R_c2e.astype(np.float64); t = t_c2e.astype(np.float64)

    info = dict(n_vis_verts=0, n_lidar=len(Z_lf),
                delta_cam_z=0., delta_ego_xy=np.zeros(2),
                delta_yaw_deg=0., scale=1.0,
                lidar_mesh_dist=float('nan'),
                lidar_mesh_dist_before=float('nan'),
                status='ok', _dbg=None)

    verts = verts_r3.astype(np.float64)

    # ── 1. Z-buffer visible verts ─────────────────────────────────────────────
    vis_mask, dbg0 = _extract_visible_verts(verts, binary_mask, K, dilation_px, zbuf_tol)
    n_vis = int(vis_mask.sum())
    info['n_vis_verts'] = n_vis
    if n_vis < min_vis_verts:
        info['status'] = f'skip_too_few_vis ({n_vis})'
        return verts_r3.astype(np.float32), info
    if len(Z_lf) < 3:
        info['status'] = f'skip_too_few_lidar ({len(Z_lf)})'
        return verts_r3.astype(np.float32), info

    # ── 2. LiDAR → ego ────────────────────────────────────────────────────────
    Ulf = u_lf.astype(np.float64); Vlf = v_lf.astype(np.float64); Zlf = Z_lf.astype(np.float64)
    li_cam = np.stack([(Ulf-cx_)*Zlf/fx_, (Vlf-cy_)*Zlf/fy_, Zlf], axis=1)
    li_ego = (R @ li_cam.T).T + t                                   # (N, 3)

    # ── 3. Z-front correction in camera depth ─────────────────────────────────
    z_front_li  = float(Zlf.min())
    z_front_vis = float(verts[vis_mask, 2].min())
    dz = float(np.clip(z_front_li - z_front_vis, -max_trans_z_m, max_trans_z_m))
    info['delta_cam_z'] = dz
    verts[:, 2] += dz

    # ── 4. Trimmed XY translation in ego ──────────────────────────────────────
    # Use NN matching (not plain centroid) for robustness
    vis_ego = (R @ verts[vis_mask].T).T + t                          # (n_vis, 3)
    dists_l, nn_idx_l = cKDTree(vis_ego).query(li_ego)
    n_keep = max(3, int(len(dists_l) * trim_frac))
    keep = np.argsort(dists_l)[:n_keep]
    li_trimmed   = li_ego[keep]
    mesh_trimmed = vis_ego[nn_idx_l[keep]]
    dxy = li_trimmed.mean(0)[:2] - mesh_trimmed.mean(0)[:2]
    mag = float(np.linalg.norm(dxy))
    if mag > max_trans_xy_m:
        dxy = dxy * max_trans_xy_m / mag
    info['delta_ego_xy'] = dxy

    # Record "before" dist using trimmed Chamfer on original+Z-shifted vis verts
    dists_before_trim = dists_l  # already computed with Z-shifted verts
    dist_before = float(np.mean(np.sort(dists_before_trim)[:n_keep]))
    info['lidar_mesh_dist_before'] = dist_before

    # Store debug data (before XY correction)
    if debug:
        info['_dbg'] = dict(
            dbg0=dbg0,                          # projection / z-buffer arrays
            li_ego=li_ego,
            vis_ego_before=vis_ego.copy(),
            keep_idx=keep,
            nn_idx_l=nn_idx_l,
            n_keep=n_keep,
        )

    # Apply XY translation
    ve = (R @ verts.T).T + t
    ve[:, :2] += dxy
    verts = (R.T @ (ve - t).T).T

    # ── 5. Yaw correction in ego XY ───────────────────────────────────────────
    dyaw = 0.0
    if do_yaw and len(Zlf) >= min_lidar_yaw:
        li_xy_c = li_ego[:, :2] - li_ego[:, :2].mean(0)
        if li_xy_c.shape[0] >= 2:
            eigvals_li, evecs_li = np.linalg.eigh(np.cov(li_xy_c.T))
            eigenratio = float(eigvals_li[-1] / max(eigvals_li[0], 1e-9))
            if eigenratio >= yaw_eigenratio_min:
                ev = evecs_li[:, -1]
                y1 = np.arctan2(ev[1], ev[0])
                y2 = y1 + np.pi if y1 <= 0 else y1 - np.pi
                diffs = [((c - current_obb_yaw) + np.pi) % (2*np.pi) - np.pi
                         for c in (y1, y2)]
                dyaw = float(min(diffs, key=abs))
                dyaw = float(np.clip(dyaw, -np.radians(max_yaw_deg), np.radians(max_yaw_deg)))
            else:
                info['status'] = info['status'] + '|yaw_skipped_flat_cluster'

    info['delta_yaw_deg'] = float(np.degrees(dyaw))
    if abs(dyaw) > 1e-6:
        ve  = (R @ verts.T).T + t
        cr  = ve.mean(0)
        cd, sd = np.cos(dyaw), np.sin(dyaw)
        Ry  = np.array([[cd, -sd, 0.], [sd, cd, 0.], [0., 0., 1.]])
        ve  = (Ry @ (ve - cr).T).T + cr
        verts = (R.T @ (ve - t).T).T

    # ── 6. Conservative scale ─────────────────────────────────────────────────
    scale = 1.0
    if do_scale and len(Zlf) >= 6:
        vf_ego   = (R @ verts[vis_mask].T).T + t
        ctr_vf   = vf_ego.mean(0)
        ctr_li   = li_ego.mean(0)
        vis_ext  = float(np.percentile(np.linalg.norm(vf_ego[:, :2]-ctr_vf[:2], axis=1), 90))
        li_ext   = float(np.percentile(np.linalg.norm(li_ego[:, :2]-ctr_li[:2], axis=1), 90))
        if vis_ext > 0.1:
            scale = float(np.clip(li_ext / vis_ext, scale_min, scale_max))
    info['scale'] = scale
    if abs(scale - 1.0) > 1e-4:
        ve  = (R @ verts.T).T + t
        cs  = ve.mean(0)
        ve  = (ve - cs) * scale + cs
        verts = (R.T @ (ve - t).T).T

    # ── 7. Final distance + rejection check ───────────────────────────────────
    vis_mask_fin, _ = _extract_visible_verts(verts, binary_mask, K, dilation_px, zbuf_tol)
    n_vis_fin = int(vis_mask_fin.sum())
    if n_vis_fin >= 3:
        vis_ego_fin = (R @ verts[vis_mask_fin].T).T + t
        d_fin, _ = cKDTree(vis_ego_fin).query(li_ego)
        n_k = max(3, int(len(d_fin) * trim_frac))
        dist_after = float(np.mean(np.sort(d_fin)[:n_k]))
    else:
        dist_after = float('inf')
    info['lidar_mesh_dist'] = dist_after

    # Reject if correction made alignment worse (allow 5% slack)
    if np.isfinite(dist_before) and dist_after > dist_before * 1.05:
        info['status'] = f'rejected_worse_dist ({dist_before:.3f} -> {dist_after:.3f})'
        return verts_r3.astype(np.float32), info

    # Reject if any single correction hit its bound exactly (suspicious)
    if (abs(abs(dz)  - max_trans_z_m) < 1e-4 or
        abs(mag - max_trans_xy_m) < 1e-4 or
        abs(abs(dyaw) - np.radians(max_yaw_deg)) < 1e-4):
        info['status'] = info['status'] + '|hit_bound'

    return verts.astype(np.float32), info


print('_extract_visible_verts, compute_lidar_vis_mesh_dist, visibility_aware_refinement defined.')
print('Parameters: max_trans_xy=1.0m  max_trans_z=0.75m  max_yaw=10deg  scale=[0.95,1.05]')
print('z-buffer tol=0.30m  trim_frac=0.80  yaw eigenratio_min=2.0')


In [ ]:
# -- [EXPERIMENTAL] VAR4: LiDAR-anchor face alignment -------------------------
#
# Pipeline:
#  1. Backproject in-mask LiDAR anchors to ego frame.
#  1.5 Anchor pre-clustering: keep dominant anchor group.
#  2. Collect nearby full-cloud candidates.
#  3. Remove ground (anchor-centered, soft clearance).
#  4. DBSCAN-cluster; keep cluster connected to kept anchors.
#  5. Z-front correction (same as VAR1/2/3).
#  6. Body OBB (trimmed percentile dims, suppresses mirrors).
#  7. Shape classification (I_front_back / I_side / L_shape / I_ambiguous).
#  8. Hypothesis testing. Optimisation loss:
#       face_residual + yaw + tangent_centering (1-face)
#       + anchor_containment (strong, untrimmed MSE)
#       + cluster_containment (soft, trimmed MSE)
#     Fits with excessive tangent offset (corner-hugging) are discarded.
#  9. L-shape post-fit validation; fallback to best I-shape if needed.
# 10. Face-loss acceptance (raw loss vs baseline).
# 11. Apply mesh BEV correction.
# 12. Size correction (OBB dims only, no mesh deformation).
# 13. Recompute anchor_inside_ratio and cluster_inside_ratio on var4_obb_final.
#     Hard-reject on anchor_inside_ratio; soft-warn on cluster_inside_ratio.
#
# VAR4 is EXPERIMENTAL. VAR2 remains the default.

# -- Geometry helpers ----------------------------------------------------------

def _v4_rot2(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, -s], [s, c]])


def _v4_trimmed_mean_abs(r, frac):
    k = max(3, int(len(r) * frac))
    return float(np.mean(np.sort(np.abs(r))[:k]))


def _v4_obb_faces(obb):
    """Return [(face_center_2d, face_normal_2d, label)] for the 4 OBB faces."""
    c  = obb['center'][:2]
    al = obb['axes'][:2, 0];  aw = obb['axes'][:2, 1]
    hl = obb['dims'][0] / 2;  hw = obb['dims'][1] / 2
    return [
        (c + hl * al,  al.copy(), '+L'),
        (c - hl * al, -al.copy(), '-L'),
        (c + hw * aw,  aw.copy(), '+W'),
        (c - hw * aw, -aw.copy(), '-W'),
    ]


def _v4_face_residuals(pts_2d, fc0, fn0, dx, dy, dyaw, obb_c2d):
    """Signed perpendicular residuals from pts to a face after (dx, dy, dyaw)."""
    R2   = _v4_rot2(dyaw)
    fn_r = R2 @ fn0
    fc_r = obb_c2d + R2 @ (fc0 - obb_c2d) + np.array([dx, dy])
    return (pts_2d - fc_r) @ fn_r


def _v4_obb_excess(pts_2d, new_c, ral, raw_, obb_hl, obb_hw):
    """Per-point excess outside OBB footprint (length + width, >=0)."""
    dp    = pts_2d - new_c
    exc_l = np.maximum(0., np.abs(dp @ ral) - obb_hl)
    exc_w = np.maximum(0., np.abs(dp @ raw_) - obb_hw)
    return exc_l + exc_w


def _v4_loss_1face(params, support_pts_2d, pts_2d, fc0, fn0, obb_c2d,
                   trim_frac, yaw_pen, along_pen, max_yaw,
                   obb_al, obb_aw, obb_hl, obb_hw,
                   anc_cont_w, cl_cont_w, tang_w, anchors_2d):
    """Loss for one-face fitting.

    perp_residual uses `support_pts_2d` (face-boundary subset of cluster).
    containment uses full `pts_2d` cluster + alignment anchors.
    tangent-centering uses support_pts median.
    """
    dx, dy, dyaw = params
    dyaw = float(np.clip(dyaw, -max_yaw, max_yaw))
    R2   = _v4_rot2(dyaw)

    # Perpendicular residual on face-support subset only
    r    = _v4_face_residuals(support_pts_2d, fc0, fn0, dx, dy, dyaw, obb_c2d)
    loss = _v4_trimmed_mean_abs(r, trim_frac)

    # Along-face sliding regulariser
    ft    = np.array([-fn0[1], fn0[0]])
    along = abs(np.array([dx, dy]) @ (R2 @ ft))

    # Rotated OBB axes
    new_c = obb_c2d + np.array([dx, dy])
    ral   = R2 @ obb_al;  raw_ = R2 @ obb_aw

    # Anchor containment: untrimmed mean squared excess — full alignment anchors
    if len(anchors_2d):
        anc_exc  = _v4_obb_excess(anchors_2d, new_c, ral, raw_, obb_hl, obb_hw)
        anc_cont = float(np.mean(anc_exc ** 2))
    else:
        anc_cont = 0.

    # Cluster containment: full cluster, untrimmed — interior points help here
    cl_exc  = _v4_obb_excess(pts_2d, new_c, ral, raw_, obb_hl, obb_hw)
    cl_cont = float(np.mean(cl_exc ** 2))

    # Tangent-centering: support-point median near face centre along tangent
    fn_r     = R2 @ fn0
    ft_r     = np.array([-fn_r[1], fn_r[0]])
    fc_r     = obb_c2d + R2 @ (fc0 - obb_c2d) + np.array([dx, dy])
    tang_off = float((np.median(support_pts_2d, axis=0) - fc_r) @ ft_r)

    return (loss
            + along_pen  * along
            + yaw_pen    * dyaw**2
            + anc_cont_w * anc_cont
            + cl_cont_w  * cl_cont
            + tang_w     * tang_off**2)


def _v4_fit_1face(support_pts_2d, pts_2d, fc0, fn0, obb_c2d, max_trans, max_yaw,
                  trim_frac, yaw_pen, along_pen,
                  obb_al, obb_aw, obb_hl, obb_hw,
                  anc_cont_w, cl_cont_w, tang_w, anchors_2d):
    """Optimise (dx, dy, dyaw) for one OBB face. Returns (dx,dy,dyaw,pen_loss,raw_loss).

    Initial guess seeded from support_pts centroid; raw_loss evaluated on support_pts.
    """
    from scipy.optimize import minimize
    best_l, best_p = float('inf'), np.zeros(3)
    for dy_deg in np.arange(-np.degrees(max_yaw), np.degrees(max_yaw) + 0.1, 5):
        dyaw_i = np.radians(dy_deg)
        fn_r   = _v4_rot2(dyaw_i) @ fn0
        fc_r   = obb_c2d + _v4_rot2(dyaw_i) @ (fc0 - obb_c2d)
        d_n    = float(np.clip((support_pts_2d.mean(0) - fc_r) @ fn_r, -max_trans, max_trans))
        p0     = [d_n * fn_r[0], d_n * fn_r[1], dyaw_i]
        res    = minimize(_v4_loss_1face, p0,
                          args=(support_pts_2d, pts_2d, fc0, fn0, obb_c2d,
                                trim_frac, yaw_pen, along_pen, max_yaw,
                                obb_al, obb_aw, obb_hl, obb_hw,
                                anc_cont_w, cl_cont_w, tang_w, anchors_2d),
                          method='Nelder-Mead',
                          options={'xatol': 0.005, 'fatol': 0.0005, 'maxiter': 400})
        if res.fun < best_l:
            best_l, best_p = res.fun, res.x
    dx    = float(np.clip(best_p[0], -max_trans, max_trans))
    dy    = float(np.clip(best_p[1], -max_trans, max_trans))
    dyaw  = float(np.clip(best_p[2], -max_yaw, max_yaw))
    raw_l = _v4_trimmed_mean_abs(
        _v4_face_residuals(support_pts_2d, fc0, fn0, dx, dy, dyaw, obb_c2d), trim_frac)
    return dx, dy, dyaw, best_l, raw_l


def _v4_loss_L(params, pts_2d, fc1, fn1, fc2, fn2, obb_c2d,
               trim_frac, yaw_pen, max_yaw,
               obb_al, obb_aw, obb_hl, obb_hw,
               anc_cont_w, cl_cont_w, anchors_2d,
               secondary_weight=1.0):
    """Loss for L-shape.

    secondary_weight=1.0 (strong L): symmetric min-residual across both faces.
    secondary_weight<1.0 (weak L): r1=primary face dominates; r2=secondary with lower weight.
    fc1/fn1 = primary face, fc2/fn2 = secondary face.
    """
    dx, dy, dyaw = params
    dyaw  = float(np.clip(dyaw, -max_yaw, max_yaw))
    R2    = _v4_rot2(dyaw)
    r1    = _v4_face_residuals(pts_2d, fc1, fn1, dx, dy, dyaw, obb_c2d)
    r2    = _v4_face_residuals(pts_2d, fc2, fn2, dx, dy, dyaw, obb_c2d)

    if secondary_weight >= 0.95:
        # Strong L: symmetric — per-point minimum residual
        r_use = np.where(np.abs(r1) < np.abs(r2), r1, r2)
        face_loss = _v4_trimmed_mean_abs(r_use, trim_frac)
    else:
        # Weak L: primary face residual + down-weighted secondary face
        face_loss = (_v4_trimmed_mean_abs(r1, trim_frac) +
                     secondary_weight * _v4_trimmed_mean_abs(r2, trim_frac))

    new_c = obb_c2d + np.array([dx, dy])
    ral   = R2 @ obb_al;  raw_ = R2 @ obb_aw

    if len(anchors_2d):
        anc_exc  = _v4_obb_excess(anchors_2d, new_c, ral, raw_, obb_hl, obb_hw)
        anc_cont = float(np.mean(anc_exc ** 2))
    else:
        anc_cont = 0.

    cl_exc  = _v4_obb_excess(pts_2d, new_c, ral, raw_, obb_hl, obb_hw)
    cl_cont = float(np.mean(cl_exc ** 2))

    return (face_loss
            + yaw_pen    * dyaw**2
            + anc_cont_w * anc_cont
            + cl_cont_w  * cl_cont)


def _v4_fit_L(pts_2d, fc1, fn1, fc2, fn2, obb_c2d, max_trans, max_yaw,
              trim_frac, yaw_pen,
              obb_al, obb_aw, obb_hl, obb_hw,
              anc_cont_w, cl_cont_w, anchors_2d,
              secondary_weight=1.0):
    """Optimise (dx, dy, dyaw) for an L-shape. fc1/fn1=primary, fc2/fn2=secondary."""
    from scipy.optimize import minimize
    cl_ctr = pts_2d.mean(0)
    best_l, best_p = float('inf'), np.zeros(3)
    for dy_deg in np.arange(-np.degrees(max_yaw), np.degrees(max_yaw) + 0.1, 5):
        dyaw_i = np.radians(dy_deg)
        p0     = [float(np.clip(cl_ctr[0] - obb_c2d[0], -max_trans, max_trans)),
                  float(np.clip(cl_ctr[1] - obb_c2d[1], -max_trans, max_trans)),
                  dyaw_i]
        res    = minimize(_v4_loss_L, p0,
                          args=(pts_2d, fc1, fn1, fc2, fn2, obb_c2d,
                                trim_frac, yaw_pen, max_yaw,
                                obb_al, obb_aw, obb_hl, obb_hw,
                                anc_cont_w, cl_cont_w, anchors_2d,
                                secondary_weight),
                          method='Nelder-Mead',
                          options={'xatol': 0.005, 'fatol': 0.0005, 'maxiter': 500})
        if res.fun < best_l:
            best_l, best_p = res.fun, res.x
    dx   = float(np.clip(best_p[0], -max_trans, max_trans))
    dy   = float(np.clip(best_p[1], -max_trans, max_trans))
    dyaw = float(np.clip(best_p[2], -max_yaw, max_yaw))
    r1   = _v4_face_residuals(pts_2d, fc1, fn1, dx, dy, dyaw, obb_c2d)
    r2   = _v4_face_residuals(pts_2d, fc2, fn2, dx, dy, dyaw, obb_c2d)
    if secondary_weight >= 0.95:
        r_comb = np.where(np.abs(r1) < np.abs(r2), r1, r2)
    else:
        r_comb = r1   # raw_loss reported on primary face
    raw_l = _v4_trimmed_mean_abs(r_comb, trim_frac)
    return dx, dy, dyaw, best_l, raw_l


def _v4_mesh_obb_ego(verts_ego):
    """BEV PCA OBB from ego-frame mesh vertices (full extent, min/max)."""
    xy   = verts_ego[:, :2];  xy_c = xy - xy.mean(0)
    _, evecs = np.linalg.eigh(np.cov(xy_c.T));  evecs = evecs[:, ::-1]
    ax_l = np.array([evecs[0, 0], evecs[1, 0], 0.])
    ax_w = np.array([evecs[0, 1], evecs[1, 1], 0.])
    if ax_l[0]*ax_w[1] - ax_l[1]*ax_w[0] < 0: ax_w = -ax_w
    axs  = np.stack([ax_l, ax_w, np.array([0., 0., 1.])], axis=1)
    prj  = verts_ego @ axs;  mid = (prj.min(0) + prj.max(0)) / 2
    ctr  = axs @ mid;  dims = prj.max(0) - prj.min(0)
    return dict(center=ctr, axes=axs, dims=dims)


def _v4_mesh_body_obb_ego(verts_ego, len_pct=(0.5, 99.5), wid_pct=(2, 98), hgt_pct=(1, 99)):
    """PCA OBB with robust percentile extents to suppress mirror/extreme protrusions.

    width p2-p98: mild mirror suppression without cutting real body.
    length p0.5-p99.5: minimal trimming.
    """
    xy   = verts_ego[:, :2];  xy_c = xy - xy.mean(0)
    _, evecs = np.linalg.eigh(np.cov(xy_c.T));  evecs = evecs[:, ::-1]
    ax_l = np.array([evecs[0, 0], evecs[1, 0], 0.])
    ax_w = np.array([evecs[0, 1], evecs[1, 1], 0.])
    if ax_l[0]*ax_w[1] - ax_l[1]*ax_w[0] < 0: ax_w = -ax_w
    axs  = np.stack([ax_l, ax_w, np.array([0., 0., 1.])], axis=1)
    prj  = verts_ego @ axs
    lo_l, hi_l = np.percentile(prj[:, 0], [len_pct[0], len_pct[1]])
    lo_w, hi_w = np.percentile(prj[:, 1], [wid_pct[0], wid_pct[1]])
    lo_z, hi_z = np.percentile(prj[:, 2], [hgt_pct[0], hgt_pct[1]])
    mid  = np.array([(lo_l + hi_l) / 2, (lo_w + hi_w) / 2, (lo_z + hi_z) / 2])
    ctr  = axs @ mid
    dims = np.array([hi_l - lo_l, hi_w - lo_w, hi_z - lo_z])
    return dict(center=ctr, axes=axs, dims=dims,
                len_pct=len_pct, wid_pct=wid_pct, hgt_pct=hgt_pct)


def _v4_containment_stats(pts_2d, dx, dy, dyaw, obb_c2d,
                           obb_al, obb_aw, obb_hl, obb_hw, trim_frac):
    """Trimmed-mean excess and inside_ratio for pts relative to a candidate OBB."""
    R2    = _v4_rot2(dyaw)
    new_c = obb_c2d + np.array([dx, dy])
    excess = _v4_obb_excess(pts_2d, new_c, R2 @ obb_al, R2 @ obb_aw, obb_hl, obb_hw)
    inside_ratio = float(np.mean(excess == 0.))
    k            = max(3, int(len(excess) * trim_frac))
    cont_loss    = float(np.mean(np.sort(excess)[:k]))
    return cont_loss, inside_ratio


def _v4_obb_inside(pts_2d, obb_c2d, obb_al, obb_aw, obb_hl, obb_hw):
    """Fraction of pts_2d inside OBB footprint + mean squared excess."""
    excess       = _v4_obb_excess(pts_2d, obb_c2d, obb_al, obb_aw, obb_hl, obb_hw)
    inside_ratio = float(np.mean(excess == 0.))
    mse          = float(np.mean(excess ** 2))
    return inside_ratio, mse


def _v4_face_support_pts(cluster_2d, c2d, fn0, support_frac=0.25, band_m=0.40):
    """Extract cluster points near the outer boundary along face normal fn0.

    Projects each cluster point onto fn0 (relative to OBB centre c2d).
    Selects the top `support_frac` of points by projection AND within
    `band_m` from the outermost point.  Returns (support_pts, debug_dict).
    """
    proj      = (cluster_2d - c2d) @ fn0
    proj_max  = float(proj.max())
    thresh_q  = float(np.percentile(proj, 100.0 * (1.0 - support_frac)))
    thresh_b  = proj_max - band_m
    threshold = max(thresh_q, thresh_b)
    mask      = proj >= threshold
    return cluster_2d[mask], {
        'face_support_count':    int(mask.sum()),
        'face_support_fraction': float(mask.mean()),
        'face_support_band_m':   float(proj_max - threshold),
        'face_support_mode':     f'top{support_frac:.0%}+band{band_m:.2f}m',
    }


# -- Core VAR4 function -------------------------------------------------------

def var4_cluster_alignment(
    verts_r3_in,
    binary_mask, K,
    u_lf, v_lf, Z_lf,
    R_c2e, t_c2e,
    R_l2e, t_l2e,
    xyz_lid,
    current_obb_yaw,
    # VAR2 prior (optional — use VAR2-corrected mesh as starting pose)
    verts_var2=None,
    info_var2=None,
    # Candidate collection
    cluster_radius=2.0,
    # Ground removal
    ground_crop_r=10.0,
    ground_clearance=0.20,
    max_obj_height_m=4.0,
    # Clustering
    dbscan_eps=0.45,
    dbscan_min_samples=3,
    max_cluster_extent=12.0,
    min_cluster_pts=5,
    # Face alignment optimisation
    max_trans_xy_m=3.0,
    max_yaw_deg=15.0,
    yaw_penalty=0.20,
    along_penalty=0.05,
    trim_frac=0.80,
    # Containment weights (both inside the optimizer)
    anchor_containment_weight=30.0,  # untrimmed MSE on cluster-filtered anchors — hard
    cluster_containment_weight=15.0, # untrimmed MSE on cluster — strong
    # Tangent-centering (one-face fits, inside optimizer)
    tangent_center_weight=0.20,
    # Face-loss acceptance
    loss_slack=1.05,
    # Inside-ratio acceptance (checked on var4_obb_final after size correction)
    anchor_inside_ratio_threshold=0.85,   # soft warn if below
    anchor_inside_ratio_floor=0.70,       # hard reject if below
    cluster_inside_ratio_threshold=0.70,  # soft warn only
    # Corner-alignment guard: max |tangent_offset| / OBB_half_tangent_dim
    tangent_max_frac=0.65,
    # Z-buffer (same as VAR1/2/3)
    zbuf_tol=0.30, dilation_px=50,
    max_trans_z_m=1.50,
    # Anchor pre-clustering
    anchor_dbscan_eps=1.0,
    anchor_dbscan_min=2,
    # Shape classification
    l_spread_min=1.0,
    l_spread_ratio=1.5,
    # L-shape validation
    min_l_face_pts_frac=0.15,
    # Ego-visible face prior (post-hoc, mode selection only)
    ego_face_weight=0.03,
    # Size correction (OBB dims only, no mesh deformation)
    size_margin=0.30,
    l_scale_min=0.80,
    l_scale_max=1.40,
    w_scale_min=0.80,
    w_scale_max=1.20,
    size_min_cluster_pts=30,  # minimum cluster pts to allow size correction
    # Side-face length correction (L-shape)
    side_face_length_margin=0.20,
    min_side_face_pts=8,
    # Face-support extraction (one-face modes only)
    face_support_frac=0.25,    # top fraction of cluster along face normal
    face_support_band_m=0.40,  # max band width from outermost point
    min_face_support_pts=6,    # minimum support pts; candidate skipped if fewer
    # Yaw penalty by mode (relative to starting pose = VAR2 if provided)
    yaw_penalty_1face=0.30,      # I-shaped: strongly resist extra rotation
    yaw_penalty_strong_l=0.10,   # strong L with good two-face support
    yaw_penalty_weak_l=0.25,     # weak / partial L-shape
    # L-shape strength classification
    l_shape_strength_threshold=0.40,  # min secondary/primary support ratio → strong L
    weak_l_secondary_weight=0.30,     # face-loss weight for secondary face in weak L
    # VAR2 fallback
    var2_improvement_margin=0.05,     # VAR4 must beat VAR2 Chamfer by this fraction
):
    """[EXPERIMENTAL] VAR4: LiDAR-anchor face alignment.

    Containment is split into anchor (strong, untrimmed MSE) and cluster (soft,
    trimmed MSE) and both are inside the optimiser.  The inside-ratio check runs
    on var4_obb_final AFTER size correction so size-corrected OBBs are not
    rejected purely because the pre-correction OBB was too tight.
    """
    from sklearn.cluster import DBSCAN

    max_yaw_rad = np.radians(max_yaw_deg)

    # -- Info dict ------------------------------------------------------------
    info = dict(
        status='ok',
        var4_mode=None,
        selected_faces=None,
        anchor_points_count=int(len(Z_lf)),
        nearby_candidates_count=0,
        cluster_points_count=0,
        rejected_cluster_count=0,
        face_loss_before=float('nan'),
        face_loss_after=float('nan'),
        dyaw=0.,
        dxy=np.zeros(2),
        var4_rejected_reason=None,
        # Standard keys shared with VAR1/2/3
        delta_cam_z=0.,
        delta_ego_xy=np.zeros(2),
        delta_yaw_deg=0.,
        scale=1.,
        n_vis_verts=0,
        lidar_mesh_dist_before=float('inf'),
        lidar_mesh_dist=float('inf'),
        # Point sets for debug plots
        anchor_ego=None,
        anchor_kept_ego=None,
        anchor_rejected_ego=[],
        anchor_groups_count=1,
        kept_anchor_count=0,
        rejected_anchor_count=0,
        anchor_opt_count=0,
        anchor_excluded_count=0,
        anchor_excluded_ego=None,
        nearby_ego=None,
        ground_removed_ego=None,
        cluster_ego=None,
        rejected_clusters_ego=[],
        ground_success=False,
        ground_z=None,
        ground_inliers_ego=None,
        mesh_obb_before=None,
        mesh_obb_after=None,
        mode_results={},
        shape_classification=None,
        l_shape_allowed=False,
        face_support_counts={},
        ego_visible_face_prior=None,
        l_shape_face_counts={},
        l_shape_rejected_reason=None,
        alignment_residuals=None,
        alignment_face_centers=None,
        alignment_face_normals=None,
        alignment_assign=None,
        # VAR2 prior / fallback
        yaw_prior_source='raw',
        yaw_delta_from_var2=float('nan'),
        var2_chamfer=float('nan'),
        var4_chamfer=float('nan'),
        var4_improves_var2=None,
        fallback_used=False,
        # L-shape strength
        l_shape_type='',
        l_shape_strength=float('nan'),
        primary_face_support_count=0,
        secondary_face_support_count=0,
        primary_face_loss=float('nan'),
        secondary_face_loss=float('nan'),
        secondary_face_weight=1.0,
        # Face-support debug (one-face modes)
        face_support_count=0,
        face_support_fraction=float('nan'),
        face_support_band_m=float('nan'),
        face_support_mode='',
        face_loss_before_support=float('nan'),
        face_loss_after_support=float('nan'),
        # Containment / tangent debug (on var4_obb_final)
        final_anchor_inside_ratio=float('nan'),
        final_cluster_inside_ratio=float('nan'),
        final_anchor_containment=float('nan'),
        final_cluster_containment=float('nan'),
        final_tangent_offset=float('nan'),
        # Explicit containment debug names (alias final_* + count fields)
        anchor_inside_ratio=float('nan'),
        cluster_inside_ratio=float('nan'),
        anchor_outside_count=0,
        cluster_outside_count=0,
        anchor_containment_loss=float('nan'),
        cluster_containment_loss=float('nan'),
        # Size correction
        size_correction_enabled=False,
        length_scale=1.0,
        width_scale=1.0,
        dims_before=None,
        dims_after=None,
        req_l_side=float('nan'),
        side_face_pts_count=0,
        var4_obb_dims_corrected=None,
        var4_obb_final=None,
    )

    R    = R_c2e.astype(np.float64);  t    = t_c2e.astype(np.float64)
    Rl2e = R_l2e.astype(np.float64);  tl2e = t_l2e.astype(np.float64)
    fx, fy     = float(K[0, 0]), float(K[1, 1])
    cx_k, cy_k = float(K[0, 2]), float(K[1, 2])
    verts      = verts_r3_in.astype(np.float64).copy()
    verts_orig = verts_r3_in.astype(np.float64).copy()

    def _bail(reason):
        info['status'] = reason
        info['var4_rejected_reason'] = reason
        return verts_r3_in.astype(np.float32), info

    # -- 1. Backproject in-mask anchors to ego --------------------------------
    if len(Z_lf) < 2:
        return _bail(f'skip_too_few_anchors ({len(Z_lf)})')
    Zlf     = Z_lf.astype(np.float64)
    li_cam  = np.stack([(u_lf - cx_k)*Zlf/fx, (v_lf - cy_k)*Zlf/fy, Zlf], axis=1)
    anchors = (R @ li_cam.T).T + t
    info['anchor_ego'] = anchors

    # -- 1.5. Anchor pre-clustering: keep dominant group ----------------------
    if len(anchors) >= anchor_dbscan_min:
        db_anc   = DBSCAN(eps=anchor_dbscan_eps,
                          min_samples=anchor_dbscan_min).fit(anchors[:, :2])
        anc_lbls = db_anc.labels_
    else:
        anc_lbls = np.zeros(len(anchors), dtype=int)

    unique_anc = sorted([l for l in set(anc_lbls) if l >= 0])
    anc_groups = []
    for lbl in unique_anc:
        pts  = anchors[anc_lbls == lbl]
        dist = float(np.linalg.norm(pts[:, :2].mean(0)))
        anc_groups.append((pts, len(pts), dist))

    rej_anc = []
    if len(anc_groups) > 1:
        anc_groups.sort(key=lambda g: (-g[1], g[2]))
        anchors_kept = anc_groups[0][0]
        for pts, _, _ in anc_groups[1:]:
            rej_anc.append(pts)
    elif len(anc_groups) == 1:
        anchors_kept = anc_groups[0][0]
    else:
        anchors_kept = anchors

    noise_anc = anchors[anc_lbls < 0]
    if len(noise_anc):
        rej_anc.append(noise_anc)

    info['anchor_groups_count']   = max(len(anc_groups), 1)
    info['kept_anchor_count']     = len(anchors_kept)
    info['rejected_anchor_count'] = len(anchors) - len(anchors_kept)
    info['anchor_kept_ego']       = anchors_kept
    info['anchor_rejected_ego']   = rej_anc

    if len(anchors_kept) < 2:
        return _bail(f'skip_too_few_kept_anchors ({len(anchors_kept)})')

    anchors_2d = anchors_kept[:, :2]   # shorthand used throughout

    # -- 2. Preliminary body OBB ----------------------------------------------
    verts_ego  = (R @ verts.T).T + t
    body_obb_0 = _v4_mesh_body_obb_ego(verts_ego)
    info['mesh_obb_before'] = body_obb_0

    # -- 3. Collect nearby full-cloud candidates (around kept anchors) ---------
    xyz_ego_full = (Rl2e @ xyz_lid.T).T + tl2e
    anc_tree     = cKDTree(anchors_2d)
    d_to_anc, _  = anc_tree.query(xyz_ego_full[:, :2])
    nearby_mask  = d_to_anc < cluster_radius
    pts_nearby   = xyz_ego_full[nearby_mask]
    info['nearby_ego']              = pts_nearby
    info['nearby_candidates_count'] = len(pts_nearby)

    if len(pts_nearby) < min_cluster_pts:
        return _bail(f'skip_too_few_nearby ({len(pts_nearby)})')

    # -- 4. Ground removal (anchor-centered, softer clearance) ----------------
    prelim_ctr = anchors_kept.mean(0)
    excl_r     = max(max(body_obb_0['dims'][0], body_obb_0['dims'][1]) * 0.55 + 0.3, 1.0)
    gnd = estimate_local_ground(
        xyz_lid, Rl2e, tl2e,
        obj_center_ego=prelim_ctr,
        crop_r=ground_crop_r, excl_r=excl_r,
        bot_frac=0.35, ransac_n=300, ransac_thresh=0.15,
        min_inliers=15, max_tilt_deg=15.,
    )
    info['ground_success'] = gnd['success']
    info['ground_z']       = gnd.get('ground_z')
    if gnd.get('inlier_mask') is not None and gnd['success']:
        info['ground_inliers_ego'] = gnd['ground_pts_ego'][gnd['inlier_mask']]
    ground_coef = gnd['plane_coeff'] if gnd['success'] else None
    ground_z    = gnd['ground_z']    if gnd['success'] else None

    if ground_coef is not None:
        a, b, cc = ground_coef
        z_plane  = pts_nearby[:, 0]*a + pts_nearby[:, 1]*b + cc
        non_gnd  = pts_nearby[:, 2] > z_plane + ground_clearance
    elif ground_z is not None:
        non_gnd  = pts_nearby[:, 2] > ground_z + ground_clearance
    else:
        gz_est   = float(anchors_kept[:, 2].min()) - ground_clearance
        ground_z = gz_est
        info['ground_z'] = gz_est
        non_gnd  = pts_nearby[:, 2] > gz_est + ground_clearance

    if ground_z is not None:
        non_gnd &= pts_nearby[:, 2] < ground_z + max_obj_height_m

    pts_ng = pts_nearby[non_gnd]
    info['ground_removed_ego'] = pts_ng
    info['face_support_counts']['n_nearby']       = len(pts_nearby)
    info['face_support_counts']['n_after_ground'] = len(pts_ng)

    if len(pts_ng) < min_cluster_pts:
        return _bail(f'skip_too_few_after_ground ({len(pts_ng)})')

    # -- 5. DBSCAN cluster; keep cluster connected to kept anchors ------------
    db     = DBSCAN(eps=dbscan_eps, min_samples=dbscan_min_samples).fit(pts_ng[:, :2])
    labels = db.labels_

    cluster_pts  = None
    best_lbl     = -1
    best_n       = 0
    rejected_cls = []

    for lbl in sorted(set(labels)):
        c_pts = pts_ng[labels == lbl]
        if lbl < 0:
            rejected_cls.append(c_pts)
            continue
        ext = max(float(np.ptp(c_pts[:, 0])), float(np.ptp(c_pts[:, 1])))
        if ext > max_cluster_extent:
            rejected_cls.append(c_pts)
            continue
        d_to_c, _ = cKDTree(c_pts[:, :2]).query(anchors_2d)
        connected  = bool((d_to_c < 1.5 * dbscan_eps).any())
        if connected and len(c_pts) >= min_cluster_pts and len(c_pts) > best_n:
            if best_lbl >= 0:
                rejected_cls.append(pts_ng[labels == best_lbl])
            best_lbl    = lbl
            best_n      = len(c_pts)
            cluster_pts = c_pts
        else:
            rejected_cls.append(c_pts)

    info['rejected_clusters_ego']  = rejected_cls
    info['rejected_cluster_count'] = sum(1 for c in rejected_cls if len(c) >= min_cluster_pts)

    if cluster_pts is None or len(cluster_pts) < min_cluster_pts:
        return _bail(f'skip_no_valid_cluster (best_n={best_n})')

    info['cluster_ego']          = cluster_pts
    info['cluster_points_count'] = len(cluster_pts)

    # -- 5.5. Filter anchors to only those near the biggest cluster -----------
    #   Anchors that belong to other smaller clusters are excluded from the
    #   optimisation so they cannot pull the OBB toward spurious detections.
    _cluster_2d_pre = cluster_pts[:, :2]
    if len(anchors_2d) and len(cluster_pts):
        _d_anc_cl, _  = cKDTree(_cluster_2d_pre).query(anchors_2d)
        _anc_opt_mask  = _d_anc_cl < 2.0 * dbscan_eps
        anchors_opt_2d = anchors_2d[_anc_opt_mask]
        anchors_excl   = anchors_kept[~_anc_opt_mask]
    else:
        _anc_opt_mask  = np.ones(len(anchors_2d), dtype=bool)
        anchors_opt_2d = anchors_2d.copy()
        anchors_excl   = np.zeros((0, 3))
    # Fallback: never leave optimisation with zero anchors
    if len(anchors_opt_2d) == 0:
        anchors_opt_2d = anchors_2d.copy()
        anchors_excl   = np.zeros((0, 3))
    info['anchor_opt_count']      = len(anchors_opt_2d)
    info['anchor_excluded_count'] = len(anchors_excl)
    info['anchor_excluded_ego']   = anchors_excl

    # -- 6. Starting mesh: VAR2 if available, else Z-corrected raw -----------
    vis0, _ = _extract_visible_verts(verts_orig, binary_mask, K, dilation_px, zbuf_tol)
    n_vis   = int(vis0.sum())
    info['n_vis_verts'] = n_vis

    if verts_var2 is not None:
        # Use VAR2 mesh as starting pose — Z+yaw correction already applied by VAR2.
        # Any (dx, dy, dyaw) found by VAR4 is a refinement ON TOP of VAR2.
        verts = verts_var2.astype(np.float64).copy()
        dz    = float((info_var2 or {}).get('delta_cam_z', 0.))
        info['yaw_prior_source']   = 'VAR2'
        info['var2_chamfer']       = float((info_var2 or {}).get('lidar_mesh_dist', float('nan')))
        info['lidar_mesh_dist_before'] = info['var2_chamfer']
    else:
        # Standard Z-front correction from raw mesh
        dz = 0.
        if n_vis >= 3:
            dz = float(np.clip(Zlf.min() - verts_orig[vis0, 2].min(),
                               -max_trans_z_m, max_trans_z_m))
        verts[:, 2] += dz
        info['yaw_prior_source'] = 'raw'
        if n_vis >= 3:
            d_b, _ = cKDTree(((R @ verts_orig.T).T + t)[vis0]).query(anchors_kept)
            nk_b   = max(3, int(len(d_b) * trim_frac))
            info['lidar_mesh_dist_before'] = float(np.mean(np.sort(d_b)[:nk_b]))

    info['delta_cam_z'] = dz

    # -- 7. Body OBB (post-Z) + shape classification --------------------------
    ve_z       = (R @ verts.T).T + t
    body_obb_z = _v4_mesh_body_obb_ego(ve_z)
    c2d        = body_obb_z['center'][:2]
    obb_al     = body_obb_z['axes'][:2, 0]
    obb_aw     = body_obb_z['axes'][:2, 1]
    obb_hl     = body_obb_z['dims'][0] / 2
    obb_hw     = body_obb_z['dims'][1] / 2
    faces      = _v4_obb_faces(body_obb_z)

    cluster_2d = cluster_pts[:, :2]
    cl_ctr_2d  = cluster_2d.mean(0)

    proj_l   = (cluster_2d - cl_ctr_2d) @ obb_al
    proj_w   = (cluster_2d - cl_ctr_2d) @ obb_aw
    spread_l = float(np.percentile(proj_l, 95) - np.percentile(proj_l, 5))
    spread_w = float(np.percentile(proj_w, 95) - np.percentile(proj_w, 5))

    l_shape_spread_ok = spread_l >= l_spread_min and spread_w >= l_spread_min
    if spread_l > l_spread_ratio * (spread_w + 0.1):
        shape_cls = 'I_side'
    elif spread_w > l_spread_ratio * (spread_l + 0.1):
        shape_cls = 'I_front_back'
    elif l_shape_spread_ok:
        shape_cls = 'L_shape'
    else:
        # I_ambiguous: one-face-like cluster where front/back vs side is not clear
        # from spread. Both I-shaped hypotheses are tested; L-shape not allowed
        # unless both-axis support is strong.
        shape_cls = 'I_ambiguous'

    info['shape_classification'] = shape_cls
    info['l_shape_allowed']      = l_shape_spread_ok
    info['face_support_counts'].update({'spread_L': round(spread_l, 3),
                                        'spread_W': round(spread_w, 3)})

    ego_dir         = -cl_ctr_2d / (np.linalg.norm(cl_ctr_2d) + 1e-9)
    face_ego_aligns = [float(np.dot(fn, ego_dir)) for _, fn, _ in faces]
    ego_vis_fi      = int(np.argmax(face_ego_aligns))
    info['ego_visible_face_prior'] = faces[ego_vis_fi][2]

    # -- 7.5. L-shape strength + mode-dependent yaw penalty ------------------
    l_shape_type     = ''
    l_shape_strength = float('nan')
    if shape_cls == 'L_shape' and l_shape_spread_ok:
        # Quick support count for each axis to classify strong vs weak L
        _sup_lp, _ = _v4_face_support_pts(cluster_2d, c2d,  obb_al, face_support_frac, face_support_band_m)
        _sup_ln, _ = _v4_face_support_pts(cluster_2d, c2d, -obb_al, face_support_frac, face_support_band_m)
        _sup_wp, _ = _v4_face_support_pts(cluster_2d, c2d,  obb_aw, face_support_frac, face_support_band_m)
        _sup_wn, _ = _v4_face_support_pts(cluster_2d, c2d, -obb_aw, face_support_frac, face_support_band_m)
        n_l_best        = max(len(_sup_lp), len(_sup_ln))
        n_w_best        = max(len(_sup_wp), len(_sup_wn))
        n_primary_est   = max(n_l_best, n_w_best)
        n_secondary_est = min(n_l_best, n_w_best)
        l_shape_strength = n_secondary_est / max(n_primary_est, 1)
        if (l_shape_strength >= l_shape_strength_threshold and
                n_secondary_est >= 2 * min_face_support_pts):
            l_shape_type = 'strong_L'
        else:
            l_shape_type = 'weak_L'
    info['l_shape_type']     = l_shape_type
    info['l_shape_strength'] = l_shape_strength

    # Yaw penalty: stronger for I-shaped/weak cases, relaxed for strong L
    if shape_cls in ('I_side', 'I_front_back', 'I_ambiguous'):
        yaw_pen_use = yaw_penalty_1face
    elif l_shape_type == 'strong_L':
        yaw_pen_use = yaw_penalty_strong_l
    else:  # weak_L or no L-shape
        yaw_pen_use = yaw_penalty_weak_l

    # Full-cluster baseline (for reference / L-shape acceptance)
    r_all         = np.stack([np.abs(_v4_face_residuals(cluster_2d, fc, fn, 0., 0., 0., c2d))
                               for fc, fn, _ in faces], axis=1)
    baseline_loss = _v4_trimmed_mean_abs(r_all.min(1), trim_frac)
    info['face_loss_before'] = baseline_loss

    # -- 8. Hypothesis testing (filtered by shape classification) -------------
    hypotheses = {
        'front_back': [(0,), (1,)],
        'side':       [(2,), (3,)],
        'L_shape':    [(0, 2), (0, 3), (1, 2), (1, 3)],
    }

    if shape_cls == 'I_front_back':
        allowed_modes = {'front_back'}
    elif shape_cls == 'I_side':
        allowed_modes = {'side'}
    elif shape_cls == 'L_shape' and l_shape_spread_ok:
        allowed_modes = {'front_back', 'side', 'L_shape'}
    else:
        allowed_modes = {'front_back', 'side'}   # I_ambiguous

    mode_results = {}

    for mode, fi_list in hypotheses.items():
        if mode not in allowed_modes:
            continue
        best_mode_l, best_mode_r = float('inf'), None
        for fi_tuple in fi_list:
            if len(fi_tuple) == 1:
                fi = fi_tuple[0]
                fc0_h, fn0_h, _ = faces[fi]

                # Extract face-support points (outer boundary along fn)
                sup_pts, sup_info = _v4_face_support_pts(
                    cluster_2d, c2d, fn0_h,
                    support_frac=face_support_frac,
                    band_m=face_support_band_m)
                if len(sup_pts) < min_face_support_pts:
                    continue  # not enough surface points for this candidate

                # Per-face baseline on support points (no transform applied yet)
                base_r_sup  = _v4_face_residuals(sup_pts, fc0_h, fn0_h, 0., 0., 0., c2d)
                baseline_sup = _v4_trimmed_mean_abs(base_r_sup, trim_frac)

                dx, dy, dyaw, pen_l, raw_l = _v4_fit_1face(
                    sup_pts, cluster_2d, fc0_h, fn0_h, c2d,
                    max_trans_xy_m, max_yaw_rad,
                    trim_frac, yaw_pen_use, along_penalty,
                    obb_al, obb_aw, obb_hl, obb_hw,
                    anchor_containment_weight, cluster_containment_weight,
                    tangent_center_weight, anchors_opt_2d)

                # Post-hoc: tangent offset on support-point median
                R2_p     = _v4_rot2(dyaw)
                fn_r_p   = R2_p @ fn0_h
                ft_r_p   = np.array([-fn_r_p[1], fn_r_p[0]])
                fc_r_p   = c2d + R2_p @ (fc0_h - c2d) + np.array([dx, dy])
                tang_off = float((np.median(sup_pts, axis=0) - fc_r_p) @ ft_r_p)

                # Corner-guard on support-point median
                half_tang = obb_hw if fi in (0, 1) else obb_hl
                if abs(tang_off) > tangent_max_frac * half_tang:
                    continue

                # Post-hoc containment on full cluster
                cl_cont, cl_ins = _v4_containment_stats(
                    cluster_2d, dx, dy, dyaw, c2d,
                    obb_al, obb_aw, obb_hl, obb_hw, trim_frac)

                ego_pen  = max(0., -face_ego_aligns[fi]) * ego_face_weight
                select_l = pen_l + ego_pen
                if select_l < best_mode_l:
                    best_mode_l = select_l
                    best_mode_r = dict(
                        dx=dx, dy=dy, dyaw=dyaw,
                        penalised_loss=pen_l, raw_loss=raw_l,
                        containment_loss=cl_cont, inside_ratio=cl_ins,
                        tangent_offset=tang_off,
                        selection_loss=select_l,
                        face_indices=[fi],
                        face_centers=[fc0_h],
                        face_normals=[fn0_h],
                        face_labels=[faces[fi][2]],
                        ego_align=face_ego_aligns[fi],
                        # face-support metadata
                        support_pts=sup_pts,
                        support_info=sup_info,
                        baseline_support_loss=baseline_sup,
                    )
            else:
                fi1, fi2 = fi_tuple
                fc1_h, fn1_h, _ = faces[fi1]
                fc2_h, fn2_h, _ = faces[fi2]

                # Per-face support counts to determine primary/secondary
                _sup1, _si1 = _v4_face_support_pts(cluster_2d, c2d, fn1_h,
                                                    face_support_frac, face_support_band_m)
                _sup2, _si2 = _v4_face_support_pts(cluster_2d, c2d, fn2_h,
                                                    face_support_frac, face_support_band_m)
                n1, n2 = len(_sup1), len(_sup2)

                # Order so fc1=primary (more support), fc2=secondary
                if n2 > n1:
                    fi1, fi2        = fi2, fi1
                    fc1_h, fc2_h    = fc2_h, fc1_h
                    fn1_h, fn2_h    = fn2_h, fn1_h
                    n1, n2          = n2, n1
                    _si1, _si2      = _si2, _si1

                # Determine secondary weight from L-shape strength
                _l_str = n2 / max(n1, 1)
                if _l_str >= l_shape_strength_threshold and n2 >= 2 * min_face_support_pts:
                    sec_w = 1.0   # strong L: symmetric
                else:
                    sec_w = weak_l_secondary_weight  # weak L: secondary de-weighted

                dx, dy, dyaw, pen_l, raw_l = _v4_fit_L(
                    cluster_2d, fc1_h, fn1_h, fc2_h, fn2_h, c2d,
                    max_trans_xy_m, max_yaw_rad,
                    trim_frac, yaw_pen_use,
                    obb_al, obb_aw, obb_hl, obb_hw,
                    anchor_containment_weight, cluster_containment_weight, anchors_opt_2d,
                    secondary_weight=sec_w)

                # Per-face losses at solution for debug
                r1_sol = _v4_face_residuals(cluster_2d, fc1_h, fn1_h, dx, dy, dyaw, c2d)
                r2_sol = _v4_face_residuals(cluster_2d, fc2_h, fn2_h, dx, dy, dyaw, c2d)
                pf_loss = _v4_trimmed_mean_abs(r1_sol, trim_frac)
                sf_loss = _v4_trimmed_mean_abs(r2_sol, trim_frac)

                cl_cont, cl_ins = _v4_containment_stats(
                    cluster_2d, dx, dy, dyaw, c2d,
                    obb_al, obb_aw, obb_hl, obb_hw, trim_frac)
                ego_pen  = (max(0., -face_ego_aligns[fi1]) +
                            max(0., -face_ego_aligns[fi2])) * ego_face_weight
                select_l = pen_l + ego_pen
                if select_l < best_mode_l:
                    best_mode_l = select_l
                    best_mode_r = dict(
                        dx=dx, dy=dy, dyaw=dyaw,
                        penalised_loss=pen_l, raw_loss=raw_l,
                        containment_loss=cl_cont, inside_ratio=cl_ins,
                        tangent_offset=0.0,
                        selection_loss=select_l,
                        face_indices=[fi1, fi2],
                        face_centers=[fc1_h, fc2_h],
                        face_normals=[fn1_h, fn2_h],
                        face_labels=[faces[fi1][2], faces[fi2][2]],
                        ego_align=(face_ego_aligns[fi1] + face_ego_aligns[fi2]) / 2,
                        support_pts=None,
                        support_info=None,
                        baseline_support_loss=baseline_loss,
                        # L-shape specific
                        primary_face_support_count=n1,
                        secondary_face_support_count=n2,
                        primary_face_loss=pf_loss,
                        secondary_face_loss=sf_loss,
                        secondary_face_weight=sec_w,
                    )
        if best_mode_r is not None:
            mode_results[mode] = best_mode_r

    info['mode_results'] = mode_results

    if not mode_results:
        return _bail('skip_no_mode_converged')

    # -- 9. Select best mode with L-shape validation and fallback ---------------
    def _do_l_shape_check(mr, results):
        """Run L-shape post-fit validation; return (mode, result, rejected_reason)."""
        r1_v = _v4_face_residuals(cluster_2d, mr['face_centers'][0], mr['face_normals'][0],
                                   mr['dx'], mr['dy'], mr['dyaw'], c2d)
        r2_v = _v4_face_residuals(cluster_2d, mr['face_centers'][1], mr['face_normals'][1],
                                   mr['dx'], mr['dy'], mr['dyaw'], c2d)
        assign_v = (np.abs(r2_v) < np.abs(r1_v)).astype(int)
        n_f1 = int((assign_v == 0).sum())
        n_f2 = int((assign_v == 1).sum())
        min_pts = max(4, int(len(cluster_2d) * min_l_face_pts_frac))
        info['l_shape_face_counts'] = {'n_f1': n_f1, 'n_f2': n_f2, 'min_required': min_pts}
        if n_f1 < min_pts or n_f2 < min_pts:
            reason = f'face_pts_too_few (f1={n_f1}, f2={n_f2} < {min_pts})'
            info['l_shape_rejected_reason'] = reason
            no_L = {k: v for k, v in results.items() if k != 'L_shape'}
            if no_L:
                nm = min(no_L, key=lambda m: no_L[m]['selection_loss'])
                return nm, no_L[nm]
            return None, None
        return 'L_shape', mr

    # Try modes in selection-loss order with acceptance check
    ordered_modes = sorted(mode_results, key=lambda m: mode_results[m]['selection_loss'])
    best_mode, best, accepted = None, None, False

    for try_mode in ordered_modes:
        candidate = mode_results[try_mode]

        # L-shape validation (may swap to a simpler mode)
        if try_mode == 'L_shape':
            try_mode, candidate = _do_l_shape_check(candidate, mode_results)
            if try_mode is None:
                continue  # L-shape + all fallbacks failed

        # Acceptance check: use per-face support baseline for 1-face, full-cluster for L
        baseline_use = candidate.get('baseline_support_loss', baseline_loss)
        loss_after_c = candidate['raw_loss']
        if np.isfinite(baseline_use) and loss_after_c > baseline_use * loss_slack:
            continue  # this mode didn't improve; try next

        best_mode, best, accepted = try_mode, candidate, True
        break

    if not accepted:
        tried = ', '.join(f'{m}:{mode_results[m]["raw_loss"]:.4f}' for m in ordered_modes)
        reason = f'loss_not_improved_all_modes ({tried})'
        info['status'] = f'rejected: {reason}'
        info['var4_rejected_reason'] = reason
        info['var4_obb_final'] = body_obb_0
        return verts_r3_in.astype(np.float32), info

    dx_opt, dy_opt, dyaw = best['dx'], best['dy'], best['dyaw']
    loss_after = best['raw_loss']

    # Store face-support debug for selected mode
    if best.get('support_info'):
        si = best['support_info']
        info['face_support_count']    = si['face_support_count']
        info['face_support_fraction'] = si['face_support_fraction']
        info['face_support_band_m']   = si['face_support_band_m']
        info['face_support_mode']     = si['face_support_mode']
    info['face_loss_before_support'] = best.get('baseline_support_loss', baseline_loss)
    info['face_loss_after_support']  = loss_after

    info['var4_mode']       = best_mode
    info['selected_faces']  = best['face_labels']
    info['face_loss_after'] = loss_after
    info['dyaw']            = dyaw
    info['dxy']             = np.array([dx_opt, dy_opt])
    info['delta_ego_xy']    = np.array([dx_opt, dy_opt])
    info['delta_yaw_deg']   = float(np.degrees(dyaw))

    # Final tangent offset (for debug)
    sup_pts_sel = best.get('support_pts')
    if len(best['face_centers']) == 1:
        fc0_b, fn0_b = best['face_centers'][0], best['face_normals'][0]
        R2_b  = _v4_rot2(dyaw)
        fn_rb = R2_b @ fn0_b;  ft_rb = np.array([-fn_rb[1], fn_rb[0]])
        fc_rb = c2d + R2_b @ (fc0_b - c2d) + np.array([dx_opt, dy_opt])
        ref_pts = sup_pts_sel if sup_pts_sel is not None and len(sup_pts_sel) else cluster_2d
        info['final_tangent_offset'] = float((np.median(ref_pts, 0) - fc_rb) @ ft_rb)
    else:
        info['final_tangent_offset'] = 0.0

    # Residual lines for debug (always on full cluster for visualization)
    R2    = _v4_rot2(dyaw)
    fcs   = best['face_centers'];  fns = best['face_normals']
    a_fcs = [c2d + R2 @ (fc - c2d) + np.array([dx_opt, dy_opt]) for fc in fcs]
    a_fns = [R2 @ fn for fn in fns]
    if len(fcs) == 1:
        r_dbg  = _v4_face_residuals(cluster_2d, fcs[0], fns[0], dx_opt, dy_opt, dyaw, c2d)
        assign = np.zeros(len(cluster_2d), dtype=int)
    else:
        r1 = _v4_face_residuals(cluster_2d, fcs[0], fns[0], dx_opt, dy_opt, dyaw, c2d)
        r2 = _v4_face_residuals(cluster_2d, fcs[1], fns[1], dx_opt, dy_opt, dyaw, c2d)
        assign = (np.abs(r2) < np.abs(r1)).astype(int)
        r_dbg  = np.where(assign == 0, r1, r2)
    info['alignment_residuals']    = r_dbg
    info['alignment_face_centers'] = a_fcs
    info['alignment_face_normals'] = a_fns
    info['alignment_assign']       = assign
    info['face_support_pts']       = sup_pts_sel  # for visualization

    # -- 10. (acceptance already enforced in step 9 loop) ----------------------

    # -- 11. Apply BEV correction to mesh -------------------------------------
    ve_yw = (R @ verts.T).T + t
    cr    = ve_yw.mean(0)
    cd, sd = np.cos(dyaw), np.sin(dyaw)
    Ry3   = np.array([[cd, -sd, 0.], [sd, cd, 0.], [0., 0., 1.]])
    ve_yw = (Ry3 @ (ve_yw - cr).T).T + cr
    ve_yw[:, :2] += [dx_opt, dy_opt]
    verts = (R.T @ (ve_yw - t).T).T

    body_obb_f = _v4_mesh_body_obb_ego((R @ verts.T).T + t)
    info['mesh_obb_after'] = body_obb_f

    vis_f, _ = _extract_visible_verts(verts, binary_mask, K, dilation_px, zbuf_tol)
    if vis_f.sum() >= 3:
        d_f, _ = cKDTree(((R @ verts.T).T + t)[vis_f]).query(anchors_kept)
        nk = max(3, int(len(d_f) * trim_frac))
        info['lidar_mesh_dist'] = float(np.mean(np.sort(d_f)[:nk]))

    # -- 12. Size correction (OBB dims only, no mesh deformation) -------------
    c2d_f   = body_obb_f['center'][:2]
    al_f    = body_obb_f['axes'][:2, 0]
    aw_f    = body_obb_f['axes'][:2, 1]
    dp_f    = cluster_2d - c2d_f
    loc_l_f = dp_f @ al_f;  loc_w_f = dp_f @ aw_f
    # Use p2-p98 to capture more of the cluster extent (previously p5-p95)
    cl_lo_l, cl_hi_l = np.percentile(loc_l_f, [2, 98])
    cl_lo_w, cl_hi_w = np.percentile(loc_w_f, [2, 98])
    req_l   = (cl_hi_l - cl_lo_l) + 2 * size_margin
    req_w   = (cl_hi_w - cl_lo_w) + 2 * size_margin
    cur_l   = body_obb_f['dims'][0];  cur_w = body_obb_f['dims'][1]

    # Size correction disabled — handled separately
    size_corr_active = False
    l_shape_active   = (best_mode == 'L_shape' and not info.get('l_shape_rejected_reason'))

    # For L-shape: also estimate required length from side-face assigned cluster points
    req_l_side   = float('nan')
    side_pts_cnt = 0
    if l_shape_active and assign is not None and len(best.get('face_indices', [])) == 2:
        fi1, fi2 = best['face_indices']
        # fi in (2,3) = W-face (side face); fi in (0,1) = L-face (front/back)
        if fi2 in (2, 3):
            side_pts = cluster_2d[assign == 1]
        elif fi1 in (2, 3):
            side_pts = cluster_2d[assign == 0]
        else:
            side_pts = cluster_2d
        side_pts_cnt = len(side_pts)
        if side_pts_cnt >= min_side_face_pts:
            loc_l_side = (side_pts - c2d_f) @ al_f
            req_l_side = float(np.percentile(loc_l_side, 98) - np.percentile(loc_l_side, 2)) + 2 * side_face_length_margin

    info['req_l_side']          = req_l_side
    info['side_face_pts_count'] = side_pts_cnt

    sc_l, sc_w = 1.0, 1.0
    if size_corr_active:
        # Bidirectional scaling: allow both enlargement and shrinkage within limits
        req_l_eff = req_l
        if l_shape_active and np.isfinite(req_l_side):
            req_l_eff = max(req_l, req_l_side)
        sc_l = float(np.clip(req_l_eff / cur_l, l_scale_min, l_scale_max))
        sc_w = float(np.clip(req_w / cur_w,    w_scale_min, w_scale_max))

        rls_str = f'{req_l_side:.2f}' if np.isfinite(req_l_side) else 'n/a'
        print(f'  VAR4 size_corr: req_l={req_l:.2f} req_l_side={rls_str} '
              f'req_l_eff={req_l_eff:.2f} cur_l={cur_l:.2f} sc_l={sc_l:.3f}  '
              f'req_w={req_w:.2f} cur_w={cur_w:.2f} sc_w={sc_w:.3f}  '
              f'enabled={abs(sc_l-1.)>0.01 or abs(sc_w-1.)>0.01}  side_pts={side_pts_cnt}')

        if abs(sc_l - 1.) > 0.01 or abs(sc_w - 1.) > 0.01:
            new_dims = np.array([cur_l * sc_l, cur_w * sc_w, body_obb_f['dims'][2]])
            corr_obb = dict(center=body_obb_f['center'].copy(),
                            axes=body_obb_f['axes'].copy(),
                            dims=new_dims)
            info['size_correction_enabled'] = True
            info['length_scale']            = float(sc_l)
            info['width_scale']             = float(sc_w)
            info['dims_before']             = body_obb_f['dims'].copy()
            info['dims_after']              = new_dims
            info['var4_obb_dims_corrected'] = corr_obb

    # Final effective OBB: corrected if size correction active, else body
    info['var4_obb_final'] = (info['var4_obb_dims_corrected']
                              if info['size_correction_enabled'] else body_obb_f)

    # -- 13. Inside-ratio check on var4_obb_final (after size correction) -----
    fin_obb = info['var4_obb_final']
    fin_c2d = fin_obb['center'][:2]
    fin_al  = fin_obb['axes'][:2, 0]
    fin_aw  = fin_obb['axes'][:2, 1]
    fin_hl  = fin_obb['dims'][0] / 2
    fin_hw  = fin_obb['dims'][1] / 2

    anc_in_r, anc_mse = _v4_obb_inside(anchors_opt_2d, fin_c2d, fin_al, fin_aw, fin_hl, fin_hw)
    cl_in_r,  cl_mse  = _v4_obb_inside(cluster_2d, fin_c2d, fin_al, fin_aw, fin_hl, fin_hw)

    # Compute outside counts (using cluster-filtered anchors)
    _anc_exc_fin = _v4_obb_excess(anchors_opt_2d, fin_c2d, fin_al, fin_aw, fin_hl, fin_hw)
    _cl_exc_fin  = _v4_obb_excess(cluster_2d, fin_c2d, fin_al, fin_aw, fin_hl, fin_hw)

    info['final_anchor_inside_ratio']  = anc_in_r
    info['final_cluster_inside_ratio'] = cl_in_r
    info['final_anchor_containment']   = anc_mse
    info['final_cluster_containment']  = cl_mse
    info['anchor_inside_ratio']        = anc_in_r
    info['cluster_inside_ratio']       = cl_in_r
    info['anchor_outside_count']       = int((_anc_exc_fin > 0).sum())
    info['cluster_outside_count']      = int((_cl_exc_fin  > 0).sum())
    info['anchor_containment_loss']    = anc_mse
    info['cluster_containment_loss']   = cl_mse

    if anc_in_r < anchor_inside_ratio_floor:
        reason = f'anchor_inside_ratio_too_low ({anc_in_r:.0%} < floor {anchor_inside_ratio_floor:.0%})'
        info['status'] = f'rejected: {reason}'
        info['var4_rejected_reason'] = reason
        return verts.astype(np.float32), info
    if anc_in_r < anchor_inside_ratio_threshold:
        info['status'] += f'|low_anc_inside({anc_in_r:.0%})'
    if cl_in_r < cluster_inside_ratio_threshold:
        info['status'] += f'|low_cl_inside({cl_in_r:.0%})'

    flags = []
    if abs(abs(dz)           - max_trans_z_m)  < 1e-4: flags.append('z')
    if abs(np.hypot(dx_opt, dy_opt) - max_trans_xy_m) < 0.05: flags.append('xy')
    if abs(abs(dyaw)          - max_yaw_rad)   < 0.01:  flags.append('yaw')
    if flags:
        info['status'] += f'|hit_bound({",".join(flags)})'

    anc_in_str = f'{anc_in_r:.0%}' if np.isfinite(anc_in_r) else 'n/a'
    cl_in_str  = f'{cl_in_r:.0%}'  if np.isfinite(cl_in_r)  else 'n/a'

    # -- 14. VAR2 fallback check ----------------------------------------------
    info['var4_chamfer']       = info['lidar_mesh_dist']
    info['yaw_delta_from_var2'] = info['delta_yaw_deg']  # extra yaw on top of VAR2 prior
    if (verts_var2 is not None
            and np.isfinite(info['var2_chamfer'])
            and np.isfinite(info['var4_chamfer'])):
        var4_improves = info['var4_chamfer'] < info['var2_chamfer'] * (1.0 + var2_improvement_margin)
        info['var4_improves_var2'] = var4_improves
        if not var4_improves:
            info['fallback_used'] = True
            info['status'] += '|var2_fallback'
            print(f'  VAR4 mode={best_mode}  shape={shape_cls}  '
                  f'var4_ch={info["var4_chamfer"]:.4f} >= var2_ch={info["var2_chamfer"]:.4f}  '
                  f'=> VAR2 fallback  status={info["status"]}')
            return verts_var2.astype(np.float32), info
    else:
        info['var4_improves_var2'] = True  # no comparison possible; keep VAR4

    print(f'  VAR4 mode={best_mode}  shape={shape_cls}  faces={best["face_labels"]}  '
          f'anchors={info["kept_anchor_count"]}/{info["anchor_points_count"]}  '
          f'cluster={info["cluster_points_count"]}  '
          f'loss {baseline_loss:.4f}->{loss_after:.4f}  '
          f'anc_in={anc_in_str}({info["anchor_outside_count"]}out)  '
          f'cl_in={cl_in_str}({info["cluster_outside_count"]}out)  '
          f'tang={info["final_tangent_offset"]:+.2f}m  '
          f'dyaw={info["delta_yaw_deg"]:+.1f}deg  '
          f'dxy=({dx_opt:+.3f},{dy_opt:+.3f})m  dz={dz:+.3f}m  '
          f'prior={info["yaw_prior_source"]}  '
          f'var4_improves={info["var4_improves_var2"]}  '
          f'status={info["status"]}')

    return verts.astype(np.float32), info


# -- Debug visualisation ------------------------------------------------------

def visualize_var4_debug(
    verts_before, verts_after, info,
    binary_mask, img_rgb, K,
    u_lf, v_lf, Z_lf,
    R_c2e, t_c2e, R_l2e, t_l2e,
    xl, xr, yt, yb,
    label='VAR4', frame_idx=0, obj_idx=0,
):
    """3x3 debug panel for VAR4."""
    R = R_c2e.astype(np.float64);  t = t_c2e.astype(np.float64)

    def _to_ego(v): return (R @ v.astype(np.float64).T).T + t

    def _draw_obb(ax, obb, color, lw=2, tag=None, ls='-', alpha=1.0):
        if obb is None: return
        c  = obb['center'][:2]
        al = obb['axes'][:2, 0];  aw = obb['axes'][:2, 1]
        hl = obb['dims'][0] / 2;  hw = obb['dims'][1] / 2
        corners = np.array([c+hl*al+hw*aw, c+hl*al-hw*aw,
                             c-hl*al-hw*aw, c-hl*al+hw*aw])
        ax.plot(list(corners[:, 0])+[corners[0, 0]],
                list(corners[:, 1])+[corners[0, 1]],
                color=color, lw=lw, ls=ls, alpha=alpha, zorder=4)
        ax.scatter([c[0]], [c[1]], c=[color], s=30, zorder=5, alpha=alpha)
        if tag: ax.annotate(tag, xy=c, fontsize=7, color=color, zorder=7)

    def _draw_face(ax, fc, fn, half_len, color='magenta', lw=2.5):
        ft = np.array([-fn[1], fn[0]])
        ax.plot([fc[0]-half_len*ft[0], fc[0]+half_len*ft[0]],
                [fc[1]-half_len*ft[1], fc[1]+half_len*ft[1]],
                color=color, lw=lw, zorder=6)
        ax.annotate('', xy=fc + fn*0.45, xytext=fc,
                    arrowprops=dict(arrowstyle='->', color=color, lw=1.5))

    ve_bef = _to_ego(verts_before)
    ve_aft = _to_ego(verts_after)

    fig, axes = plt.subplots(3, 3, figsize=(27, 21))
    (ax00, ax01, ax02), (ax10, ax11, ax12), (ax20, ax21, ax22) = axes

    rej_cols     = ['tomato', 'orangered', 'gold', 'hotpink', 'plum', 'coral']
    anc_rej_cols = ['salmon', 'sandybrown', 'khaki']

    anchors_kept = info.get('anchor_kept_ego')
    anchors_rej  = info.get('anchor_rejected_ego', [])
    anchors_excl = info.get('anchor_excluded_ego')  # anchors excluded from optimisation
    nearby       = info.get('nearby_ego')
    ng_pts       = info.get('ground_removed_ego')
    cl_pts       = info.get('cluster_ego')
    rej_cls      = info.get('rejected_clusters_ego', [])
    obb_bef      = info.get('mesh_obb_before')
    obb_aft      = info.get('mesh_obb_after')
    obb_final    = info.get('var4_obb_final') or obb_aft
    size_corr    = info.get('size_correction_enabled', False)
    dxy          = info['dxy'];  mag_xy = float(np.linalg.norm(dxy))
    anc_in_r     = info.get('anchor_inside_ratio', float('nan'))
    cl_in_r      = info.get('cluster_inside_ratio', float('nan'))

    anc_in_str = f'{anc_in_r:.0%}' if np.isfinite(anc_in_r) else 'n/a'
    cl_in_str  = f'{cl_in_r:.0%}'  if np.isfinite(cl_in_r)  else 'n/a'

    # Determine anchor and cluster inside/outside status for coloring
    anc_inside_mask = None
    cl_inside_mask  = None
    if obb_final is not None:
        _fc2d = obb_final['center'][:2]
        _fal  = obb_final['axes'][:2, 0]
        _faw  = obb_final['axes'][:2, 1]
        _fhl  = obb_final['dims'][0] / 2
        _fhw  = obb_final['dims'][1] / 2
        if anchors_kept is not None and len(anchors_kept):
            _n_opt = info.get('anchor_opt_count', len(anchors_kept))
            _opt_anc_2d = anchors_kept[:_n_opt, :2]
            _aexc = _v4_obb_excess(_opt_anc_2d, _fc2d, _fal, _faw, _fhl, _fhw)
            anc_inside_mask = (_aexc == 0.)
        if cl_pts is not None and len(cl_pts):
            _cexc = _v4_obb_excess(cl_pts[:, :2], _fc2d, _fal, _faw, _fhl, _fhw)
            cl_inside_mask = (_cexc == 0.)

    # -- [0,0] Camera: before + all in-mask anchors --------------------------
    img_b = draw_mesh_on_image(img_rgb, verts_before, K, color=(255, 100, 0))
    ax00.imshow(img_b[yt:yb, xl:xr])
    if len(Z_lf):
        ax00.scatter(u_lf-xl, v_lf-yt, c=Z_lf, cmap='cool', s=25, alpha=0.9,
                     zorder=5, edgecolors='teal', linewidths=0.4)
    ax00.set_title('Before VAR4 (orange) + in-mask anchors (cool=near)', fontsize=9)
    ax00.axis('off')

    # -- [0,1] Camera: after VAR4 --------------------------------------------
    img_a = draw_mesh_on_image(img_rgb, verts_after, K, color=(50, 220, 80))
    ax01.imshow(img_a[yt:yb, xl:xr])
    ax01.set_title(
        f'After {label} (green)  mode={info["var4_mode"]}\n'
        f'shape={info["shape_classification"]}  faces={info["selected_faces"]}  '
        f'dZ={info["delta_cam_z"]:+.3f}m  dXY={mag_xy:.3f}m  dyaw={info["delta_yaw_deg"]:+.1f}deg\n'
        f'anc_inside={anc_in_str}  cl_inside={cl_in_str}  status: {info["status"]}',
        fontsize=8)
    ax01.axis('off')

    # -- [0,2] Anchor pre-clustering BEV -------------------------------------
    for ai, rj in enumerate(anchors_rej):
        if len(rj):
            ax02.scatter(rj[:,0], rj[:,1], c=anc_rej_cols[ai % len(anc_rej_cols)],
                         s=18, alpha=0.8, marker='x', zorder=5,
                         label=f'Rej anc grp {ai+1} ({len(rj)})')
    if anchors_kept is not None and len(anchors_kept):
        # Split into opt (used in optimisation) vs excluded anchors
        _n_opt = info.get('anchor_opt_count', len(anchors_kept))
        ax02.scatter(anchors_kept[:_n_opt, 0], anchors_kept[:_n_opt, 1],
                     c='cyan', s=30, alpha=1., edgecolors='teal', zorder=7,
                     label=f'Opt anchors ({_n_opt})')
    if anchors_excl is not None and len(anchors_excl):
        ax02.scatter(anchors_excl[:,0], anchors_excl[:,1],
                     c='hotpink', s=35, alpha=0.85, marker='x', linewidths=1.5, zorder=8,
                     label=f'Excl from opt ({len(anchors_excl)})')
    if nearby is not None and len(nearby):
        ax02.scatter(nearby[:,0], nearby[:,1], c='lightgray', s=2, alpha=0.25,
                     zorder=1, label=f'Nearby ({len(nearby)})')
    _draw_obb(ax02, obb_bef, 'orange', tag='body OBB')
    ax02.set_aspect('equal'); ax02.grid(alpha=0.3)
    ax02.set_xlabel('Ego X (m)'); ax02.set_ylabel('Ego Y (m)')
    _bobb = obb_bef or {}
    _blp  = _bobb.get('len_pct', '?');  _bwp = _bobb.get('wid_pct', '?')
    ax02.set_title(
        f'Anchor pre-clustering  groups={info["anchor_groups_count"]}\n'
        f'kept={info["kept_anchor_count"]}  rej={info["rejected_anchor_count"]}\n'
        f'opt={info.get("anchor_opt_count","-")}  excl={info.get("anchor_excluded_count",0)}\n'
        f'body OBB: len_pct={_blp}  wid_pct={_bwp}', fontsize=8)
    ax02.legend(loc='upper right', fontsize=6, facecolor='k', labelcolor='w', framealpha=0.7)

    # -- [1,0] BEV: ground removal + DBSCAN ----------------------------------
    if nearby is not None and len(nearby):
        ax10.scatter(nearby[:,0], nearby[:,1], c='lightgray', s=2, alpha=0.2, zorder=1)
    if ng_pts is not None and len(ng_pts):
        ax10.scatter(ng_pts[:,0], ng_pts[:,1], c='steelblue', s=3, alpha=0.4,
                     zorder=2, label=f'Ground-removed ({len(ng_pts)})')
    for ci, rc in enumerate(rej_cls):
        if not len(rc): continue
        ax10.scatter(rc[:,0], rc[:,1], c=rej_cols[ci % len(rej_cols)],
                     s=4, alpha=0.6, zorder=3, label=f'DBSCAN rej {ci+1} ({len(rc)})')
    if cl_pts is not None and len(cl_pts):
        if cl_inside_mask is not None:
            _cl_in  = cl_pts[cl_inside_mask];   _cl_out = cl_pts[~cl_inside_mask]
            if len(_cl_in):
                ax10.scatter(_cl_in[:,0],  _cl_in[:,1],  c='yellow', s=8, alpha=0.9,
                             zorder=5, label=f'Cl inside ({len(_cl_in)})')
            if len(_cl_out):
                ax10.scatter(_cl_out[:,0], _cl_out[:,1], c='darkorange', s=14, alpha=0.85,
                             zorder=6, marker='x', linewidths=1.2,
                             label=f'Cl outside ({len(_cl_out)})')
        else:
            ax10.scatter(cl_pts[:,0], cl_pts[:,1], c='yellow', s=8, alpha=0.9,
                         zorder=5, label=f'Kept cluster ({len(cl_pts)} pts)')
    if anchors_kept is not None and len(anchors_kept):
        _n_opt = info.get('anchor_opt_count', len(anchors_kept))
        ax10.scatter(anchors_kept[:_n_opt, 0], anchors_kept[:_n_opt, 1],
                     c='cyan', s=22, alpha=1., edgecolors='teal', zorder=7,
                     label=f'Opt anchors ({_n_opt})')
    if anchors_excl is not None and len(anchors_excl):
        ax10.scatter(anchors_excl[:,0], anchors_excl[:,1],
                     c='hotpink', s=30, alpha=0.85, marker='x', linewidths=1.5, zorder=8,
                     label=f'Excl from opt ({len(anchors_excl)})')
    _draw_obb(ax10, obb_bef, 'orange', tag='body OBB before')
    ax10.set_aspect('equal'); ax10.grid(alpha=0.3)
    ax10.set_xlabel('Ego X (m)'); ax10.set_ylabel('Ego Y (m)')
    gz     = info.get('ground_z')
    gz_str = f'{gz:+.3f}m' if gz is not None else 'n/a'
    fsc    = info.get('face_support_counts', {})
    ax10.set_title(
        f'Ground removal (z={gz_str}) + DBSCAN  '
        f'before={fsc.get("n_nearby","-")} -> after_gnd={fsc.get("n_after_ground","-")}\n'
        f'cluster={info["cluster_points_count"]}  '
        f'rej_cls={info["rejected_cluster_count"]}  '
        f'spread L={fsc.get("spread_L","?"):.2f}m W={fsc.get("spread_W","?"):.2f}m  '
        f'shape={info["shape_classification"]}', fontsize=8)
    ax10.legend(loc='upper right', fontsize=6, facecolor='k', labelcolor='w', framealpha=0.7)

    # -- [1,1] BEV: alignment -- ENLARGED zoom --------------------------------
    a_fcs  = info.get('alignment_face_centers', [])
    a_fns  = info.get('alignment_face_normals', [])
    assign_viz = info.get('alignment_assign')
    resids = info.get('alignment_residuals')

    if cl_pts is not None and len(cl_pts):
        if cl_inside_mask is not None:
            _cl_in  = cl_pts[cl_inside_mask];   _cl_out = cl_pts[~cl_inside_mask]
            if len(_cl_in):
                ax11.scatter(_cl_in[:,0],  _cl_in[:,1],  c='yellow', s=12, alpha=0.9,
                             zorder=5, label=f'Cl inside ({len(_cl_in)})')
            if len(_cl_out):
                ax11.scatter(_cl_out[:,0], _cl_out[:,1], c='darkorange', s=22, alpha=0.9,
                             zorder=6, marker='x', linewidths=1.8,
                             label=f'Cl outside ({len(_cl_out)})')
        else:
            ax11.scatter(cl_pts[:,0], cl_pts[:,1], c='yellow', s=12, alpha=0.9,
                         zorder=5, label=f'Cluster ({len(cl_pts)})')

    face_colors = ['magenta', 'deepskyblue']
    if cl_pts is not None and resids is not None and a_fcs and len(a_fcs):
        for pi in range(len(cl_pts)):
            fi_a = int(assign_viz[pi]) if assign_viz is not None else 0
            fn_a = a_fns[fi_a]
            p    = cl_pts[pi, :2]
            foot = p - resids[pi] * fn_a
            ax11.plot([p[0], foot[0]], [p[1], foot[1]],
                      color=face_colors[fi_a], alpha=0.35, lw=0.8, zorder=3)

    best_mr    = info.get('mode_results', {}).get(info.get('var4_mode'), {})
    fi_indices = best_mr.get('face_indices', [])
    hl_f_ = float(obb_final['dims'][0]/2) if obb_final else 2.
    hw_f_ = float(obb_final['dims'][1]/2) if obb_final else 2.
    for fi_d, (fc_a, fn_a) in enumerate(zip(a_fcs or [], a_fns or [])):
        fi_idx  = fi_indices[fi_d] if fi_d < len(fi_indices) else 0
        hl_draw = hw_f_ if fi_idx in (0, 1) else hl_f_
        _draw_face(ax11, fc_a, fn_a, hl_draw, color=face_colors[fi_d], lw=3)
        lbl_f = (info['selected_faces'][fi_d]
                 if info.get('selected_faces') and fi_d < len(info['selected_faces']) else '')
        ax11.annotate(lbl_f, xy=fc_a, fontsize=9, color=face_colors[fi_d], zorder=8)

    _draw_obb(ax11, obb_bef,   'orange',    lw=1.5, ls='--', alpha=0.7, tag='body_before')
    if size_corr:
        _draw_obb(ax11, obb_aft,  'limegreen', lw=1.0, ls='--', alpha=0.5, tag='body_after')
    _draw_obb(ax11, obb_final, 'limegreen', lw=2.5, ls='-', tag='VAR4_final')

    # Color opt anchors by inside/outside final OBB; show excluded as hotpink
    if anchors_kept is not None and len(anchors_kept):
        _n_opt = info.get('anchor_opt_count', len(anchors_kept))
        _opt_pts = anchors_kept[:_n_opt]
        if anc_inside_mask is not None and len(anc_inside_mask) == len(_opt_pts):
            n_in_pts  = _opt_pts[anc_inside_mask]
            n_out_pts = _opt_pts[~anc_inside_mask]
            if len(n_in_pts):
                ax11.scatter(n_in_pts[:,0], n_in_pts[:,1], c='cyan', s=30, zorder=8,
                             edgecolors='teal', label=f'Anc inside ({len(n_in_pts)})')
            if len(n_out_pts):
                ax11.scatter(n_out_pts[:,0], n_out_pts[:,1], c='red', s=50, zorder=9,
                             marker='x', linewidths=2, label=f'Anc outside ({len(n_out_pts)})')
        else:
            ax11.scatter(_opt_pts[:,0], _opt_pts[:,1], c='cyan', s=22, zorder=7,
                         label=f'Opt anchors ({len(_opt_pts)})')
    if anchors_excl is not None and len(anchors_excl):
        ax11.scatter(anchors_excl[:,0], anchors_excl[:,1],
                     c='hotpink', s=35, alpha=0.8, marker='x', linewidths=1.5, zorder=6,
                     label=f'Excl from opt ({len(anchors_excl)})')

    # Overlay face-support points used in fitting (lime green, smaller)
    sup_pts_viz = info.get('face_support_pts')
    if sup_pts_viz is not None and len(sup_pts_viz):
        ax11.scatter(sup_pts_viz[:, 0], sup_pts_viz[:, 1],
                     c='lime', s=18, alpha=0.95, zorder=10, marker='o',
                     edgecolors='darkgreen', linewidths=0.5,
                     label=f'Face support ({len(sup_pts_viz)})')

    obb_ref    = obb_final or obb_bef
    obb_extent = max(obb_ref['dims'][0], obb_ref['dims'][1]) if obb_ref else 4.
    cl_ext_max = float(np.max(np.ptp(cl_pts[:, :2], axis=0))) if cl_pts is not None and len(cl_pts) else 2.
    r_z        = max(4., 1.5 * obb_extent, cl_ext_max + 2.)
    ref_xy     = cl_pts.mean(0)[:2] if cl_pts is not None and len(cl_pts) else ve_bef.mean(0)[:2]
    ax11.set_xlim(ref_xy[0]-r_z, ref_xy[0]+r_z)
    ax11.set_ylim(ref_xy[1]-r_z, ref_xy[1]+r_z)
    ax11.set_aspect('equal'); ax11.grid(alpha=0.3)
    ax11.set_xlabel('Ego X (m)'); ax11.set_ylabel('Ego Y (m)')
    ax11.set_title(
        f'Alignment  mode={info.get("var4_mode")}  shape={info.get("shape_classification")}\n'
        f'faces={info.get("selected_faces")}  ego_prior={info.get("ego_visible_face_prior")}\n'
        f'anc_inside={anc_in_str}  cl_inside={cl_in_str}  '
        f'tang={info.get("final_tangent_offset", 0.):+.2f}m  '
        f'{"(size_corr)" if size_corr else ""}', fontsize=8)
    ax11.legend(loc='upper right', fontsize=6, facecolor='k', labelcolor='w', framealpha=0.7)

    # -- [1,2] BEV overview ---------------------------------------------------
    ax12.scatter(ve_bef[:,0], ve_bef[:,1], c='orange',    s=1, alpha=0.2, zorder=1)
    ax12.scatter(ve_aft[:,0], ve_aft[:,1], c='limegreen', s=1, alpha=0.2, zorder=1)
    if nearby is not None and len(nearby):
        ax12.scatter(nearby[:,0], nearby[:,1], c='lightgray', s=1, alpha=0.2, zorder=1)
    for ci, rc in enumerate(rej_cls):
        if len(rc):
            ax12.scatter(rc[:,0], rc[:,1], c=rej_cols[ci % len(rej_cols)],
                         s=2, alpha=0.4, zorder=2)
    if cl_pts is not None and len(cl_pts):
        if cl_inside_mask is not None:
            _cl_in  = cl_pts[cl_inside_mask];   _cl_out = cl_pts[~cl_inside_mask]
            if len(_cl_in):
                ax12.scatter(_cl_in[:,0],  _cl_in[:,1],  c='yellow', s=5, alpha=0.9, zorder=5,
                             label=f'Cl inside ({len(_cl_in)})')
            if len(_cl_out):
                ax12.scatter(_cl_out[:,0], _cl_out[:,1], c='darkorange', s=12, alpha=0.85,
                             zorder=6, marker='x', linewidths=1.2,
                             label=f'Cl outside ({len(_cl_out)})')
        else:
            ax12.scatter(cl_pts[:,0], cl_pts[:,1], c='yellow', s=5, alpha=0.9, zorder=5,
                         label=f'Cluster ({len(cl_pts)})')
    if anchors_kept is not None:
        _n_opt = info.get('anchor_opt_count', len(anchors_kept))
        _opt_pts = anchors_kept[:_n_opt]
        if anc_inside_mask is not None and len(anc_inside_mask) == len(_opt_pts):
            n_in_pts  = _opt_pts[anc_inside_mask]
            n_out_pts = _opt_pts[~anc_inside_mask]
            if len(n_in_pts):
                ax12.scatter(n_in_pts[:,0],  n_in_pts[:,1],  c='cyan', s=20, zorder=7,
                             label=f'Anc inside ({len(n_in_pts)})')
            if len(n_out_pts):
                ax12.scatter(n_out_pts[:,0], n_out_pts[:,1], c='red',  s=35, marker='x', zorder=9,
                             linewidths=2, label=f'Anc outside ({len(n_out_pts)})')
        else:
            ax12.scatter(_opt_pts[:,0], _opt_pts[:,1], c='cyan', s=15, zorder=7,
                         label=f'Opt anchors ({len(_opt_pts)})')
    if anchors_excl is not None and len(anchors_excl):
        ax12.scatter(anchors_excl[:,0], anchors_excl[:,1],
                     c='hotpink', s=20, alpha=0.8, marker='x', linewidths=1.2, zorder=6,
                     label=f'Excl from opt ({len(anchors_excl)})')
    _draw_obb(ax12, obb_bef,   'orange',    lw=1.5, ls='--', alpha=0.7)
    _draw_obb(ax12, obb_final, 'limegreen', lw=2)
    ax12.set_aspect('equal'); ax12.grid(alpha=0.3)
    ax12.set_xlabel('Ego X (m)'); ax12.set_ylabel('Ego Y (m)')
    ax12.legend(loc='upper right', fontsize=6, facecolor='k', labelcolor='w', framealpha=0.7)
    ax12.set_title(
        f'BEV overview  yellow=cl_in  orange×=cl_out  cyan=anc_in  red×=anc_out\n'
        f'orange--=body_before  green=VAR4_final{"(corr)" if size_corr else ""}', fontsize=8)

    # -- [2,0] Ground removal detail ------------------------------------------
    gnd_in = info.get('ground_inliers_ego')
    if nearby is not None and len(nearby):
        ax20.scatter(nearby[:,0], nearby[:,1], c='lightgray', s=2, alpha=0.3, label='Nearby')
    if gnd_in is not None and len(gnd_in):
        ax20.scatter(gnd_in[:,0], gnd_in[:,1], c='limegreen', s=5, alpha=0.7,
                     label=f'Ground inliers ({len(gnd_in)})')
    if ng_pts is not None and len(ng_pts):
        ax20.scatter(ng_pts[:,0], ng_pts[:,1], c='steelblue', s=3, alpha=0.5,
                     label=f'Non-ground ({len(ng_pts)})')
    ax20.set_aspect('equal'); ax20.grid(alpha=0.3)
    ax20.set_xlabel('Ego X (m)'); ax20.set_ylabel('Ego Y (m)')
    ax20.set_title(f'Ground removal  z={gz_str}  success={info["ground_success"]}', fontsize=8)
    ax20.legend(loc='upper right', fontsize=6, facecolor='k', labelcolor='w', framealpha=0.7)

    # -- [2,1] Mode comparison bar chart -------------------------------------
    mode_res = info.get('mode_results', {})
    if mode_res:
        modes    = list(mode_res.keys())
        pen_ls   = [mode_res[m]['penalised_loss']            for m in modes]
        raw_ls   = [mode_res[m]['raw_loss']                  for m in modes]
        sel_ls   = [mode_res[m]['selection_loss']            for m in modes]
        cont_ls  = [mode_res[m].get('containment_loss', 0.) for m in modes]
        bar_cols = ['limegreen' if m == info.get('var4_mode') else 'steelblue' for m in modes]
        xpos     = np.arange(len(modes))
        bars     = ax21.bar(xpos,       sel_ls,  color=bar_cols,        alpha=0.8, width=0.22, label='Selection loss')
        ax21.bar(xpos+0.22, pen_ls,  color='cornflowerblue', alpha=0.7, width=0.22, label='Pen. loss (in opt)')
        ax21.bar(xpos+0.44, raw_ls,  color='orange',         alpha=0.6, width=0.22, label='Raw perp loss')
        ax21.bar(xpos+0.66, cont_ls, color='salmon',         alpha=0.6, width=0.22, label='Cl. cont (post)')
        ax21.axhline(info['face_loss_before'], color='red', lw=1.5, ls='--',
                     label=f'Baseline {info["face_loss_before"]:.4f}m')
        ax21.set_xticks(xpos+0.33); ax21.set_xticklabels(modes, fontsize=9)
        ax21.set_ylabel('Loss (m)'); ax21.grid(axis='y', alpha=0.3)
        ax21.set_title(
            f'Mode comparison  selected={info.get("var4_mode")}  '
            f'shape={info.get("shape_classification")}\n'
            f'baseline={info["face_loss_before"]:.4f}  '
            f'after={info["face_loss_after"]:.4f}  '
            f'l_shape_allowed={info["l_shape_allowed"]}', fontsize=8)
        ax21.legend(fontsize=7)
        for bar, l in zip(bars, sel_ls):
            ax21.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                      f'{l:.4f}', ha='center', va='bottom', fontsize=7)

    # -- [2,2] Summary text ---------------------------------------------------
    ax22.axis('off')
    fsc  = info.get('face_support_counts', {})
    lfc  = info.get('l_shape_face_counts', {})
    db   = info.get('dims_before')
    da   = info.get('dims_after')
    db_str = f'L={db[0]:.2f}m W={db[1]:.2f}m' if db is not None else 'n/a'
    da_str = f'L={da[0]:.2f}m W={da[1]:.2f}m' if da is not None else 'n/a'
    rls    = info.get('req_l_side', float('nan'))
    rls_str = f'{rls:.2f}m' if np.isfinite(rls) else 'n/a'
    lines_txt = [
        f'[EXPERIMENTAL] {label}',
        f'Frame {frame_idx}  obj [{obj_idx}]',
        '',
        f'Status                  : {info["status"]}',
        '',
        f'-- Anchors ----------------------------------',
        f'anchor_points_count     : {info["anchor_points_count"]}',
        f'anchor_groups_count     : {info["anchor_groups_count"]}',
        f'kept_anchor_count       : {info["kept_anchor_count"]}',
        f'rejected_anchor_count   : {info["rejected_anchor_count"]}',
        f'anchor_opt_count        : {info.get("anchor_opt_count","-")}  (used in opt)',
        f'anchor_excluded_count   : {info.get("anchor_excluded_count",0)}  (other clusters)',
        f'nearby_candidates       : {info["nearby_candidates_count"]}',
        f'cluster_points_count    : {info["cluster_points_count"]}',
        f'rejected_cluster_cnt    : {info["rejected_cluster_count"]}',
        '',
        f'-- Body OBB trimming ------------------------',
        f'len_pct / wid_pct       : {_blp} / {_bwp}',
        '',
        f'-- Shape classification ---------------------',
        f'shape_classification    : {info["shape_classification"]}',
        f'l_shape_allowed         : {info["l_shape_allowed"]}',
        f'spread L / W            : {fsc.get("spread_L","?")} / {fsc.get("spread_W","?")} m',
        f'l_shape_face_counts     : {lfc}',
        f'l_shape_rejected_reason : {info["l_shape_rejected_reason"]}',
        '',
        f'-- Alignment --------------------------------',
        f'var4_mode               : {info["var4_mode"]}',
        f'selected_faces          : {info["selected_faces"]}',
        f'ego_visible_face_prior  : {info["ego_visible_face_prior"]}',
        f'final_tangent_offset    : {info["final_tangent_offset"]:+.3f} m',
        '',
        f'-- Containment (on var4_obb_final) ----------',
        f'anchor_inside_ratio     : {anc_in_str}',
        f'cluster_inside_ratio    : {cl_in_str}',
        f'anchor_outside_count    : {info["anchor_outside_count"]}',
        f'cluster_outside_count   : {info["cluster_outside_count"]}',
        f'anchor_containment_loss : {info["anchor_containment_loss"]:.4f}',
        f'cluster_containment_loss: {info["cluster_containment_loss"]:.4f}',
        '',
        f'-- VAR2 prior / fallback --------------------',
        f'yaw_prior_source        : {info.get("yaw_prior_source", "raw")}',
        f'var2_chamfer            : {info.get("var2_chamfer", float("nan")):.4f} m' if np.isfinite(info.get("var2_chamfer", float("nan"))) else f'var2_chamfer            : n/a',
        f'var4_chamfer            : {info.get("var4_chamfer", float("nan")):.4f} m' if np.isfinite(info.get("var4_chamfer", float("nan"))) else f'var4_chamfer            : n/a',
        f'var4_improves_var2      : {info.get("var4_improves_var2")}',
        f'fallback_used           : {info.get("fallback_used", False)}',
        f'yaw_delta_from_var2     : {info.get("yaw_delta_from_var2", float("nan")):+.2f} deg' if np.isfinite(info.get("yaw_delta_from_var2", float("nan"))) else f'yaw_delta_from_var2     : n/a',
        '',
        f'-- L-shape strength -------------------------',
        f'l_shape_type            : {info.get("l_shape_type", "")}',
        f'l_shape_strength        : {info.get("l_shape_strength", float("nan")):.2f}' if np.isfinite(info.get("l_shape_strength", float("nan"))) else f'l_shape_strength        : n/a',
        f'primary_face_support_ct : {info.get("primary_face_support_count", 0)}',
        f'secondary_face_support  : {info.get("secondary_face_support_count", 0)}',
        f'primary_face_loss       : {info.get("primary_face_loss", float("nan")):.4f} m' if np.isfinite(info.get("primary_face_loss", float("nan"))) else f'primary_face_loss       : n/a',
        f'secondary_face_loss     : {info.get("secondary_face_loss", float("nan")):.4f} m' if np.isfinite(info.get("secondary_face_loss", float("nan"))) else f'secondary_face_loss     : n/a',
        f'secondary_face_weight   : {info.get("secondary_face_weight", 1.0):.2f}',
        '',
        f'-- Corrections ------------------------------',
        f'dZ                      : {info["delta_cam_z"]:+.4f} m',
        f'dXY                     : {mag_xy:.4f} m  ({dxy[0]:+.3f}, {dxy[1]:+.3f})',
        f'dyaw                    : {info["delta_yaw_deg"]:+.2f} deg',
        '',
        f'-- Size correction --------------------------',
        f'size_correction_enabled : {info["size_correction_enabled"]}',
        f'length_scale            : {info["length_scale"]:.3f}',
        f'width_scale             : {info["width_scale"]:.3f}',
        f'req_l_side              : {rls_str}',
        f'side_face_pts_count     : {info["side_face_pts_count"]}',
        f'dims before             : {db_str}',
        f'dims after              : {da_str}',
        '',
        f'-- Face support (1-face modes) --------------',
        f'face_support_count      : {info.get("face_support_count", 0)}',
        f'face_support_fraction   : {info.get("face_support_fraction", float("nan")):.1%}' if np.isfinite(info.get("face_support_fraction", float("nan"))) else f'face_support_fraction   : n/a',
        f'face_support_band_m     : {info.get("face_support_band_m", float("nan")):.3f} m' if np.isfinite(info.get("face_support_band_m", float("nan"))) else f'face_support_band_m     : n/a',
        f'face_support_mode       : {info.get("face_support_mode", "")}',
        '',
        f'-- Losses -----------------------------------',
        f'face_loss_before        : {info["face_loss_before"]:.4f} m  (full cluster)',
        f'face_loss_before_sup    : {info.get("face_loss_before_support", float("nan")):.4f} m' if np.isfinite(info.get("face_loss_before_support", float("nan"))) else f'face_loss_before_sup    : n/a',
        f'face_loss_after_sup     : {info.get("face_loss_after_support", float("nan")):.4f} m' if np.isfinite(info.get("face_loss_after_support", float("nan"))) else f'face_loss_after_sup     : n/a',
        f'face_loss_after         : {info["face_loss_after"]:.4f} m',
        f'Chamfer before          : {info["lidar_mesh_dist_before"]:.4f} m',
        f'Chamfer after           : {info["lidar_mesh_dist"]:.4f} m',
        '',
        f'-- Ground -----------------------------------',
        f'ground_z                : {gz_str}',
        f'ground_success          : {info["ground_success"]}',
        '',
        f'-- Rejection --------------------------------',
        f'var4_rejected_reason    : {info["var4_rejected_reason"]}',
    ]
    ax22.text(0.04, 0.97, '\n'.join(lines_txt), transform=ax22.transAxes,
              fontsize=7.5, va='top', family='monospace', color='white',
              bbox=dict(boxstyle='round', facecolor='#1a1a2e',
                        edgecolor='#ff8800', alpha=0.93))

    plt.suptitle(f'[EXPERIMENTAL] VAR4 debug | frame {frame_idx} | obj [{obj_idx}]',
                 fontsize=12, fontweight='bold', color='darkorange')
    plt.tight_layout(); plt.show()


print('VAR4 functions defined.')

In [ ]:
# ── Debug visualization for one VAR result ────────────────────────────────────
def visualize_var_debug(verts_before, verts_after, info, binary_mask, img_rgb,
                         K, u_lf, v_lf, Z_lf, R_c2e, t_c2e,
                         xl=None, xr=None, yt=None, yb=None,
                         dilation_px=50, zbuf_tol=0.30, label='VAR',
                         frame_idx=None, obj_idx=None):
    """
    6-panel debug plot for one VAR strategy.

    Panel layout (2 rows x 3 cols):
      [0] Image: in-dil-mask verts (gray), z-buf visible (green), LiDAR (cyan)
      [1] Z-buffer depth map + visible vert overlay
      [2] Ego XY: visible verts + LiDAR + trimmed matched pairs
      [3] Image before: mesh + LiDAR overlay
      [4] Image after:  mesh + LiDAR overlay
      [5] BEV before (orange) / after (green) in ego frame
    """
    H, W = binary_mask.shape
    fx_ = float(K[0,0]); fy_ = float(K[1,1])
    cx_ = float(K[0,2]); cy_ = float(K[1,2])
    R = R_c2e.astype(np.float64); t_ = t_c2e.astype(np.float64)

    # Crop bounds
    if xl is None:
        ys_m_, xs_m_ = np.where(binary_mask)
        pad = 80
        xl  = max(0, int(xs_m_.min()) - pad); xr  = min(W, int(xs_m_.max()) + pad)
        yt  = max(0, int(ys_m_.min()) - pad); yb  = min(H, int(ys_m_.max()) + pad)
    crop = np.s_[yt:yb, xl:xr]

    # Re-extract visibility for before/after
    vis_before, dbg = _extract_visible_verts(verts_before.astype(np.float64), binary_mask,
                                              K, dilation_px, zbuf_tol)
    vis_after,  _   = _extract_visible_verts(verts_after.astype(np.float64),  binary_mask,
                                              K, dilation_px, zbuf_tol)

    # LiDAR in ego
    Ulf = u_lf.astype(np.float64); Vlf = v_lf.astype(np.float64); Zlf = Z_lf.astype(np.float64)
    li_cam = np.stack([(Ulf-cx_)*Zlf/fx_, (Vlf-cy_)*Zlf/fy_, Zlf], axis=1)
    li_ego = (R @ li_cam.T).T + t_

    vb64 = verts_before.astype(np.float64)
    vis_ego_before = (R @ vb64[vis_before].T).T + t_

    # Trimmed NN pairs (from info if available)
    dbg_i = info.get('_dbg', None)

    fig, axes = plt.subplots(2, 3, figsize=(24, 14))

    # ── Panel 0: vertex categories on image ───────────────────────────────────
    ax = axes[0, 0]
    ax.imshow(img_rgb[crop])
    in_img  = dbg['in_img']
    in_dil  = dbg['in_dil']
    zb_vis  = dbg['zbuf_vis']
    uf = dbg['u_f']; vf_ = dbg['v_f']

    # All in-image verts (gray)
    ax.scatter(uf[in_img]-xl, vf_[in_img]-yt,
               c='gray', s=1, alpha=0.3, zorder=2, label=f'in-image ({in_img.sum()})')
    # In-dilated-mask but NOT z-buf vis (blue)
    in_dil_only = in_dil & ~zb_vis
    if in_dil_only.any():
        ax.scatter(uf[in_dil_only]-xl, vf_[in_dil_only]-yt,
                   c='royalblue', s=6, alpha=0.6, zorder=3,
                   label=f'in-mask !z-buf ({in_dil_only.sum()})')
    # Z-buffer visible (green)
    if vis_before.any():
        ax.scatter(uf[vis_before]-xl, vf_[vis_before]-yt,
                   c='lime', s=8, alpha=0.8, zorder=4,
                   label=f'z-buf visible ({vis_before.sum()})')
    # LiDAR
    ax.scatter(Ulf-xl, Vlf-yt, c=Zlf, cmap='cool',
               s=40, alpha=0.9, zorder=5, edgecolors='teal', linewidths=0.4,
               label=f'LiDAR ({len(Zlf)})')
    # Dilated mask contour
    kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilation_px*2+1,)*2)
    mdil = cv2.dilate(binary_mask.astype(np.uint8), kern)
    cnts, _ = cv2.findContours(mdil, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in cnts:
        cp = cnt[:, 0, :]
        cp = np.vstack([cp, cp[0]])  # close the contour visually
        ax.plot(cp[:, 0] - xl, cp[:, 1] - yt, 'yellow', lw=1, zorder=6)
    cnts2, _ = cv2.findContours(binary_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in cnts2:
        cp = cnt[:, 0, :]
        cp = np.vstack([cp, cp[0]])
        ax.plot(cp[:, 0] - xl, cp[:, 1] - yt, 'lime', lw=1.5, zorder=7)
    ax.set_title(f'Vertex categories\ngray=in-img  blue=in-mask!zbuf  green=zbuf-vis  cyan=LiDAR')
    ax.legend(loc='upper right', fontsize=6, facecolor='black', labelcolor='white')
    ax.axis('off')

    # ── Panel 1: Z-buffer depth map ───────────────────────────────────────────
    ax = axes[0, 1]
    dbuf = dbg['depth_buf']
    dbuf_crop = dbuf[yt:yb, xl:xr].copy()
    finite_mask = np.isfinite(dbuf_crop)
    dbuf_vis = np.zeros_like(dbuf_crop)
    if finite_mask.any():
        dbuf_vis[finite_mask] = dbuf_crop[finite_mask]
    im = ax.imshow(dbuf_vis, cmap='plasma_r', interpolation='nearest')
    plt.colorbar(im, ax=ax, fraction=0.03, label='depth (m)')
    if vis_before.any():
        ax.scatter(dbg['u_f'][vis_before]-xl, dbg['v_f'][vis_before]-yt,
                   c='lime', s=6, alpha=0.7, zorder=4, label='z-buf visible')
    ax.scatter(Ulf-xl, Vlf-yt, c='cyan', s=30, alpha=0.9, zorder=5, label='LiDAR')
    ax.set_title(f'Z-buffer depth map\n(brighter = closer to camera)\nzbuf_tol={zbuf_tol}m')
    ax.legend(fontsize=7); ax.axis('off')

    # ── Panel 2: ego XY — visible mesh + LiDAR + matched pairs ───────────────
    ax = axes[0, 2]
    if vis_ego_before.shape[0] > 0:
        ax.scatter(vis_ego_before[:,0], vis_ego_before[:,1],
                   c='lime', s=8, alpha=0.7, zorder=3, label=f'vis mesh ({len(vis_ego_before)})')
    ax.scatter(li_ego[:,0], li_ego[:,1],
               c='cyan', s=30, alpha=0.9, zorder=4,
               edgecolors='teal', linewidths=0.4, label=f'LiDAR ({len(li_ego)})')

    # Draw trimmed matched pairs as lines
    if dbg_i is not None and vis_ego_before.shape[0] > 0:
        keep = dbg_i['keep_idx']; nn = dbg_i['nn_idx_l']
        vis_e = dbg_i['vis_ego_before']
        n_lines = min(len(keep), 60)
        for ki in keep[:n_lines]:
            lp = li_ego[ki]; mp = vis_e[nn[ki]]
            ax.plot([lp[0], mp[0]], [lp[1], mp[1]],
                    color='yellow', alpha=0.5, lw=0.8, zorder=2)
        ax.scatter(li_ego[keep[:n_lines], 0], li_ego[keep[:n_lines], 1],
                   c='yellow', s=20, zorder=5, label=f'trimmed pairs ({len(keep)})')

    ax.set_aspect('equal'); ax.grid(alpha=0.3)
    ax.set_xlabel('ego X (m)'); ax.set_ylabel('ego Y (m)')
    ax.set_title('Ego XY: vis-mesh + LiDAR + trimmed pairs\n(yellow lines = matched)')
    ax.legend(fontsize=7)

    # ── Panel 3: before image overlay ─────────────────────────────────────────
    ax = axes[1, 0]
    img_b = draw_mesh_on_image(img_rgb, verts_before, K, color=(255, 140, 0))
    ax.imshow(img_b[crop])
    ax.scatter(Ulf-xl, Vlf-yt, c=Zlf, cmap='cool', s=25, alpha=0.8, zorder=5)
    if vis_before.any():
        ax.scatter(dbg['u_f'][vis_before]-xl, dbg['v_f'][vis_before]-yt,
                   c='lime', s=4, alpha=0.6, zorder=6)
    ax.set_title(f'Before ({label})\nli_dist={info["lidar_mesh_dist_before"]:.3f}m')
    ax.axis('off')

    # ── Panel 4: after image overlay ──────────────────────────────────────────
    ax = axes[1, 1]
    img_a = draw_mesh_on_image(img_rgb, verts_after, K, color=(0, 200, 80))
    ax.imshow(img_a[crop])
    ax.scatter(Ulf-xl, Vlf-yt, c=Zlf, cmap='cool', s=25, alpha=0.8, zorder=5)
    if vis_after.any():
        uf_a = np.where(verts_after[:,2]>1e-6,
                        fx_*verts_after[:,0]/np.maximum(verts_after[:,2],1e-6)+cx_, -9999.)
        vf_a = np.where(verts_after[:,2]>1e-6,
                        fy_*verts_after[:,1]/np.maximum(verts_after[:,2],1e-6)+cy_, -9999.)
        ax.scatter(uf_a[vis_after]-xl, vf_a[vis_after]-yt,
                   c='yellow', s=4, alpha=0.6, zorder=6)
    li_s = info['lidar_mesh_dist']
    li_str = f'{li_s:.3f}m' if np.isfinite(li_s) else '-'
    ax.set_title(f'After ({label})\nli_dist={li_str}  '
                 f'dZ={info["delta_cam_z"]:+.2f}m  '
                 f'dXY={np.linalg.norm(info["delta_ego_xy"]):.2f}m\n'
                 f'dyaw={info["delta_yaw_deg"]:+.1f}deg  s={info["scale"]:.3f}')
    ax.axis('off')

    # ── Panel 5: ego XY BEV before / after ────────────────────────────────────
    ax = axes[1, 2]
    va64 = verts_after.astype(np.float64)
    vb_ego = (R @ vb64.T).T + t_
    va_ego = (R @ va64.T).T + t_
    ctr = vb_ego.mean(0)
    r5  = 6.0
    # All mesh verts (before: orange, after: green)
    ax.scatter(vb_ego[:,0], vb_ego[:,1], c='orange', s=1, alpha=0.2, zorder=2, label='mesh before')
    ax.scatter(va_ego[:,0], va_ego[:,1], c='limegreen', s=1, alpha=0.2, zorder=2, label='mesh after')
    # Z-buf vis verts
    if vis_before.any():
        ax.scatter(vis_ego_before[:,0], vis_ego_before[:,1],
                   c='orange', s=12, alpha=0.8, zorder=4, edgecolors='darkorange', lw=0.3)
    if vis_after.any():
        va_vis_ego = (R @ va64[vis_after].T).T + t_
        ax.scatter(va_vis_ego[:,0], va_vis_ego[:,1],
                   c='lime', s=12, alpha=0.8, zorder=4, edgecolors='darkgreen', lw=0.3)
    # LiDAR
    ax.scatter(li_ego[:,0], li_ego[:,1], c='cyan', s=40, alpha=0.9, zorder=5,
               edgecolors='teal', lw=0.4, label='LiDAR')
    ax.set_xlim(ctr[0]-r5, ctr[0]+r5); ax.set_ylim(ctr[1]-r5, ctr[1]+r5)
    ax.set_aspect('equal'); ax.grid(alpha=0.3)
    ax.set_xlabel('ego X (m)'); ax.set_ylabel('ego Y (m)')
    ax.set_title('Ego XY BEV: orange=before  green=after  cyan=LiDAR\n'
                 '(filled=z-buf visible, faint=all mesh)')
    ax.legend(fontsize=7)

    _fi = frame_idx if frame_idx is not None else FRAME_IDX
    _oi = obj_idx   if obj_idx   is not None else OBJ_IDX
    plt.suptitle(f'VAR debug — {label} — frame {_fi} | obj [{_oi}]\n'
                 f'n_vis={info["n_vis_verts"]}  n_lidar={info["n_lidar"]}  '
                 f'status={info["status"]}',
                 fontsize=11)
    plt.tight_layout()
    plt.show()


print('visualize_var_debug defined.')

In [ ]:
# ── OBB helper functions: RANSAC ground estimation + gravity-aligned OBB ─────

def estimate_local_ground(
        xyz_lid, R_l2e, t_l2e,
        obj_center_ego,
        crop_r=8.0,
        excl_r=None,
        bot_frac=0.35,
        ransac_n=200,
        ransac_thresh=0.1,
        min_inliers=15,
        max_tilt_deg=12.0,
):
    """
    Estimate the local ground plane under an object using RANSAC on nearby LiDAR.

    Algorithm
    ---------
    1. Transform the full LiDAR sweep to ego frame (Z = world-up).
    2. BEV-crop to a circle of radius `crop_r` around the object centre.
    3. Exclude a circle of radius `excl_r` (approximate object footprint).
    4. Keep the bottom `bot_frac` of the remaining points by Z.
    5. RANSAC: fit z = a*x + b*y + c on random 3-point samples.
    6. Horizontality guard: plane normal must be within `max_tilt_deg` of vertical.
    7. Returns ground_z = a*cx + b*cy + c at the object centre.

    Parameters
    ----------
    xyz_lid        : (N,3) full LiDAR sweep in *LiDAR* frame
    R_l2e, t_l2e   : lidar->ego rigid transform  (ego Z = world-up)
    obj_center_ego : (3,) object centre in *ego* frame
    crop_r         : BEV crop radius around object [m]
    excl_r         : BEV exclusion radius (object footprint) [m]; None = no excl.
    bot_frac       : keep bottom this fraction of cropped points by Z
    ransac_n       : number of RANSAC iterations
    ransac_thresh  : per-point inlier Z residual threshold [m]
    min_inliers    : minimum inliers required for a valid plane
    max_tilt_deg   : max allowed angle between plane normal and vertical [deg]

    Returns
    -------
    dict with keys:
        success        : bool
        ground_z       : float | None  -- estimated ground height at obj centre (ego Z)
        plane_coeff    : (3,) | None   -- [a, b, c] for  z = a*x + b*y + c
        n_ground_pts   : int           -- points in local crop after BEV/excl filter
        n_low_pts      : int           -- after bot_frac filter
        n_inliers      : int
        inlier_mask    : bool array    -- over the *low* pts array
        ground_pts_ego : (n_low_pts,3) -- low pts used for RANSAC (ego frame)
        rmse           : float | None
        tilt_deg       : float | None
        msg            : str
    """
    # Transform to ego frame
    xyz_ego = (R_l2e @ xyz_lid.T).T + t_l2e   # (N, 3)

    cx, cy = float(obj_center_ego[0]), float(obj_center_ego[1])
    dx  = xyz_ego[:, 0] - cx
    dy  = xyz_ego[:, 1] - cy
    bev = np.sqrt(dx**2 + dy**2)

    # BEV crop + object exclusion
    mask = bev < crop_r
    if excl_r is not None and excl_r > 0:
        mask &= (bev > excl_r)

    pts = xyz_ego[mask]
    n_ground_pts = len(pts)

    if n_ground_pts < min_inliers:
        return dict(success=False, ground_z=None, plane_coeff=None,
                    n_ground_pts=n_ground_pts, n_low_pts=0, n_inliers=0,
                    inlier_mask=None, ground_pts_ego=pts, rmse=None,
                    tilt_deg=None,
                    msg=f'FAIL: only {n_ground_pts} pts in local crop (need {min_inliers})')

    # Keep bottom fraction by Z
    z_cut = float(np.percentile(pts[:, 2], bot_frac * 100))
    low   = pts[:, 2] <= z_cut
    pts_low = pts[low]
    n_low_pts = len(pts_low)

    if n_low_pts < min_inliers:
        return dict(success=False, ground_z=None, plane_coeff=None,
                    n_ground_pts=n_ground_pts, n_low_pts=n_low_pts, n_inliers=0,
                    inlier_mask=None, ground_pts_ego=pts_low, rmse=None,
                    tilt_deg=None,
                    msg=f'FAIL: only {n_low_pts} low pts after bot_frac={bot_frac:.0%}')

    # ── RANSAC ────────────────────────────────────────────────────────────────
    n = len(pts_low)
    rng = np.random.default_rng(42)
    best_mask  = np.zeros(n, dtype=bool)
    best_count = 0

    for _ in range(ransac_n):
        idx = rng.choice(n, 3, replace=False)
        samp = pts_low[idx]
        A_s  = np.c_[samp[:, 0], samp[:, 1], np.ones(3)]
        try:
            coeff_s, _, rank, _ = np.linalg.lstsq(A_s, samp[:, 2], rcond=None)
            if rank < 3:
                continue
        except Exception:
            continue
        z_pred = pts_low[:, 0]*coeff_s[0] + pts_low[:, 1]*coeff_s[1] + coeff_s[2]
        m = np.abs(pts_low[:, 2] - z_pred) < ransac_thresh
        if m.sum() > best_count:
            best_count = int(m.sum())
            best_mask  = m

    if best_count < min_inliers:
        return dict(success=False, ground_z=None, plane_coeff=None,
                    n_ground_pts=n_ground_pts, n_low_pts=n_low_pts,
                    n_inliers=best_count, inlier_mask=best_mask,
                    ground_pts_ego=pts_low, rmse=None, tilt_deg=None,
                    msg=f'FAIL: RANSAC found {best_count} inliers (need {min_inliers})')

    # Refit on all inliers
    pts_in = pts_low[best_mask]
    A_in   = np.c_[pts_in[:, 0], pts_in[:, 1], np.ones(len(pts_in))]
    coeff, _, _, _ = np.linalg.lstsq(A_in, pts_in[:, 2], rcond=None)

    # Recompute inlier mask with final coefficients
    z_pred_all  = pts_low[:, 0]*coeff[0] + pts_low[:, 1]*coeff[1] + coeff[2]
    inlier_mask = np.abs(pts_low[:, 2] - z_pred_all) < ransac_thresh
    n_inliers   = int(inlier_mask.sum())
    rmse = float(np.sqrt(np.mean(
        (pts_low[inlier_mask, 2] - z_pred_all[inlier_mask])**2)))

    # ── Horizontality guard ───────────────────────────────────────────────────
    a, b, c = coeff
    normal   = np.array([-a, -b, 1.0])
    normal  /= np.linalg.norm(normal)
    tilt_deg = float(np.degrees(np.arccos(np.clip(abs(normal[2]), 0, 1))))

    if tilt_deg > max_tilt_deg:
        return dict(success=False, ground_z=None, plane_coeff=coeff,
                    n_ground_pts=n_ground_pts, n_low_pts=n_low_pts,
                    n_inliers=n_inliers, inlier_mask=inlier_mask,
                    ground_pts_ego=pts_low, rmse=rmse, tilt_deg=tilt_deg,
                    msg=f'FAIL: plane tilt {tilt_deg:.1f} deg > {max_tilt_deg} deg limit')

    ground_z = float(a * cx + b * cy + c)

    return dict(success=True, ground_z=ground_z, plane_coeff=coeff,
                n_ground_pts=n_ground_pts, n_low_pts=n_low_pts,
                n_inliers=n_inliers, inlier_mask=inlier_mask,
                ground_pts_ego=pts_low, rmse=rmse, tilt_deg=tilt_deg,
                msg=(f'OK  n_crop={n_ground_pts}  n_low={n_low_pts}  '
                     f'inliers={n_inliers}  RMSE={rmse:.4f}m  '
                     f'tilt={tilt_deg:.1f}deg  ground_z={ground_z:+.3f}m'))


def compute_obb_gravity_aligned(verts_r3, R_c2e, t_c2e,
                                 ground_z=None,
                                 height_pct=(1, 99),
                                 height_clamp=(0.9, 2.8)):
    """
    Fit a gravity-aligned OBB to mesh vertices in R3 camera space.

    Footprint (yaw / length / width)
        PCA of mesh XY footprint in ego frame — unchanged from original.

    Vertical extent
        If `ground_z` is provided (from RANSAC ground estimation):
          robust_height = percentile(ego_Z, height_pct[1])
                        - percentile(ego_Z, height_pct[0])
          clamped to `height_clamp` [m]
          z_bottom = ground_z
          z_top    = ground_z + robust_height
        Otherwise: raw mesh min/max Z (original behaviour).

    Parameters
    ----------
    ground_z     : ego-frame ground height at OBB centre, or None
    height_pct   : (low_pct, high_pct) percentiles for robust mesh height
    height_clamp : (min_h, max_h) plausible vehicle height range [m]
    """
    verts_ego = (R_c2e @ verts_r3.T).T + t_c2e

    xy   = verts_ego[:, :2]
    xy_c = xy - xy.mean(axis=0)
    _, evecs = np.linalg.eigh(np.cov(xy_c.T))
    evecs = evecs[:, ::-1]

    ax_l = np.array([evecs[0, 0], evecs[1, 0], 0.], dtype=np.float64)
    ax_w = np.array([evecs[0, 1], evecs[1, 1], 0.], dtype=np.float64)
    ax_h = np.array([0., 0., 1.], dtype=np.float64)
    if ax_l[0] * ax_w[1] - ax_l[1] * ax_w[0] < 0:
        ax_w = -ax_w
    axes = np.stack([ax_l, ax_w, ax_h], axis=1)

    proj  = verts_ego @ axes
    min_p = proj.min(axis=0).copy()
    max_p = proj.max(axis=0).copy()

    # ── Vertical extent ───────────────────────────────────────────────────────
    if ground_z is not None:
        vz = verts_ego[:, 2]
        raw_h = float(np.percentile(vz, height_pct[1]) - np.percentile(vz, height_pct[0]))
        robust_height = float(np.clip(raw_h, height_clamp[0], height_clamp[1]))
        min_p[2] = ground_z
        max_p[2] = ground_z + robust_height
    # else: keep raw min/max

    mid_p      = (min_p + max_p) / 2
    center_ego = axes @ mid_p

    off = np.array([
        [min_p[0]-mid_p[0], min_p[1]-mid_p[1], min_p[2]-mid_p[2]],
        [max_p[0]-mid_p[0], min_p[1]-mid_p[1], min_p[2]-mid_p[2]],
        [max_p[0]-mid_p[0], max_p[1]-mid_p[1], min_p[2]-mid_p[2]],
        [min_p[0]-mid_p[0], max_p[1]-mid_p[1], min_p[2]-mid_p[2]],
        [min_p[0]-mid_p[0], min_p[1]-mid_p[1], max_p[2]-mid_p[2]],
        [max_p[0]-mid_p[0], min_p[1]-mid_p[1], max_p[2]-mid_p[2]],
        [max_p[0]-mid_p[0], max_p[1]-mid_p[1], max_p[2]-mid_p[2]],
        [min_p[0]-mid_p[0], max_p[1]-mid_p[1], max_p[2]-mid_p[2]],
    ])
    corners_ego = (axes @ off.T).T + center_ego
    dims    = max_p - min_p
    yaw_rad = float(np.arctan2(evecs[1, 0], evecs[0, 0]))
    return corners_ego, center_ego, dims, yaw_rad, axes


print('estimate_local_ground + compute_obb_gravity_aligned defined.')


In [ ]:
# ── §3 helper: instance selection ────────────────────────────────────────────
def _select_instance(obj_idx, frame_cache, make_plots=True):
    """Select one detection, compute crop bounds. Returns state dict."""
    img_rgb    = frame_cache['img_rgb']
    H, W       = frame_cache['H'], frame_cache['W']
    detections = frame_cache['detections']

    assert obj_idx < len(detections), \
        f'obj_idx={obj_idx} out of range — {len(detections)} detections'
    det         = detections[obj_idx]
    binary_mask = det['binary_mask']
    x1, y1, x2, y2 = det['box_xyxy']
    _COLORS     = [(255,0,0),(0,255,0),(0,0,255),(255,165,0),(0,255,255),
                   (255,0,255),(255,255,0),(128,0,128),(0,128,128)]
    sel_color   = _COLORS[obj_idx % len(_COLORS)]
    sel_color_f = tuple(c/255 for c in sel_color)

    ys_m, xs_m = np.where(binary_mask)
    pad = 80
    xl = max(0,int(xs_m.min())-pad); xr = min(W,int(xs_m.max())+pad)
    yt = max(0,int(ys_m.min())-pad); yb = min(H,int(ys_m.max())+pad)

    if make_plots:
        overlay = img_rgb.copy().astype(np.float32)
        fill    = np.zeros_like(overlay); fill[binary_mask] = sel_color
        overlay = (overlay*.6 + fill*.4).clip(0,255).astype(np.uint8)
        fig, axes = plt.subplots(1,2,figsize=(16,6))
        axes[0].imshow(overlay)
        axes[0].add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,lw=3,
                          edgecolor=sel_color_f,facecolor='none'))
        axes[0].text(x1+4,y1+22,f'[{obj_idx}] selected',color=sel_color_f,fontsize=13,
                     fontweight='bold',bbox=dict(boxstyle='round,pad=0.2',fc='black',alpha=.65))
        axes[0].set_title(f'obj {obj_idx}  prob={det["prob"]:.3f}  mask_px={binary_mask.sum()}')
        axes[0].axis('off')
        axes[1].imshow(overlay[max(0,int(y1)-pad):min(H,int(y2)+pad),
                                max(0,int(x1)-pad):min(W,int(x2)+pad)])
        axes[1].set_title('Zoom'); axes[1].axis('off')
        plt.tight_layout(); plt.show()

    return dict(binary_mask=binary_mask, det=det, sel_color_f=sel_color_f,
                x1=x1, y1=y1, x2=x2, y2=y2, xs_m=xs_m, ys_m=ys_m,
                xl=xl, xr=xr, yt=yt, yb=yb)

print('_select_instance defined.')


In [ ]:
# ── §4 helper: in-mask LiDAR depth-window filter ────────────────────────────
def _filter_lidar_obj(obj_idx, inst, frame_cache, config, make_plots=True):
    """Filter in-mask LiDAR with depth-window. Returns state dict."""
    binary_mask = inst['binary_mask']
    sel_color_f = inst['sel_color_f']
    xl,xr,yt,yb = inst['xl'],inst['xr'],inst['yt'],inst['yb']
    img_rgb = frame_cache['img_rgb']
    H, W    = frame_cache['H'], frame_cache['W']
    u_int   = frame_cache['u_int']; v_int = frame_cache['v_int']
    Z_v     = frame_cache['Z_v']
    u_f_v   = frame_cache['u_f_v']; v_f_v = frame_cache['v_f_v']
    MIN_LIDAR_HITS   = config['MIN_LIDAR_HITS']
    MAX_DEPTH_SPAN = config['MAX_DEPTH_SPAN']
    FRAME_IDX        = config['FRAME_IDX']

    in_mask = binary_mask[v_int, u_int]
    n_hits  = int(in_mask.sum())
    print(f'  {n_hits} LiDAR points in mask')

    Z_lidar_anchor = None
    if n_hits >= MIN_LIDAR_HITS:
        Z_in_mask      = Z_v[in_mask]
        Z_lidar_anchor = float(Z_in_mask.min())
        print(f'  Z_front={Z_lidar_anchor:.3f}m  Z_median={float(np.median(Z_in_mask)):.3f}m')
    elif n_hits > 0:
        print(f'  WARNING: fewer than {MIN_LIDAR_HITS} hits')

    Z_in_mask_arr = Z_v[in_mask]
    u_in_mask_f   = u_f_v[in_mask]
    v_in_mask_f   = v_f_v[in_mask]
    obj_filter_ok = False

    if n_hits >= 2:
        Z_sorted = np.sort(Z_in_mask_arr)
        best_count, best_lo, best_hi = 0, Z_sorted[0], Z_sorted[0]+MAX_DEPTH_SPAN
        for z0 in Z_sorted:
            z1 = z0 + MAX_DEPTH_SPAN
            cnt = int(np.sum((Z_in_mask_arr>=z0)&(Z_in_mask_arr<=z1)))
            if cnt > best_count:
                best_count, best_lo, best_hi = cnt, float(z0), float(z1)
        valid_obj_mask = (Z_in_mask_arr>=best_lo)&(Z_in_mask_arr<=best_hi)
        n_valid_obj    = int(valid_obj_mask.sum())
        n_rejected     = n_hits - n_valid_obj
        u_obj = u_in_mask_f[valid_obj_mask]; v_obj = v_in_mask_f[valid_obj_mask]
        Z_obj = Z_in_mask_arr[valid_obj_mask]
        obj_filter_ok = (n_valid_obj >= 3)
        print(f'  window [{best_lo:.2f},{best_hi:.2f}]m  valid={n_valid_obj}  rej={n_rejected}')
    else:
        valid_obj_mask = np.ones(n_hits,dtype=bool) if n_hits>0 else np.array([],dtype=bool)
        u_obj=u_in_mask_f.copy(); v_obj=v_in_mask_f.copy(); Z_obj=Z_in_mask_arr.copy()
        n_valid_obj=n_hits; n_rejected=0
        best_lo = float(Z_in_mask_arr.min()) if n_hits>0 else 0.
        best_hi = best_lo + MAX_DEPTH_SPAN
        print('  < 2 in-mask hits — skipping depth-window filter')

    Z_lidar_front_obj = float(Z_obj.min()) if (obj_filter_ok and n_valid_obj>0) else None
    if Z_lidar_front_obj is not None:
        print(f'  Z_lidar_front_obj = {Z_lidar_front_obj:.3f}m')

    if make_plots:
        fig, axes = plt.subplots(1,2,figsize=(14,5))
        if n_hits > 0:
            bins = np.linspace(Z_in_mask_arr.min()-.5,Z_in_mask_arr.max()+.5,
                               max(20,n_hits//2+1))
            axes[0].hist(Z_in_mask_arr[~valid_obj_mask],bins=bins,color='tomato',
                         alpha=.8,label=f'rejected ({n_rejected})',zorder=3)
            axes[0].hist(Z_in_mask_arr[valid_obj_mask],bins=bins,color='steelblue',
                         alpha=.85,label=f'valid ({n_valid_obj})',zorder=4)
            axes[0].axvspan(best_lo,best_hi,alpha=.12,color='green',
                            label=f'window [{best_lo:.1f},{best_hi:.1f}]m')
            axes[0].axvline(best_lo,color='green',lw=2,ls='--')
            axes[0].axvline(best_hi,color='green',lw=2,ls='--')
            axes[0].set_xlabel('LiDAR depth (m)'); axes[0].set_ylabel('count')
            axes[0].set_title(f'Depth distribution  obj [{obj_idx}]')
            axes[0].legend(fontsize=9); axes[0].grid(alpha=.3)
            if n_hits > 1:
                order = np.argsort(Z_in_mask_arr)
                colors = ['steelblue' if valid_obj_mask[order[i]] else 'tomato'
                          for i in range(n_hits)]
                axes[1].scatter(np.arange(n_hits),Z_in_mask_arr[order],c=colors,s=30,alpha=.8)
                axes[1].axhline(best_lo,color='green',lw=1.5,ls='--',label=f'lo={best_lo:.2f}m')
                axes[1].axhline(best_hi,color='green',lw=1.5,ls='--',label=f'hi={best_hi:.2f}m')
                axes[1].set_xlabel('rank'); axes[1].set_ylabel('Z (m)')
                axes[1].set_title('Sorted depths')
                axes[1].legend(fontsize=9); axes[1].grid(alpha=.3)
        plt.suptitle(f'Depth-window filter — obj [{obj_idx}]',fontsize=12)
        plt.tight_layout(); plt.show()

        fig, axes = plt.subplots(1,2,figsize=(18,7))
        for ax, do_zoom in zip(axes,[False,True]):
            ax.imshow(img_rgb)
            for cnt in cv2.findContours(binary_mask.astype(np.uint8),
                                         cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)[0]:
                cp=cnt[:,0,:]
                ax.plot(np.append(cp[:,0],cp[0,0]),np.append(cp[:,1],cp[0,1]),
                        color=sel_color_f,lw=1.5)
            if n_rejected > 0:
                ax.scatter(u_in_mask_f[~valid_obj_mask],v_in_mask_f[~valid_obj_mask],
                           c='red',s=50,alpha=.85,zorder=5,label=f'rejected ({n_rejected})',
                           edgecolors='darkred',linewidths=.5)
            if n_valid_obj > 0:
                sc = ax.scatter(u_obj,v_obj,c=Z_obj,cmap='plasma_r',s=60,alpha=.95,
                                zorder=6,vmin=best_lo-1,vmax=best_hi+1,
                                label=f'valid ({n_valid_obj})',edgecolors='white',linewidths=.3)
                plt.colorbar(sc,ax=ax,label='depth (m)',fraction=.025)
            ax.legend(loc='upper right',fontsize=9); ax.axis('off')
            if do_zoom:
                ax.set_xlim(xl,xr); ax.set_ylim(yb,yt)
            ax.set_title(f'{"Zoom" if do_zoom else "Full frame"} — frame {FRAME_IDX} | obj [{obj_idx}]')
        plt.suptitle(f'Depth-window filter | valid={n_valid_obj}  rej={n_rejected}',fontsize=12)
        plt.tight_layout(); plt.show()

    return dict(n_hits=n_hits, n_valid_obj=n_valid_obj, n_rejected=n_rejected,
                u_obj=u_obj, v_obj=v_obj, Z_obj=Z_obj,
                obj_filter_ok=obj_filter_ok, Z_lidar_front_obj=Z_lidar_front_obj,
                valid_obj_mask=valid_obj_mask,
                u_in_mask_f=u_in_mask_f, v_in_mask_f=v_in_mask_f,
                Z_in_mask_arr=Z_in_mask_arr, best_lo=best_lo, best_hi=best_hi)

print('_filter_lidar_obj defined.')


In [ ]:
# ── §7 helper: local affine + MAD calibration ────────────────────────────────
def _fit_local_affine_obj(obj_idx, filt, frame_cache, config, make_plots=True):
    """Fit local affine Z_lidar = a*Z_moge + b with MAD rejection. Returns state dict."""
    u_obj = filt['u_obj']; v_obj = filt['v_obj']; Z_obj = filt['Z_obj']
    n_valid_obj = filt['n_valid_obj']
    Z_moge_map  = frame_cache['Z_moge_map']
    a_global    = frame_cache['a_global']; b_global = frame_cache['b_global']
    r2_g        = frame_cache['r2_g'];     rmse_g   = frame_cache['rmse_g']
    Z_moge_g    = frame_cache['Z_moge_g']; Z_lidar_g = frame_cache['Z_lidar_g']
    n_fit_g     = frame_cache['n_fit_g']
    MAD_SIGMA_THRESH = config['MAD_SIGMA_THRESH']
    MIN_MOGE_SPAN    = config['MIN_MOGE_SPAN']
    MIN_LIDAR_SPAN   = config['MIN_LIDAR_SPAN']
    FRAME_IDX        = config['FRAME_IDX']

    if n_valid_obj > 0:
        u_oi = np.round(u_obj).astype(int); v_oi = np.round(v_obj).astype(int)
        Z_ms = Z_moge_map[v_oi, u_oi]
        vld  = np.isfinite(Z_ms) & (Z_obj > 0.5)
        Z_moge_l_init  = Z_ms[vld]; Z_lidar_l_init = Z_obj[vld]
    else:
        Z_moge_l_init = Z_lidar_l_init = np.array([])
    n_fit_l_init = len(Z_moge_l_init)

    local_fit_ok   = False
    n_mad_rejected = 0
    inlier_mask_l  = np.ones(max(n_fit_l_init,0), dtype=bool)
    a_local_init   = b_local_init = float('nan')
    Z_moge_l       = Z_moge_l_init.copy(); Z_lidar_l = Z_lidar_l_init.copy()
    n_fit_l        = n_fit_l_init
    r2_l = rmse_l  = float('nan'); rel_err_l = np.array([])

    if n_fit_l_init >= 3:
        moge_span  = float(Z_moge_l_init.max() - Z_moge_l_init.min())
        lidar_span = float(Z_lidar_l_init.max() - Z_lidar_l_init.min())
        if moge_span < MIN_MOGE_SPAN or lidar_span < MIN_LIDAR_SPAN:
            print(f'[local fit] low variation → plain fit')
            A_l = np.stack([Z_moge_l,np.ones_like(Z_moge_l)],axis=1)
            (a_local,b_local),_,_,_ = np.linalg.lstsq(A_l,Z_lidar_l,rcond=None)
            a_local_init,b_local_init = a_local,b_local
            local_fit_ok = True
        else:
            A0 = np.stack([Z_moge_l_init,np.ones_like(Z_moge_l_init)],axis=1)
            (a_local_init,b_local_init),_,_,_ = np.linalg.lstsq(A0,Z_lidar_l_init,rcond=None)
            res    = Z_lidar_l_init-(a_local_init*Z_moge_l_init+b_local_init)
            med_r  = float(np.median(res))
            mad    = float(np.median(np.abs(res-med_r)))
            thresh = MAD_SIGMA_THRESH*1.4826*mad
            inlier_mask_l  = np.abs(res-med_r)<=thresh
            n_mad_rejected = int((~inlier_mask_l).sum())
            n_inliers      = int(inlier_mask_l.sum())
            print(f'[local MAD] n={n_fit_l_init}  rejected={n_mad_rejected}')
            if n_inliers < 3:
                inlier_mask_l = np.ones(n_fit_l_init,dtype=bool); n_mad_rejected=0
            else:
                Z_moge_l=Z_moge_l_init[inlier_mask_l]; Z_lidar_l=Z_lidar_l_init[inlier_mask_l]
                n_fit_l=n_inliers
            A_l = np.stack([Z_moge_l,np.ones_like(Z_moge_l)],axis=1)
            (a_local,b_local),_,_,_ = np.linalg.lstsq(A_l,Z_lidar_l,rcond=None)
            local_fit_ok = True
    else:
        print(f'[local fit] n={n_fit_l_init} < 3 → global fallback')

    if not local_fit_ok:
        a_local,b_local = a_global,b_global
    else:
        Z_pred = a_local*Z_moge_l+b_local
        ss_res = float(np.sum((Z_lidar_l-Z_pred)**2))
        ss_tot = float(np.sum((Z_lidar_l-Z_lidar_l.mean())**2)) if len(Z_lidar_l)>1 else 1.
        r2_l   = 1.-ss_res/ss_tot if ss_tot>0 else float('nan')
        rmse_l = float(np.sqrt(ss_res/n_fit_l))
        print(f'[local fit] n={n_fit_l}  Z={a_local:.4f}×Z_moge+{b_local:.4f}  '
              f'R²={r2_l:.4f}  RMSE={rmse_l:.3f}m')

    if make_plots:
        fig, axes = plt.subplots(1,2,figsize=(16,6))
        ax=axes[0]
        if n_fit_l_init > 0:
            z_rng=np.linspace(Z_moge_l_init.min()-.02,Z_moge_l_init.max()+.02,200)
            if n_mad_rejected>0:
                ax.scatter(Z_moge_l_init[~inlier_mask_l],Z_lidar_l_init[~inlier_mask_l],
                           c='tomato',s=70,alpha=.9,zorder=4,
                           label=f'MAD out ({n_mad_rejected})',edgecolors='darkred',lw=.6)
            ax.scatter(Z_moge_l_init[inlier_mask_l],Z_lidar_l_init[inlier_mask_l],
                       c='mediumseagreen',s=70,alpha=.85,zorder=5,
                       label=f'inliers ({int(inlier_mask_l.sum())})',edgecolors='darkgreen',lw=.6)
            if np.isfinite(a_local_init):
                ax.plot(z_rng,a_local_init*z_rng+b_local_init,
                        color='darkorange',lw=1.8,ls='--',zorder=3,
                        label=f'init: {a_local_init:.3f}xZ+{b_local_init:.3f}')
            if local_fit_ok:
                ax.plot(z_rng,a_local*z_rng+b_local,color='crimson',lw=2.5,ls='-',zorder=6,
                        label=f'robust: {a_local:.3f}xZ+{b_local:.3f}')
        ax.set_xlabel('Z_moge'); ax.set_ylabel('Z_lidar (m)')
        ax.set_title(f'MoGe vs LiDAR — MAD  obj [{obj_idx}]')
        ax.legend(fontsize=8); ax.grid(alpha=.3)
        ax2=axes[1]
        if n_fit_l_init>0 and np.isfinite(a_local_init):
            res_p=Z_lidar_l_init-(a_local_init*Z_moge_l_init+b_local_init)
            xs=np.arange(n_fit_l_init)
            if n_mad_rejected>0:
                ax2.scatter(xs[~inlier_mask_l],res_p[~inlier_mask_l],
                            c='tomato',s=70,alpha=.9,zorder=4,label=f'rejected ({n_mad_rejected})')
            ax2.scatter(xs[inlier_mask_l],res_p[inlier_mask_l],
                        c='mediumseagreen',s=70,alpha=.85,zorder=5,
                        label=f'inliers ({int(inlier_mask_l.sum())})')
            med_r=float(np.median(res_p)); mad_r=float(np.median(np.abs(res_p-med_r)))
            thr_r=MAD_SIGMA_THRESH*1.4826*mad_r
            ax2.axhline(med_r,color='black',lw=1.,ls=':'
                        ); ax2.axhline(0,color='black',lw=.8)
            ax2.axhline(med_r+thr_r,color='orange',lw=1.5,ls='--',
                        label=f'±{MAD_SIGMA_THRESH}σ_MAD ({thr_r:.3f}m)')
            ax2.axhline(med_r-thr_r,color='orange',lw=1.5,ls='--')
            ax2.set_xlabel('point index'); ax2.set_ylabel('residual (m)')
            ax2.set_title(f'Residuals  MAD={mad_r:.4f}')
            ax2.legend(fontsize=8); ax2.grid(alpha=.3)
        plt.suptitle(f'MAD rejection — obj [{obj_idx}]',fontsize=12)
        plt.tight_layout(); plt.show()

        fig, axes2 = plt.subplots(1,2,figsize=(14,6))
        ax=axes2[0]
        ax.scatter(Z_moge_g,Z_lidar_g,c='steelblue',s=4,alpha=.35,label='all in-image')
        z_rng=np.linspace(Z_moge_g.min(),Z_moge_g.max(),200)
        ax.plot(z_rng,a_global*z_rng+b_global,'r-',lw=2,
                label=f'{a_global:.3f}xZ+{b_global:.3f}')
        ax.set_xlabel('Z_moge'); ax.set_ylabel('Z_lidar (m)')
        ax.set_title(f'Global  R²={r2_g:.3f}  RMSE={rmse_g:.3f}m  (n={n_fit_g:,})')
        ax.legend(fontsize=9); ax.grid(alpha=.3)
        ax2b=axes2[1]
        if local_fit_ok and n_fit_l_init>0:
            ax2b.scatter(Z_moge_l_init[~inlier_mask_l],Z_lidar_l_init[~inlier_mask_l],
                         c='tomato',s=60,alpha=.85,zorder=4,label=f'MAD out ({n_mad_rejected})')
            ax2b.scatter(Z_moge_l_init[inlier_mask_l],Z_lidar_l_init[inlier_mask_l],
                         c='mediumseagreen',s=60,alpha=.85,zorder=4,
                         label=f'inliers ({inlier_mask_l.sum()})')
            z_rng2=np.linspace(Z_moge_l_init.min(),Z_moge_l_init.max(),200)
            ax2b.plot(z_rng2,a_local*z_rng2+b_local,'darkgreen',lw=2.5,
                      label=f'robust: {a_local:.3f}xZ+{b_local:.3f}')
            ax2b.set_title(f'Local  R²={r2_l:.3f}  RMSE={rmse_l:.3f}m')
        else:
            ax2b.text(.5,.5,'Local fit unavailable',ha='center',va='center',
                      transform=ax2b.transAxes,fontsize=12)
        ax2b.set_xlabel('Z_moge'); ax2b.set_ylabel('Z_lidar (m)')
        ax2b.legend(fontsize=9); ax2b.grid(alpha=.3)
        plt.suptitle(f'Affine calibration — frame {FRAME_IDX} | obj [{obj_idx}]',fontsize=12)
        plt.tight_layout(); plt.show()

    return dict(a_local=a_local, b_local=b_local, r2_l=r2_l, rmse_l=rmse_l,
                local_fit_ok=local_fit_ok, n_mad_rejected=n_mad_rejected,
                inlier_mask_l=inlier_mask_l, n_fit_l_init=n_fit_l_init,
                Z_moge_l_init=Z_moge_l_init, Z_lidar_l_init=Z_lidar_l_init,
                a_local_init=a_local_init, b_local_init=b_local_init)

print('_fit_local_affine_obj defined.')


In [ ]:
# ── §8–9 helper: build pointmaps + run SAM3D inference ───────────────────────
def _run_inference_obj(obj_idx, inst, filt, aff, frame_cache, config, make_plots=True):
    """Build fused pointmaps, run SAM3D, extract meshes. Returns state dict."""
    binary_mask = inst['binary_mask']
    xl,xr,yt,yb = inst['xl'],inst['xr'],inst['yt'],inst['yb']
    xs_m,ys_m   = inst['xs_m'],inst['ys_m']
    u_obj=filt['u_obj']; v_obj=filt['v_obj']; Z_obj=filt['Z_obj']
    n_valid_obj = filt['n_valid_obj']; Z_lidar_front_obj = filt['Z_lidar_front_obj']
    a_local=aff['a_local']; b_local=aff['b_local']
    Z_moge_map  = frame_cache['Z_moge_map']; rgb = frame_cache['rgb']
    a_global    = frame_cache['a_global'];   b_global = frame_cache['b_global']
    u_grid=frame_cache['u_grid']; v_grid=frame_cache['v_grid']
    fx,fy,cx,cy = frame_cache['fx'],frame_cache['fy'],frame_cache['cx'],frame_cache['cy']
    K_np        = frame_cache['K_np']
    R_cam2ego   = frame_cache['R_cam2ego']; t_cam2ego = frame_cache['t_cam2ego']
    FRAME_IDX   = config["FRAME_IDX"]

    def _build_ptmap(a, b, label):
        Z_m = a*Z_moge_map+b
        Z_m = np.where(np.isfinite(Z_moge_map), np.maximum(Z_m,.1), np.nan)
        X_r3=(u_grid-cx)*Z_m/fx; Y_r3=(v_grid-cy)*Z_m/fy
        ptmap=np.stack([-X_r3,-Y_r3,Z_m],axis=-1).astype(np.float32)
        print(f'{label}: a={a:.4f} b={b:.4f}  Z∈[{np.nanmin(Z_m):.1f},{np.nanmax(Z_m):.1f}]m')
        return ptmap, Z_m

    ptmap_global_np, Z_metric_global = _build_ptmap(a_global, b_global, 'Global ptmap')
    ptmap_local_np,  Z_metric_local  = _build_ptmap(a_local,  b_local,  'Local  ptmap')

    if make_plots and n_valid_obj > 0:
        u_oi=np.round(u_obj).astype(int); v_oi=np.round(v_obj).astype(int)
        Z_fg=Z_metric_global[v_oi,u_oi]; Z_fl=Z_metric_local[v_oi,u_oi]
        print('Fused ptmap validation:')
        lims=[min(Z_obj.min(),Z_fg.min(),Z_fl.min())-.5,
              max(Z_obj.max(),Z_fg.max(),Z_fl.max())+.5]
        for lbl,Z_f in [('Global',Z_fg),('Local',Z_fl)]:
            err=Z_f-Z_obj
            print(f'  {lbl}: RMSE={np.sqrt((err**2).mean()):.3f}m  mean={err.mean():+.3f}m')
        fig,axes=plt.subplots(1,2,figsize=(14,6))
        for ax,(lbl,Z_f,col) in zip(axes,[('Global',Z_fg,'steelblue'),('Local',Z_fl,'darkorange')]):
            err=Z_f-Z_obj
            ax.scatter(Z_obj,Z_f,c=col,s=40,alpha=.75,edgecolors='none',zorder=5)
            ax.plot(lims,lims,'r--',lw=1.5,label='y=x')
            ax.set_xlabel('Z_LiDAR'); ax.set_ylabel(f'Z_fused_{lbl.lower()}')
            ax.set_title(f'{lbl}  RMSE={np.sqrt((err**2).mean()):.3f}m')
            ax.set_xlim(lims); ax.set_ylim(lims)
            ax.legend(fontsize=9); ax.grid(alpha=.3)
        plt.suptitle(f'Fused ptmap vs LiDAR — obj [{obj_idx}]',fontsize=12)
        plt.tight_layout(); plt.show()

    def _run_inf(ptmap_np, label):
        ptmap_t=torch.tensor(ptmap_np,dtype=torch.float32)
        print(f'Running SAM3D [{label}]...')
        out=inference_model(rgb,binary_mask,seed=42,pointmap=ptmap_t)
        print(f'  translation={out["translation"].squeeze().tolist()}')
        return out

    output_global = _run_inf(ptmap_global_np, 'Global')
    output_local  = _run_inf(ptmap_local_np,  'Local')
    verts_g,faces_g = mesh_to_r3(output_global)
    verts_l,faces_l = mesh_to_r3(output_local)

    m_g = compute_metrics(verts_g,binary_mask,K_np,Z_lidar_front_obj,'Global raw')
    m_l = compute_metrics(verts_l,binary_mask,K_np,Z_lidar_front_obj,'Local  raw')
    for lbl,m in [('Global',m_g),('Local',m_l)]:
        ze_s=(f'{m["Z_front"]-Z_lidar_front_obj:+.3f}m' if Z_lidar_front_obj else '--')
        print(f'[{lbl}] IoU={m["iou"]:.3f}  cov={m["coverage"]:.3f}  Z_err={ze_s}')

    # Quick preliminary OBB yaw for VAR
    _ve=((R_cam2ego.astype(np.float64)@verts_l.astype(np.float64).T).T
         +t_cam2ego.astype(np.float64))
    _xy=_ve[:,:2]-_ve[:,:2].mean(0)
    _,_ev=np.linalg.eigh(np.cov(_xy.T)); _ev=_ev[:,::-1]
    obb_yaw_prelim=float(np.arctan2(_ev[1,0],_ev[0,0]))
    print(f'Prelim OBB yaw: {np.degrees(obb_yaw_prelim):.1f}°')

    if make_plots:
        fig,axes=plt.subplots(1,2,figsize=(18,7))
        for ax,(label,verts,m) in zip(axes,[('Global raw',verts_g,m_g),('Local  raw',verts_l,m_l)]):
            img_m=draw_mesh_on_image(frame_cache['img_rgb'],verts,K_np,color=(255,100,0))
            ax.imshow(img_m[yt:yb,xl:xr])
            bb=m['bbox_mesh']
            ax.add_patch(plt.Rectangle((bb[0]-xl,bb[1]-yt),bb[2]-bb[0],bb[3]-bb[1],
                                       lw=2,edgecolor='red',facecolor='none',label='mesh bbox'))
            ax.add_patch(plt.Rectangle((xs_m.min()-xl,ys_m.min()-yt),
                                       xs_m.max()-xs_m.min(),ys_m.max()-ys_m.min(),
                                       lw=2,edgecolor='lime',facecolor='none',label='mask bbox'))
            if n_valid_obj>0:
                ax.scatter(u_obj-xl,v_obj-yt,c=Z_obj,cmap='cool',s=30,alpha=.7,
                           zorder=6,label='LiDAR (filtered)')
            ax.set_title(f'{label}\nIoU={m["iou"]:.3f}  cov={m["coverage"]:.3f}  '
                         f'Z_err={m["Z_front_err"]:+.3f}m\n'
                         f'dims={m["dims"][0]:.2f}×{m["dims"][1]:.2f}×{m["dims"][2]:.2f}m')
            ax.legend(loc='upper right',fontsize=7); ax.axis('off')
        plt.suptitle(f'Option C raw meshes — obj [{obj_idx}]',fontsize=12)
        plt.tight_layout(); plt.show()

    return dict(verts_g=verts_g, faces_g=faces_g, verts_l=verts_l, faces_l=faces_l,
                m_g=m_g, m_l=m_l, obb_yaw_prelim=obb_yaw_prelim)

print('_run_inference_obj defined.')


In [ ]:
# ── §10–12 helper: VAR refinement + comparison viz + metrics ─────────────────
def _run_var_obj(obj_idx, inst, filt, inf, frame_cache, config, make_plots=True):
    """Run VAR1/2/3, comparison plot, metrics table. Returns state dict."""
    binary_mask = inst['binary_mask']
    xl,xr,yt,yb = inst['xl'],inst['xr'],inst['yt'],inst['yb']
    u_obj=filt['u_obj']; v_obj=filt['v_obj']; Z_obj=filt['Z_obj']
    n_valid_obj=filt['n_valid_obj']; Z_lidar_front_obj=filt['Z_lidar_front_obj']
    obj_filter_ok=filt['obj_filter_ok']
    verts_g=inf['verts_g']; verts_l=inf['verts_l']
    m_g=inf['m_g']; m_l=inf['m_l']; obb_yaw_prelim=inf['obb_yaw_prelim']
    K_np=frame_cache['K_np']
    R_cam2ego=frame_cache['R_cam2ego']; t_cam2ego=frame_cache['t_cam2ego']
    R_l2e=frame_cache['R_l2e']; t_l2e=frame_cache['t_l2e']
    xyz_lid=frame_cache['xyz_lid']
    img_rgb=frame_cache['img_rgb']
    OBJ_PRIOR_DIMS=config['OBJ_PRIOR_DIMS']
    FRAME_IDX=config['FRAME_IDX']
    a_global=frame_cache['a_global']; b_global=frame_cache['b_global']
    r2_g=frame_cache['r2_g']


    if obj_filter_ok and n_valid_obj>=3:
        u_var=u_obj.astype(np.float64); v_var=v_obj.astype(np.float64)
        Z_var=Z_obj.astype(np.float64)
    else:
        u_var=v_var=Z_var=np.array([],dtype=np.float64)
        print('No usable LiDAR — VAR returns raw meshes.')

    _Rc2e=R_cam2ego.astype(np.float64); _tc2e=t_cam2ego.astype(np.float64)
    _kw=dict(binary_mask=binary_mask,K=K_np,u_lf=u_var,v_lf=v_var,Z_lf=Z_var,
             R_c2e=_Rc2e,t_c2e=_tc2e,current_obb_yaw=float(obb_yaw_prelim))
    vl64=verts_l.astype(np.float64)

    verts_var1,info_var1=visibility_aware_refinement(vl64,do_yaw=False,do_scale=False,**_kw)
    m_var1=compute_metrics(verts_var1,binary_mask,K_np,Z_lidar_front_obj,'VAR1')
    dist_var1,_=compute_lidar_vis_mesh_dist(verts_var1,binary_mask,K_np,u_var,v_var,Z_var,_Rc2e,_tc2e)

    verts_var2,info_var2=visibility_aware_refinement(vl64,do_yaw=True, do_scale=False,**_kw)
    m_var2=compute_metrics(verts_var2,binary_mask,K_np,Z_lidar_front_obj,'VAR2')
    dist_var2,_=compute_lidar_vis_mesh_dist(verts_var2,binary_mask,K_np,u_var,v_var,Z_var,_Rc2e,_tc2e)

    verts_var3,info_var3=visibility_aware_refinement(vl64,do_yaw=True, do_scale=True, **_kw)
    m_var3=compute_metrics(verts_var3,binary_mask,K_np,Z_lidar_front_obj,'VAR3')
    dist_var3,_=compute_lidar_vis_mesh_dist(verts_var3,binary_mask,K_np,u_var,v_var,Z_var,_Rc2e,_tc2e)

    dist_raw,_=compute_lidar_vis_mesh_dist(verts_l,binary_mask,K_np,u_var,v_var,Z_var,_Rc2e,_tc2e)

    # [EXPERIMENTAL] VAR4: expanded-cluster ego-facing face alignment
    verts_var4,info_var4=var4_cluster_alignment(
        vl64,binary_mask,K_np,u_var,v_var,Z_var,
        _Rc2e,_tc2e,R_l2e.astype(np.float64),t_l2e.astype(np.float64),
        xyz_lid,float(obb_yaw_prelim),
        verts_var2=verts_var2,info_var2=info_var2)
    m_var4=compute_metrics(verts_var4,binary_mask,K_np,Z_lidar_front_obj,'VAR4')
    dist_var4,_=compute_lidar_vis_mesh_dist(verts_var4,binary_mask,K_np,u_var,v_var,Z_var,_Rc2e,_tc2e)

    print()
    for lbl,iv in [('VAR1',info_var1),('VAR2',info_var2),('VAR3',info_var3),
                   ('VAR4 [EXP]',info_var4)]:
        dxy=float(np.linalg.norm(iv['delta_ego_xy']))
        extra=''
        if 'cluster_points_count' in iv:  # VAR4 keys
            extra=(f'  cl={iv["cluster_points_count"]}pts  '
                   f'mode={iv["var4_mode"]}  '
                   f'faces={iv["selected_faces"]}  '
                   f'loss:{iv["face_loss_before"]:.3f}->{iv["face_loss_after"]:.3f}m  '
                   f'rej_cls={iv["rejected_cluster_count"]}')
        elif 'n_selected_cluster' in iv:  # legacy fallback
            extra=(f'  cl={iv["n_selected_cluster"]}pts  '
                   f'seed_in={iv["n_seed_in_cluster"]}  '
                   f'face:{iv["face_dist_before"]:.3f}->{iv["face_dist_after"]:.3f}m')
        print(f'[{lbl}] {iv["status"]}  n_vis={iv["n_vis_verts"]}  '
              f'dZ={iv["delta_cam_z"]:+.3f}m  dXY={dxy:.3f}m  '
              f'dyaw={iv["delta_yaw_deg"]:+.1f}°  s={iv["scale"]:.3f}  '
              f'dist:{iv["lidar_mesh_dist_before"]:.3f}→{iv["lidar_mesh_dist"]:.3f}m{extra}')

    if make_plots:
        print('Generating VAR2 debug plot...')
        visualize_var_debug(verts_before=verts_l,verts_after=verts_var2,info=info_var2,
            binary_mask=binary_mask,img_rgb=img_rgb,K=K_np,
            u_lf=u_var,v_lf=v_var,Z_lf=Z_var,R_c2e=_Rc2e,t_c2e=_tc2e,
            xl=xl,xr=xr,yt=yt,yb=yb,label='VAR2: yaw+Z+trans',
            frame_idx=FRAME_IDX,obj_idx=obj_idx)

        print('Generating VAR4 [EXPERIMENTAL] debug plot...')
        visualize_var4_debug(verts_before=verts_l,verts_after=verts_var4,info=info_var4,
            binary_mask=binary_mask,img_rgb=img_rgb,K=K_np,
            u_lf=u_var,v_lf=v_var,Z_lf=Z_var,
            R_c2e=_Rc2e,t_c2e=_tc2e,
            R_l2e=R_l2e.astype(np.float64),t_l2e=t_l2e.astype(np.float64),
            xl=xl,xr=xr,yt=yt,yb=yb,label='VAR4 [EXP]: cluster-face',
            frame_idx=FRAME_IDX,obj_idx=obj_idx)

        

        prior_lwh_m=np.array([OBJ_PRIOR_DIMS[2],OBJ_PRIOR_DIMS[0],OBJ_PRIOR_DIMS[1]])
        rows=[('Global raw',m_g,None,dist_raw),('Local raw',m_l,None,dist_raw),
              ('VAR1: Z+trans',m_var1,info_var1,dist_var1),
              ('VAR2: yaw+Z+trans',m_var2,info_var2,dist_var2),
              ('VAR3: +scale (diag)',m_var3,info_var3,dist_var3),
              ('VAR4: cluster-face [*]',m_var4,info_var4,dist_var4)]
        hdr=(f'{"Strategy":26s}  {"IoU":>6}  {"Cov":>6}  {"Zferr":>8}  '
             f'{"dim_err":>9}  {"Chamfer":>9}  {"dZ":>7}  {"dXY":>7}  {"dyaw°":>8}')
        print('='*95)
        print(f'Frame {FRAME_IDX}  obj [{obj_idx}]  n_lidar={n_valid_obj}')
        print('='*95); print(hdr); print('-'*95)
        for lbl,m,info,dist in rows:
            ze=m['Z_front']-Z_lidar_front_obj if Z_lidar_front_obj else float('nan')
            de=float(np.abs(m['dims']-prior_lwh_m).mean()) if 'dims' in m else float('nan')
            dz =info['delta_cam_z']                  if info else float('nan')
            dxy=np.linalg.norm(info['delta_ego_xy']) if info else float('nan')
            dy =info['delta_yaw_deg']                if info else float('nan')
            tag=' <-- DEFAULT' if '[*]' in lbl else ''
            print(f'{lbl:26s}  {m["iou"]:6.3f}  {m["coverage"]:6.3f}  {ze:+8.3f}  '
                  f'{de:9.4f}  {dist:9.4f}  {dz:+7.3f}  {dxy:7.3f}  {dy:+8.1f}{tag}')
        print('='*95)

    return dict(verts_var1=verts_var1,verts_var2=verts_var2,verts_var3=verts_var3,
                verts_var4=verts_var4,
                m_var1=m_var1,m_var2=m_var2,m_var3=m_var3,m_var4=m_var4,
                info_var1=info_var1,info_var2=info_var2,info_var3=info_var3,info_var4=info_var4,
                dist_raw=dist_raw,dist_var1=dist_var1,dist_var2=dist_var2,
                dist_var3=dist_var3,dist_var4=dist_var4,
                u_var=u_var,v_var=v_var,Z_var=Z_var,_Rc2e=_Rc2e,_tc2e=_tc2e)

print('_run_var_obj defined.')

In [ ]:
# ── §13–14 helper: OBB + ground diagnostic + LiDAR 3D/BEV ───────────────────
def _run_obb_and_lidar_plots(obj_idx, inst, filt, inf, var, frame_cache, config, make_plots=True):
    """RANSAC ground, OBB computation, and all 3D/BEV plots. Returns state dict."""
    binary_mask=inst['binary_mask']; xl,xr,yt,yb=inst['xl'],inst['xr'],inst['yt'],inst['yb']
    u_obj=filt['u_obj']; v_obj=filt['v_obj']; Z_obj=filt['Z_obj']
    n_valid_obj=filt['n_valid_obj']; Z_lidar_front_obj=filt['Z_lidar_front_obj']
    verts_g=inf['verts_g']; verts_l=inf['verts_l']
    verts_var2=var['verts_var2']; verts_var3=var['verts_var3']
    verts_var4=var.get('verts_var4', verts_var2)  # VAR4 [EXPERIMENTAL]
    u_var=var['u_var']; v_var=var['v_var']; Z_var=var['Z_var']
    xyz_lid=frame_cache['xyz_lid']
    K_np=frame_cache['K_np']
    fx,fy,cx,cy=frame_cache['fx'],frame_cache['fy'],frame_cache['cx'],frame_cache['cy']
    R_cam2ego=frame_cache['R_cam2ego']; t_cam2ego=frame_cache['t_cam2ego']
    R_ego2cam=frame_cache['R_ego2cam']; t_ego2cam=frame_cache['t_ego2cam']
    R_l2e=frame_cache['R_l2e']; t_l2e=frame_cache['t_l2e']
    img_rgb=frame_cache['img_rgb']
    OBJ_PRIOR_DIMS=config['OBJ_PRIOR_DIMS']
    SCENE_IDX=config['NUSCENES_SCENE_IDX']; FRAME_IDX=config['FRAME_IDX']

    _vl64=verts_l.astype(np.float64)
    _Rc2e=R_cam2ego.astype(np.float64); _tc2e=t_cam2ego.astype(np.float64)

    _corners_prelim,_center_prelim,_dims_prelim,_,_=compute_obb_gravity_aligned(
        _vl64,_Rc2e,_tc2e)
    _excl_r=float(max(_dims_prelim[0],_dims_prelim[1]))*0.55+0.3

    gnd=estimate_local_ground(xyz_lid,R_l2e,t_l2e,obj_center_ego=_center_prelim,
        crop_r=10.,excl_r=_excl_r,bot_frac=.35,ransac_n=250,ransac_thresh=.07,
        min_inliers=15,max_tilt_deg=12.)
    print(f'Ground: {gnd["msg"]}')

    if gnd['success']:
        ground_z=gnd['ground_z']; _gnd_src='RANSAC'
    else:
        _vz=(_Rc2e@_vl64.T).T[:,2]+_tc2e[2]
        ground_z=float(np.percentile(_vz,8))
        _gnd_src=f'FALLBACK(mesh p8={ground_z:+.3f}m)'
        print(f'WARNING: RANSAC failed → ground_z={ground_z:+.3f}m')

    corners_ego,center_ego,obb_dims,obb_yaw,obb_axes=compute_obb_gravity_aligned(
        _vl64,_Rc2e,_tc2e,ground_z=ground_z)
    corners_lidar=(R_l2e.T@(corners_ego-t_l2e).T).T
    corners_r3   =(R_ego2cam@corners_ego.T+t_ego2cam[:,None]).T
    print(f'OBB: {obb_dims[0]:.2f}×{obb_dims[1]:.2f}×{obb_dims[2]:.2f}m  '
          f'yaw={np.degrees(obb_yaw):.1f}°  ground_z={ground_z:+.3f}m ({_gnd_src})')

    def _obb(verts_r3):
        return compute_obb_gravity_aligned(verts_r3.astype(np.float64),_Rc2e,_tc2e,
                                           ground_z=ground_z)
    ce_g,_,obb_dims_g,obb_yaw_g,obb_axes_g=_obb(verts_g)
    cl_g=(R_l2e.T@(ce_g-t_l2e).T).T
    ce_v2,_,obb_dims_v2,obb_yaw_v2,obb_axes_v2=_obb(verts_var2)
    cl_v2=(R_l2e.T@(ce_v2-t_l2e).T).T
    ce_v3,_,obb_dims_v3,obb_yaw_v3,obb_axes_v3=_obb(verts_var3)
    cl_v3=(R_l2e.T@(ce_v3-t_l2e).T).T
    ce_v4,_,obb_dims_v4,obb_yaw_v4,obb_axes_v4=_obb(verts_var4)  # VAR4 [EXP]
    cl_v4=(R_l2e.T@(ce_v4-t_l2e).T).T

    prior_lwh=np.array([OBJ_PRIOR_DIMS[2],OBJ_PRIOR_DIMS[0],OBJ_PRIOR_DIMS[1]])
    cmp_rows=[('Global raw',obb_dims_g,obb_yaw_g,cl_g),
              ('Local raw',obb_dims,obb_yaw,corners_lidar),
              ('VAR2',obb_dims_v2,obb_yaw_v2,cl_v2),
              ('VAR3 (+scale)',obb_dims_v3,obb_yaw_v3,cl_v3),
              ('VAR4 [EXP]',obb_dims_v4,obb_yaw_v4,cl_v4),
              ('Car prior',prior_lwh,float('nan'),None)]
    print(f'\n{"":20s}  {"L":>6}  {"W":>6}  {"H":>6}  {"Yaw°":>7}  {"dim_err":>8}  {"ctrZ":>7}')
    print('-'*70)
    for lbl,dims,yaw,cl in cmp_rows:
        de=float(np.abs(dims-prior_lwh).mean())
        cz=f'{cl.mean(0)[2]:+7.2f}' if cl is not None else '   -- '
        ys=f'{np.degrees(yaw):7.1f}' if np.isfinite(yaw) else '   -- '
        print(f'{lbl:20s}  {dims[0]:6.2f}  {dims[1]:6.2f}  {dims[2]:6.2f}  {ys}  {de:8.3f}  {cz}')
    print(f'ground_z = {ground_z:+.3f}m ({_gnd_src})')

    if not make_plots:
        return dict(obb_dims=obb_dims, obb_yaw=obb_yaw, ground_z=ground_z,
                gnd_src=_gnd_src, gnd=gnd,
                corners_lidar=corners_lidar, obb_axes=obb_axes,
                cl_v2=cl_v2, obb_dims_v2=obb_dims_v2, obb_axes_v2=obb_axes_v2,
                cl_v4=cl_v4, obb_dims_v4=obb_dims_v4, obb_axes_v4=obb_axes_v4,
                pts_obj_lidar=pts_obj_lidar, n_valid_obj=n_valid_obj)

    # Ground diagnostic
    fig_gnd,axes_gnd=plt.subplots(1,2,figsize=(16,7))
    ax=axes_gnd[0]
    pts_low=gnd['ground_pts_ego']; inl=gnd['inlier_mask']
    if len(pts_low)>0:
        if inl is not None:
            ax.scatter(pts_low[~inl,0],pts_low[~inl,1],c='tomato',s=4,alpha=.5,
                       label='RANSAC outliers',zorder=2)
            ax.scatter(pts_low[inl,0],pts_low[inl,1],c='limegreen',s=6,alpha=.8,
                       label='RANSAC inliers',zorder=3)
        else:
            ax.scatter(pts_low[:,0],pts_low[:,1],c='gray',s=4,alpha=.5,
                       label='low pts (failed)',zorder=2)
    ax.add_patch(plt.Circle((_center_prelim[0],_center_prelim[1]),_excl_r,
                             color='gold',fill=False,ls=':',lw=1.5,
                             label=f'excl r={_excl_r:.1f}m',zorder=4))
    def _bev_obb(ax2,cnrs,color,ls,lw,label):
        zs=cnrs[:,2]; face=cnrs[:4] if zs[:4].mean()<=zs[4:].mean() else cnrs[4:]
        xs=list(face[:,0])+[face[0,0]]; ys=list(face[:,1])+[face[0,1]]
        ax2.plot(xs,ys,color=color,ls=ls,lw=lw,label=label,zorder=5)
    _bev_obb(ax,_corners_prelim,'gray','--',1.5,'OBB raw')
    _bev_obb(ax,corners_ego,'orangered','-',2.5,'OBB ground-anchored')
    ax.scatter([_center_prelim[0]],[_center_prelim[1]],c='yellow',s=60,zorder=6,marker='*')
    r_show=max(12.,_excl_r*3)
    ax.set_xlim(_center_prelim[0]-r_show,_center_prelim[0]+r_show)
    ax.set_ylim(_center_prelim[1]-r_show,_center_prelim[1]+r_show)
    ax.set_aspect('equal'); ax.grid(alpha=.3)
    ax.set_xlabel('Ego X (m)'); ax.set_ylabel('Ego Y (m)')
    ax.set_title(f'BEV — RANSAC ground\n{gnd["msg"]}',fontsize=9)
    ax.legend(loc='upper right',fontsize=8,facecolor='k',labelcolor='w',framealpha=.7)
    ax2=axes_gnd[1]
    _xyz_ego=(R_l2e@xyz_lid.T).T+t_l2e
    _dx=_xyz_ego[:,0]-_center_prelim[0]; _dy=_xyz_ego[:,1]-_center_prelim[1]
    _crop=_xyz_ego[np.sqrt(_dx**2+_dy**2)<10.]
    if len(_crop)>0:
        bins=np.linspace(float(np.percentile(_crop[:,2],1))-.3,
                         float(np.percentile(_crop[:,2],99))+.3,60)
        ax2.hist(_crop[:,2],bins=bins,color='steelblue',alpha=.4,
                 label='local crop (10m)',orientation='horizontal')
    if len(pts_low)>0:
        ax2.hist(pts_low[:,2],bins=60,color='gray',alpha=.7,
                 label=f'low pts ({gnd["n_low_pts"]})',orientation='horizontal')
        if inl is not None and inl.sum()>0:
            ax2.hist(pts_low[inl,2],bins=40,color='limegreen',alpha=.9,
                     label=f'inliers ({gnd["n_inliers"]})',orientation='horizontal')
    if gnd['ground_z'] is not None:
        ax2.axhline(gnd['ground_z'],color='gold',lw=2.5,ls='--',
                    label=f'ground_z={gnd["ground_z"]:+.3f}m')
    _vz_ego=(_Rc2e@_vl64.T).T[:,2]+_tc2e[2]
    ax2.axhline(float(np.percentile(_vz_ego,5)),color='orangered',lw=1.5,ls=':',
                label=f'mesh p5={np.percentile(_vz_ego,5):+.3f}m')
    ax2.axhline(float(np.percentile(_vz_ego,98)),color='orangered',lw=1.5,ls='-.',
                label=f'mesh p98={np.percentile(_vz_ego,98):+.3f}m')
    ax2.axhline(corners_ego[:,2].min(),color='orangered',lw=2.5,ls='-',
                label=f'OBB bottom={corners_ego[:,2].min():+.3f}m')
    ax2.axhline(corners_ego[:,2].max(),color='darkorange',lw=2.5,ls='-',
                label=f'OBB top={corners_ego[:,2].max():+.3f}m')
    ax2.set_xlabel('Count'); ax2.set_ylabel('Ego Z (m)')
    ax2.set_title(f'Z profile  OBB H={obb_dims[2]:.2f}m',fontsize=9)
    ax2.legend(loc='upper right',fontsize=8); ax2.grid(alpha=.3,axis='y')
    plt.suptitle(f'Ground diagnostic — scene {SCENE_IDX}/frame {FRAME_IDX} | obj [{obj_idx}]',fontsize=11)
    plt.tight_layout(); plt.show()

    # OBB on camera image
    BBOX_EDGES=[(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7)]
    def _proj(cnrs,K):
        valid=cnrs[:,2]>1e-3
        u=K[0,0]*cnrs[:,0]/np.maximum(cnrs[:,2],1e-6)+K[0,2]
        v=K[1,1]*cnrs[:,1]/np.maximum(cnrs[:,2],1e-6)+K[1,2]
        uv=np.stack([u,v],axis=1); uv[~valid]=np.nan; return uv
    uv_c=_proj(corners_r3,K_np)
    fig,axes_obb=plt.subplots(1,2,figsize=(18,7))
    for ax,do_zoom in zip(axes_obb,[False,True]):
        img_m=draw_mesh_on_image(img_rgb,verts_l,K_np,color=(255,140,0))
        ax.imshow(img_m)
        for ei,(a,b) in enumerate(BBOX_EDGES):
            ua,va=uv_c[a]; ub,vb=uv_c[b]
            if np.isnan(ua) or np.isnan(ub): continue
            ax.plot([ua,ub],[va,vb],
                    color='red' if ei<4 else 'deepskyblue' if ei<8 else 'white',lw=2.5,zorder=8)
        for i,(u_c2,v_c2) in enumerate(uv_c):
            if np.isnan(u_c2): continue
            ax.scatter([u_c2],[v_c2],c='yellow',s=40,zorder=10,edgecolors='black',lw=.5)
        for cnt in cv2.findContours(binary_mask.astype(np.uint8),
                                     cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)[0]:
            cp=cnt[:,0,:]
            ax.plot(np.append(cp[:,0],cp[0,0]),np.append(cp[:,1],cp[0,1]),
                    color='lime',lw=1.5,zorder=7)
        if n_valid_obj>0:
            ax.scatter(u_obj,v_obj,c=Z_obj,cmap='cool',s=25,alpha=.8,zorder=9,
                       label='LiDAR (filtered)')
        ax.legend(loc='upper right',fontsize=8); ax.axis('off')
        if do_zoom: ax.set_xlim(xl,xr); ax.set_ylim(yb,yt)
        ax.set_title(f'{"Zoom" if do_zoom else "Full frame"} — '
                     f'OBB {obb_dims[0]:.2f}×{obb_dims[1]:.2f}×{obb_dims[2]:.2f}m')
    from matplotlib.lines import Line2D
    axes_obb[1].legend(handles=[Line2D([0],[0],color=c,lw=2,label=l) for c,l in
        [('red','OBB bottom'),('deepskyblue','OBB top'),('white','OBB pillars'),('lime','mask')]],
        loc='upper right',fontsize=8,facecolor='black',labelcolor='white')
    plt.suptitle(f'OBB on image — frame {FRAME_IDX} | obj [{obj_idx}]',fontsize=12)
    plt.tight_layout(); plt.show()

    # LiDAR 3D + BEV
    if n_valid_obj>0:
        u_oa=u_obj.astype(np.float64); v_oa=v_obj.astype(np.float64)
        X_r3=(u_oa-cx)*Z_obj/fx; Y_r3=(v_oa-cy)*Z_obj/fy
        pts_obj_ego=(R_cam2ego@np.stack([X_r3,Y_r3,Z_obj],1).T).T+t_cam2ego
        pts_obj_lidar=(R_l2e.T@(pts_obj_ego-t_l2e).T).T
    else:
        pts_obj_lidar=np.zeros((0,3))

    _all_c=np.vstack([corners_lidar,cl_g,cl_v2,cl_v3,cl_v4])
    ctr_cmp=_all_c.mean(0)
    r_cmp=max(max(obb_dims.max(),obb_dims_g.max(),obb_dims_v2.max(),
                  obb_dims_v3.max(),obb_dims_v4.max())*2.5,8.)

    import plotly.graph_objects as go
    _EDGES=[(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7)]
    def _wire(cnrs,dims,lbl,col):
        xs,ys,zs=[],[],[]
        for a,b in _EDGES: xs+=[cnrs[a,0],cnrs[b,0],None];ys+=[cnrs[a,1],cnrs[b,1],None];zs+=[cnrs[a,2],cnrs[b,2],None]
        return go.Scatter3d(x=xs,y=ys,z=zs,mode='lines',line=dict(color=col,width=5),
                            name=f'{lbl} {dims[0]:.2f}×{dims[1]:.2f}×{dims[2]:.2f}m')
    def _arr(cnrs,axes_mat,dims,col,lbl):
        ctr=cnrs.mean(0); fwd=R_l2e.T@axes_mat[:,0]; tip=ctr+fwd*(dims[0]/2+0.4)
        return go.Scatter3d(x=[ctr[0],tip[0]],y=[ctr[1],tip[1]],z=[ctr[2],tip[2]],
                            mode='lines+markers',line=dict(color=col,width=6),
                            marker=dict(size=[2,8],color=col,symbol=['circle','diamond']),
                            name=f'{lbl} fwd')
    tr=[]
    tr.append(go.Scatter3d(x=xyz_lid[:,0],y=xyz_lid[:,1],z=xyz_lid[:,2],mode='markers',
               marker=dict(size=1,color=xyz_lid[:,2],colorscale='Turbo',opacity=.35,showscale=False),
               name='LiDAR',hoverinfo='skip'))
    if n_valid_obj>0:
        tr.append(go.Scatter3d(x=pts_obj_lidar[:,0],y=pts_obj_lidar[:,1],z=pts_obj_lidar[:,2],
                               mode='markers',marker=dict(size=4,color='cyan',opacity=.9),
                               name=f'LiDAR in mask ({n_valid_obj})'))
    if gnd['success'] and gnd['inlier_mask'] is not None:
        _gi=gnd['ground_pts_ego'][gnd['inlier_mask']]; _gil=(R_l2e.T@(_gi-t_l2e).T).T
        tr.append(go.Scatter3d(x=_gil[:,0],y=_gil[:,1],z=_gil[:,2],mode='markers',
                               marker=dict(size=2,color='limegreen',opacity=.6),
                               name=f'Ground inliers ({gnd["n_inliers"]})'))
    tr+=[_wire(cl_g,obb_dims_g,'Global raw','rgb(30,144,255)'),
         _arr(cl_g,obb_axes_g,obb_dims_g,'rgb(30,144,255)','Global raw'),
         _wire(corners_lidar,obb_dims,'Local raw','rgb(255,100,0)'),
         _arr(corners_lidar,obb_axes,obb_dims,'rgb(255,100,0)','Local raw'),
         _wire(cl_v2,obb_dims_v2,'VAR2','rgb(50,220,80)'),
         _arr(cl_v2,obb_axes_v2,obb_dims_v2,'rgb(50,220,80)','VAR2'),
         _wire(cl_v3,obb_dims_v3,'VAR3 +scale','rgb(200,80,220)'),
         _arr(cl_v3,obb_axes_v3,obb_dims_v3,'rgb(200,80,220)','VAR3 +scale'),
         _wire(cl_v4,obb_dims_v4,'VAR4 [EXP]','rgb(255,165,0)'),
         _arr(cl_v4,obb_axes_v4,obb_dims_v4,'rgb(255,165,0)','VAR4 [EXP]')]
    ctr=ctr_cmp; r=r_cmp
    fig14=go.Figure(data=tr)
    fig14.update_layout(
        title=f'Scene {SCENE_IDX}/Frame {FRAME_IDX} | obj [{obj_idx}] — LiDAR + OBBs<br>ground_z={ground_z:+.3f}m ({_gnd_src})',
        scene=dict(xaxis=dict(title='X',range=[ctr[0]-r,ctr[0]+r]),
                   yaxis=dict(title='Y',range=[ctr[1]-r,ctr[1]+r]),
                   zaxis=dict(title='Z',range=[ctr[2]-r,ctr[2]+r]),
                   aspectmode='cube',camera=dict(eye=dict(x=0,y=-.5,z=1.8))),
        margin=dict(l=0,r=0,b=0,t=60),height=700)
    fig14.show()

    # BEV
    fig_bev,ax_bev=plt.subplots(figsize=(9,9))
    def _bev_f(ax,cnrs,col,ls,lw,lbl):
        zs=cnrs[:,2]; face=cnrs[:4] if zs[:4].mean()<=zs[4:].mean() else cnrs[4:]
        xs=list(face[:,0])+[face[0,0]]; ys=list(face[:,1])+[face[0,1]]
        ax.plot(xs,ys,color=col,ls=ls,lw=lw,label=lbl,zorder=3)
        ax.scatter([cnrs[:,0].mean()],[cnrs[:,1].mean()],color=col,s=40,zorder=5)
    def _bev_a(ax,cnrs,axes_mat,dims,col):
        c_=cnrs.mean(0)[:2]; f=(R_l2e.T@axes_mat[:,0])[:2]; tip=c_+f*(dims[0]/2+0.4)
        ax.annotate('',xy=tip,xytext=c_,arrowprops=dict(arrowstyle='->',color=col,lw=2))
    ax_bev.scatter(xyz_lid[:,0],xyz_lid[:,1],c=xyz_lid[:,2],cmap='Greys',s=1,alpha=.2,zorder=1)
    if n_valid_obj>0:
        ax_bev.scatter(pts_obj_lidar[:,0],pts_obj_lidar[:,1],c='cyan',s=18,alpha=.85,
                       zorder=4,label=f'LiDAR in mask ({n_valid_obj})')
    if gnd['success'] and gnd['inlier_mask'] is not None:
        _gi=gnd['ground_pts_ego'][gnd['inlier_mask']]; _gil=(R_l2e.T@(_gi-t_l2e).T).T
        ax_bev.scatter(_gil[:,0],_gil[:,1],c='limegreen',s=3,alpha=.5,
                       zorder=2,label=f'Ground inliers ({gnd["n_inliers"]})')
    _bev_f(ax_bev,cl_g,'dodgerblue','-',2.5,f'Global {obb_dims_g[0]:.2f}×{obb_dims_g[1]:.2f}×{obb_dims_g[2]:.2f}m')
    _bev_a(ax_bev,cl_g,obb_axes_g,obb_dims_g,'dodgerblue')
    _bev_f(ax_bev,corners_lidar,'orangered','--',2.5,f'Local {obb_dims[0]:.2f}×{obb_dims[1]:.2f}×{obb_dims[2]:.2f}m')
    _bev_a(ax_bev,corners_lidar,obb_axes,obb_dims,'orangered')
    _bev_f(ax_bev,cl_v2,'limegreen','-.',2.5,f'VAR2 {obb_dims_v2[0]:.2f}×{obb_dims_v2[1]:.2f}×{obb_dims_v2[2]:.2f}m')
    _bev_a(ax_bev,cl_v2,obb_axes_v2,obb_dims_v2,'limegreen')
    _bev_f(ax_bev,cl_v3,'mediumpurple',(0,(3,1,1,1)),2.5,f'VAR3 {obb_dims_v3[0]:.2f}×{obb_dims_v3[1]:.2f}×{obb_dims_v3[2]:.2f}m')
    _bev_a(ax_bev,cl_v3,obb_axes_v3,obb_dims_v3,'mediumpurple')
    _bev_f(ax_bev,cl_v4,'darkorange',(0,(5,2)),2.5,f'VAR4 [EXP] {obb_dims_v4[0]:.2f}×{obb_dims_v4[1]:.2f}×{obb_dims_v4[2]:.2f}m')
    _bev_a(ax_bev,cl_v4,obb_axes_v4,obb_dims_v4,'darkorange')
    ax_bev.set_xlim(ctr_cmp[0]-r_cmp,ctr_cmp[0]+r_cmp); ax_bev.set_ylim(ctr_cmp[1]-r_cmp,ctr_cmp[1]+r_cmp)
    ax_bev.set_aspect('equal'); ax_bev.grid(alpha=.35)
    ax_bev.set_xlabel('LiDAR X (m)'); ax_bev.set_ylabel('LiDAR Y (m)')
    ax_bev.set_title(f'BEV — Scene {SCENE_IDX}/Frame {FRAME_IDX} | obj [{obj_idx}]\n'
                     f'ground_z={ground_z:+.3f}m ({_gnd_src})  ' \
                     f'blue=Global  orange=Local  green=VAR2  purple=VAR3  darkorange=VAR4[EXP]')
    ax_bev.legend(loc='upper right',fontsize=8,facecolor='k',labelcolor='w',framealpha=.7)
    plt.tight_layout(); plt.show()

    return dict(obb_dims=obb_dims, obb_yaw=obb_yaw, ground_z=ground_z,
                gnd_src=_gnd_src, gnd=gnd,
                corners_lidar=corners_lidar, obb_axes=obb_axes,
                cl_v2=cl_v2, obb_dims_v2=obb_dims_v2, obb_axes_v2=obb_axes_v2,
                cl_v4=cl_v4, obb_dims_v4=obb_dims_v4, obb_axes_v4=obb_axes_v4,
                pts_obj_lidar=pts_obj_lidar, n_valid_obj=n_valid_obj)

print('_run_obb_and_lidar_plots defined.')


In [ ]:
# ── process_object: orchestrator ─────────────────────────────────────────────
def process_object(obj_idx, frame_cache, config, make_plots=True):
    """
    Run the full per-object pipeline.  Returns a result dict for the summary table.

    Pipeline stages (each a separate helper function above):
      _select_instance        §3  instance selection + crop bounds
      _filter_lidar_obj       §4  in-mask LiDAR depth-window filter
      _fit_local_affine_obj   §7  local affine + MAD calibration
      _run_inference_obj      §8–9 fused pointmap + SAM3D inference
      _run_var_obj            §10–12 VAR1/2/3 + comparison viz + metrics
      _run_obb_and_lidar_plots §13–14 OBB + ground diagnostic + LiDAR 3D/BEV
    """
    print(f'\n{"="*70}')
    print(f'  obj [{obj_idx}]  —  scene {config["NUSCENES_SCENE_IDX"]} / frame {config["FRAME_IDX"]}')
    print(f'{"="*70}')

    inst  = _select_instance(obj_idx, frame_cache, make_plots)
    filt  = _filter_lidar_obj(obj_idx, inst, frame_cache, config, make_plots)
    aff   = _fit_local_affine_obj(obj_idx, filt, frame_cache, config, make_plots)
    inf   = _run_inference_obj(obj_idx, inst, filt, aff, frame_cache, make_plots)
    var   = _run_var_obj(obj_idx, inst, filt, inf, frame_cache, config, make_plots)
    obb   = _run_obb_and_lidar_plots(obj_idx, inst, filt, inf, var, frame_cache, config, make_plots)

    return dict(
        obj_idx           = obj_idx,
        n_valid_obj       = filt['n_valid_obj'],
        Z_lidar_front_obj = filt['Z_lidar_front_obj'],
        local_fit_ok      = aff['local_fit_ok'],
        a_local           = aff['a_local'],
        b_local           = aff['b_local'],
        r2_l              = aff['r2_l'],
        m_g               = inf['m_g'],
        m_l               = inf['m_l'],
        m_var1            = var['m_var1'],
        m_var2            = var['m_var2'],
        m_var3            = var['m_var3'],
        m_var4            = var['m_var4'],
        info_var1         = var['info_var1'],
        info_var2         = var['info_var2'],
        info_var3         = var['info_var3'],
        info_var4         = var['info_var4'],
        dist_raw          = var['dist_raw'],
        dist_var1         = var['dist_var1'],
        dist_var2         = var['dist_var2'],
        dist_var3         = var['dist_var3'],
        dist_var4         = var['dist_var4'],
        obb_dims          = obb['obb_dims'],
        obb_yaw           = obb['obb_yaw'],
        ground_z          = obb['ground_z'],
        gnd_src           = obb['gnd_src'],
    )

print('process_object defined.')


## §1. Load nuScenes — calibration, camera frame, nearest LiDAR sweep

Loads all calibration matrices (camera intrinsics, camera<->ego, ego<->LiDAR),
reads the camera image, and finds the temporally closest LiDAR sweep.


In [ ]:
from nuscenes.nuscenes import NuScenes
from pyquaternion import Quaternion

print('Loading nuScenes...')
nusc  = NuScenes(version='v1.0-trainval', dataroot=DATA_ROOT, verbose=False)
scene = nusc.scene[NUSCENES_SCENE_IDX]
print(f'  Scene: {scene["name"]}')

# Walk CAM_FRONT frames
cam_token = nusc.get('sample', scene['first_sample_token'])['data']['CAM_FRONT']
frame_records = []
while cam_token:
    rec = nusc.get('sample_data', cam_token)
    frame_records.append(rec)
    cam_token = rec['next']
print(f'  {len(frame_records)} CAM_FRONT frames')

rec_cam  = frame_records[FRAME_IDX]
img_path = nusc.get_sample_data_path(rec_cam['token'])

cal_cam   = nusc.get('calibrated_sensor', rec_cam['calibrated_sensor_token'])
K_np      = np.array(cal_cam['camera_intrinsic'], dtype=np.float64)   # (3,3)
fx, fy    = K_np[0,0], K_np[1,1]
cx, cy    = K_np[0,2], K_np[1,2]
R_cam2ego = Quaternion(cal_cam['rotation']).rotation_matrix            # (3,3)
t_cam2ego = np.array(cal_cam['translation'], dtype=np.float64)         # (3,)
R_ego2cam = R_cam2ego.T
t_ego2cam = -R_ego2cam @ t_cam2ego

print(f'  ECP intrinsics: fx={fx:.1f}  fy={fy:.1f}  cx={cx:.1f}  cy={cy:.1f}')
print(f'  Image: {img_path}')

In [ ]:
# Find LiDAR sweep nearest in time to this camera frame
lt = nusc.get('sample', scene['first_sample_token'])['data']['LIDAR_TOP']
lidar_records = []
while lt:
    lr = nusc.get('sample_data', lt)
    lidar_records.append(lr)
    lt = lr['next']

lidar_ts  = np.array([lr['timestamp'] for lr in lidar_records], dtype=np.int64)
best_idx  = int(np.argmin(np.abs(lidar_ts - rec_cam['timestamp'])))
rec_lidar = lidar_records[best_idx]
dt_ms     = abs(rec_lidar['timestamp'] - rec_cam['timestamp']) / 1000.0

lidar_path = nusc.get_sample_data_path(rec_lidar['token'])
cal_lidar  = nusc.get('calibrated_sensor', rec_lidar['calibrated_sensor_token'])
R_l2e      = Quaternion(cal_lidar['rotation']).rotation_matrix     # (3,3) lidar→ego
t_l2e      = np.array(cal_lidar['translation'], dtype=np.float64)  # (3,)

print(f'  LiDAR sweep: {lidar_path}')
print(f'  Time offset to camera: {dt_ms:.1f} ms')

## §2. Transform LiDAR to camera space and project to image

Transforms the full LiDAR sweep through ego space into camera (R3/OpenCV) space,
then applies pinhole projection to get pixel coordinates for each point.


In [ ]:
# Load LiDAR point cloud (.bin = float32, columns: x y z intensity ring)
pc_raw    = np.fromfile(lidar_path, dtype=np.float32).reshape(-1, 5)
pts_lidar = pc_raw[:, :3].T   # (3, N) in LiDAR frame
print(f'  Loaded {pts_lidar.shape[1]:,} LiDAR points')

# lidar → ego → camera (R3/OpenCV: X right, Y down, Z forward)
pts_ego = R_l2e @ pts_lidar + t_l2e[:, np.newaxis]
pts_cam = R_ego2cam @ pts_ego + t_ego2cam[:, np.newaxis]

# Keep only points in front of camera
in_front = pts_cam[2] > 0.5
pts_cam  = pts_cam[:, in_front]
print(f'  {pts_cam.shape[1]:,} points in front of camera')

# Pinhole projection
Z_all = pts_cam[2]
u_all = fx * pts_cam[0] / Z_all + cx
v_all = fy * pts_cam[1] / Z_all + cy

img_bgr = cv2.imread(img_path)
H, W    = img_bgr.shape[:2]
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# Keep only points that project within image bounds
in_img      = (u_all >= 0) & (u_all < W) & (v_all >= 0) & (v_all < H)
u_vis       = u_all[in_img]
v_vis       = v_all[in_img]
Z_vis       = Z_all[in_img]
pts_cam_vis = pts_cam[:, in_img]
print(f'  {pts_cam_vis.shape[1]:,} points project into image')
xyz_lid = pts_lidar.T.astype(np.float64)  # (N,3) LiDAR frame

# -- Frame-level LiDAR pixel snap (shared across all objects) -----------------
u_int     = np.round(u_vis).astype(int)
v_int     = np.round(v_vis).astype(int)
in_bounds = (u_int >= 0) & (u_int < W) & (v_int >= 0) & (v_int < H)
u_int     = u_int[in_bounds]
v_int     = v_int[in_bounds]
Z_v       = Z_vis[in_bounds]
u_f_v     = u_vis[in_bounds]
v_f_v     = v_vis[in_bounds]
# Pixel coordinate grids (reused per object in fused pointmap)
u_grid, v_grid = np.meshgrid(np.arange(W), np.arange(H))


In [ ]:
# Visualize all projected LiDAR points — colored by depth
fig, ax = plt.subplots(1, 1, figsize=(16, 8))
ax.imshow(img_rgb)
sc = ax.scatter(u_vis, v_vis, c=Z_vis, cmap='plasma_r', s=2, alpha=0.7, vmin=0, vmax=50)
plt.colorbar(sc, ax=ax, label='depth (m)')
ax.set_title(f'All LiDAR projected to image — frame {FRAME_IDX}')
ax.axis('off')
plt.tight_layout()
plt.show()

## §3. Load SAM3 masks -- overview and instance selection

Loads pre-computed SAM3 segmentation masks for this frame.
Set `OBJ_IDX` in the config cell to the instance you want to reconstruct.


In [ ]:
# Load SAM3 masks for this frame (class: OBJ_CLASS)
mask_path = SAM3_MASK_DIR / f'{SAM3_SCENE_PREFIX}{OBJ_CLASS}.npy'
sam3_data = np.load(str(mask_path), allow_pickle=True).item()

frame      = sam3_data[FRAME_IDX]
obj_ids    = frame['out_obj_ids']
probs      = np.asarray(frame['out_probs'],       dtype=np.float32)
raw_masks  = frame['out_binary_masks']
boxes_xywh = np.asarray(frame['out_boxes_xywh'],  dtype=np.float32)

# Collect all valid detections
detections = []
for i, (oid, prob) in enumerate(zip(obj_ids, probs)):
    if float(prob) < MIN_PROB:
        continue
    bm = np.asarray(raw_masks[i], dtype=bool)
    if bm.sum() < MIN_MASK_PX:
        continue
    bx = boxes_xywh[i]
    detections.append({
        'obj_id': int(oid), 'prob': float(prob),
        'binary_mask': bm,
        'box_xyxy': [float(bx[0])*W, float(bx[1])*H,
                     (float(bx[0])+float(bx[2]))*W,
                     (float(bx[1])+float(bx[3]))*H],
    })

print(f'{len(detections)} {OBJ_CLASS} detection(s) above threshold in frame {FRAME_IDX}:')
for i, d in enumerate(detections):
    n_px = d['binary_mask'].sum()
    b    = d['box_xyxy']
    print(f'  [{i}] obj_id={d["obj_id"]}  prob={d["prob"]:.3f}  '
          f'mask_px={n_px}  bbox=[{b[0]:.0f},{b[1]:.0f},{b[2]:.0f},{b[3]:.0f}]')

In [ ]:
# ── Overview: all detected cars on a single image, each with index label ──────
_COLORS = [
    (0, 230, 0),    # green
    (255, 80, 0),   # orange
    (0, 180, 255),  # blue
    (255, 0, 200),  # magenta
    (255, 230, 0),  # yellow
    (0, 255, 200),  # teal
    (180, 0, 255),  # purple
    (255, 255, 255),# white
]

overview = img_rgb.copy().astype(np.float32)
for idx, det in enumerate(detections):
    color = _COLORS[idx % len(_COLORS)]
    mask  = det['binary_mask']
    fill  = np.zeros_like(overview)
    fill[mask] = color
    overview = overview * 0.65 + fill * 0.35

overview = overview.clip(0, 255).astype(np.uint8)

fig, ax = plt.subplots(1, 1, figsize=(16, 8))
ax.imshow(overview)

for idx, det in enumerate(detections):
    color_f = tuple(c/255 for c in _COLORS[idx % len(_COLORS)])
    x1, y1, x2, y2 = det['box_xyxy']
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                              lw=2, edgecolor=color_f, facecolor='none')
    ax.add_patch(rect)
    ax.text(x1+4, y1+20, f'[{idx}]',
            color=color_f, fontsize=14, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.6))
    ax.text(x1+4, y1+42, f'p={det["prob"]:.2f}',
            color=color_f, fontsize=9,
            bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.5))

ax.set_title(
    f'All detected {OBJ_CLASS}s — frame {FRAME_IDX}  ({len(detections)} instances)\n'
    f'→ Set OBJ_IDXS in the config cell to the index(es) you want to analyse',
    fontsize=12
)
ax.axis('off')
plt.tight_layout()
plt.show()

## §6. MoGe monocular depth inference

Runs MoGe on the full camera image to produce a dense relative-depth pointmap `Z_moge_map (H, W)`. These depths are in non-metric MoGe units and must be calibrated with in-mask LiDAR measurements before SAM3D can use them.

MoGe also estimates camera intrinsics `K_moge` which we use as the projection matrix for the fused pointmap.


In [ ]:
rgb = img_bgr[:, :, ::-1].copy()
rgba_full = np.concatenate([rgb, np.full((H, W, 1), 255, dtype=np.uint8)], axis=-1)

with inference_model._pipeline.device:
    ptmap_dict = inference_model._pipeline.compute_pointmap(rgba_full)

norm_K  = ptmap_dict['intrinsics'].cpu().numpy()
K_moge  = np.array([[norm_K[0,0]*W, 0., norm_K[0,2]*W],
                     [0., norm_K[1,1]*H, norm_K[1,2]*H],
                     [0., 0., 1.]], dtype=np.float64)
# moge_ptmap_p3d: (H,W,3) in P3D space — Z channel == R3 depth (identical under the transform)
moge_ptmap_p3d = ptmap_dict['pointmap'].cpu().permute(1, 2, 0)
Z_moge_map     = moge_ptmap_p3d[:, :, 2].numpy()   # (H,W) non-metric depth

print(f'MoGe K: fx={K_moge[0,0]:.1f}  fy={K_moge[1,1]:.1f}  cx={K_moge[0,2]:.1f}  cy={K_moge[1,2]:.1f}')
print(f'GT   K: fx={fx:.1f}  fy={fy:.1f}  cx={cx:.1f}  cy={cy:.1f}')
print(f'Z_moge range: [{np.nanmin(Z_moge_map):.4f}, {np.nanmax(Z_moge_map):.4f}]')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
im = axes[0].imshow(Z_moge_map, cmap='plasma_r', interpolation='nearest')
plt.colorbar(im, ax=axes[0], fraction=0.03, label='MoGe Z (normalised)')
axes[0].set_title(f'MoGe depth map (full frame)\n'
                  f'Z∈[{np.nanmin(Z_moge_map):.3f}, {np.nanmax(Z_moge_map):.3f}]')
axes[0].axis('off')

axes[1].imshow(img_rgb)
axes[1].set_title(f'Camera image — frame {FRAME_IDX}\n'
                  f'({W}×{H} px)')
axes[1].axis('off')

plt.suptitle(f'MoGe inference — scene {NUSCENES_SCENE_IDX} / frame {FRAME_IDX}', fontsize=12)
plt.tight_layout(); plt.show()


## §7. Affine depth calibration: Z_lidar = a * Z_moge + b

We fit a linear mapping from MoGe units to metric LiDAR depth using two strategies:

| Strategy | Fit data | Strength |
|----------|----------|----------|
| **Global** | All valid in-image LiDAR points | Stable, scene-wide |
| **Local + MAD** | Depth-window-filtered in-mask cluster, outliers removed | Best per-object accuracy |

The local fit uses only the `n_valid_obj` points that survived both the depth-window filter and the MAD outlier rejection. When fewer than 3 clean inliers remain, the pipeline falls back gracefully to the global fit.


In [ ]:
# ── Global: sample Z_moge at ALL in-image LiDAR pixel locations ───────────────
Z_moge_all = Z_moge_map[v_int, u_int]   # (N_in_image,)
valid_global = np.isfinite(Z_moge_all) & (Z_v > 0.5) & (Z_v < 80.0)
Z_moge_g = Z_moge_all[valid_global]
Z_lidar_g = Z_v[valid_global]
n_fit_g   = int(valid_global.sum())

A_g = np.stack([Z_moge_g, np.ones_like(Z_moge_g)], axis=1)
(a_global, b_global), _, _, _ = np.linalg.lstsq(A_g, Z_lidar_g, rcond=None)

Z_pred_g = a_global * Z_moge_g + b_global
ss_res_g  = float(np.sum((Z_lidar_g - Z_pred_g)**2))
ss_tot_g  = float(np.sum((Z_lidar_g - Z_lidar_g.mean())**2))
r2_g      = 1.0 - ss_res_g / ss_tot_g
rmse_g    = float(np.sqrt(ss_res_g / n_fit_g))
rel_err_g = (Z_pred_g - Z_lidar_g) / Z_lidar_g

print(f'Global fit  (n={n_fit_g:,}):  Z = {a_global:.4f} × Z_moge + {b_global:.4f}')
print(f'  R²={r2_g:.4f}  RMSE={rmse_g:.3f} m  median|rel_err|={np.median(np.abs(rel_err_g))*100:.1f}%')

In [ ]:
# ── Build frame_cache: all data shared across objects ────────────────────────
frame_cache = dict(
    img_rgb    = img_rgb,   img_bgr = img_bgr,
    H          = H,         W       = W,
    K_np       = K_np,      fx=fx,  fy=fy,  cx=cx,  cy=cy,
    R_cam2ego  = R_cam2ego, t_cam2ego = t_cam2ego,
    R_ego2cam  = R_ego2cam, t_ego2cam = t_ego2cam,
    R_l2e      = R_l2e,     t_l2e     = t_l2e,
    xyz_lid    = xyz_lid,
    u_int      = u_int,     v_int   = v_int,
    Z_v        = Z_v,       u_f_v   = u_f_v,  v_f_v = v_f_v,
    u_grid     = u_grid,    v_grid  = v_grid,
    detections = detections, n_detections = len(detections),
    Z_moge_map = Z_moge_map, rgb    = rgb,
    a_global   = a_global,  b_global = b_global,
    r2_g       = r2_g,      rmse_g   = rmse_g,
    Z_moge_g   = Z_moge_g,  Z_lidar_g = Z_lidar_g,  n_fit_g = n_fit_g,
)
print(f'frame_cache built — {len(detections)} detection(s) in frame {FRAME_IDX}')


## Run: step-by-step multi-object execution
Run each cell below in order. Each stage processes all objects in `OBJ_IDXS`.
Plots appear after each stage so you can inspect results before continuing.
To run everything in one go, use `process_object` directly (see end of cell).

In [ ]:
# ── Object list setup ────────────────────────────────────────────────────────
n_det    = frame_cache['n_detections']
obj_list = list(range(n_det)) if OBJ_IDXS == 'all' else [
    i for i in OBJ_IDXS if i < n_det
]
print(f'Processing {len(obj_list)} object(s): {obj_list}  '
      f'({n_det} detection(s) in frame {FRAME_IDX})')
# State dicts — populated by each stage cell below
_inst = {}; _filt = {}; _aff = {}; _inf = {}; _var = {}; _obb = {}


### §3–4.  Instance selection + LiDAR depth-window filter

In [90]:
# ── §3–4: instance select + depth-window LiDAR filter ────────────────────────
for _oi in obj_list:
    print(f'\n{"─"*55}  obj [{_oi}]')
    _inst[_oi] = _select_instance(_oi, frame_cache, make_plots=True)
    _filt[_oi] = _filter_lidar_obj(_oi, _inst[_oi], frame_cache, config, make_plots=True)


### §7.  Local affine calibration (MAD-robust)

In [ ]:
# ── §7: local affine + MAD calibration ───────────────────────────────────────
for _oi in obj_list:
    print(f'\n{"─"*55}  obj [{_oi}]')
    _aff[_oi] = _fit_local_affine_obj(_oi, _filt[_oi], frame_cache, config, make_plots=True)


### §8–9.  SAM3D inference (global + local pointmap)

In [ ]:
# ── §8–9: fused pointmaps + SAM3D inference + raw meshes ────────────────────
for _oi in obj_list:
    print(f'\n{"─"*55}  obj [{_oi}]')
    _inf[_oi] = _run_inference_obj(_oi, _inst[_oi], _filt[_oi], _aff[_oi],
                                    frame_cache, config, make_plots=True)


### §10–12.  Visibility-Aware Refinement (VAR) + comparison

In [ ]:
# ── §10–12: VAR1/2/3 + strategy comparison + metrics table ──────────────────
for _oi in obj_list:
    print(f'\n{"─"*55}  obj [{_oi}]')
    _var[_oi] = _run_var_obj(_oi, _inst[_oi], _filt[_oi], _inf[_oi],
                              frame_cache, config, make_plots=True)


### §13–14.  OBB + ground anchoring + LiDAR 3D / BEV

In [ ]:
# ── §13–14: RANSAC ground + OBB + LiDAR 3D/BEV plots ───────────────────────
for _oi in obj_list:
    print(f'\n{"─"*55}  obj [{_oi}]')
    _obb[_oi] = _run_obb_and_lidar_plots(_oi, _inst[_oi], _filt[_oi], _inf[_oi],
                                          _var[_oi], frame_cache, config, make_plots=True)


### Summary

In [ ]:
# ── Cross-object summary table ────────────────────────────────────────────────
prior_lwh = np.array([config['OBJ_PRIOR_DIMS'][2],
                      config['OBJ_PRIOR_DIMS'][0],
                      config['OBJ_PRIOR_DIMS'][1]])
hdr = (f'{"Obj":>4}  {"nLiDAR":>7}  {"IoU_v2":>7}  {"Cov_v2":>7}  '
       f'{"ZfErr_v2(m)":>12}  {"DimErr_v2(m)":>13}  {"Chamfer_v2(m)":>14}  '
       f'{"OBB L×W×H (m)":>16}  {"ground_z(m)":>12}')
print('='*110)
print(f'Summary — scene {config["NUSCENES_SCENE_IDX"]} / frame {config["FRAME_IDX"]}')
print('='*110); print(hdr); print('-'*110)
for _oi in sorted(_obb):
    m2   = _var[_oi]['m_var2']
    dims = _obb[_oi]['obb_dims']
    zlf  = _filt[_oi]['Z_lidar_front_obj']
    ze   = m2['Z_front'] - zlf if zlf is not None else float('nan')
    de   = float(np.abs(m2['dims'] - prior_lwh).mean()) if 'dims' in m2 else float('nan')
    dist = _var[_oi]['dist_var2']
    dist_s = f'{dist:.4f}' if np.isfinite(dist) else '  --  '
    de_s   = f'{de:.4f}'   if np.isfinite(de)   else '  --  '
    ze_s   = f'{ze:+.3f}'  if np.isfinite(ze)   else '  --  '
    print(f'{_oi:>4}  {_filt[_oi]["n_valid_obj"]:>7}  {m2["iou"]:>7.3f}  '
          f'{m2["coverage"]:>7.3f}  {ze_s:>12}  {de_s:>13}  {dist_s:>14}  '
          f'{dims[0]:.2f}×{dims[1]:.2f}×{dims[2]:.2f}{"":>5}  '
          f'{_obb[_oi]["ground_z"]:>+12.3f}')
print('='*110)


In [ ]:
# ── Combined Plotly: all VAR2 OBBs in one 3-D view ──────────────────────────
import plotly.graph_objects as go

_OBJ_COLS = ['rgb(50,220,80)','rgb(255,100,0)','rgb(30,144,255)',
             'rgb(220,50,220)','rgb(255,210,0)','rgb(0,220,220)',
             'rgb(255,80,150)','rgb(80,255,210)']
_EDGES = [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7)]

def _wire(cnrs, dims, lbl, col):
    xs,ys,zs=[],[],[]
    for a,b in _EDGES:
        xs+=[cnrs[a,0],cnrs[b,0],None]; ys+=[cnrs[a,1],cnrs[b,1],None]; zs+=[cnrs[a,2],cnrs[b,2],None]
    return go.Scatter3d(x=xs,y=ys,z=zs,mode='lines',line=dict(color=col,width=5),
                        name=f'obj[{lbl}]  {dims[0]:.2f}×{dims[1]:.2f}×{dims[2]:.2f}m')

def _fwd_arrow(cnrs, axes_mat, dims, col, lbl):
    R_l2e_=frame_cache['R_l2e']
    ctr=cnrs.mean(0); fwd=(R_l2e_.T@axes_mat[:,0]); tip=ctr+fwd*(dims[0]/2+0.5)
    return go.Scatter3d(x=[ctr[0],tip[0]],y=[ctr[1],tip[1]],z=[ctr[2],tip[2]],
                        mode='lines+markers',line=dict(color=col,width=4),
                        marker=dict(size=[2,8],color=col,symbol=['circle','diamond']),
                        name=f'obj[{lbl}] fwd', showlegend=False)

xyz_lid = frame_cache['xyz_lid']
tr = [go.Scatter3d(x=xyz_lid[:,0],y=xyz_lid[:,1],z=xyz_lid[:,2],mode='markers',
                   marker=dict(size=1,color=xyz_lid[:,2],colorscale='Turbo',
               opacity=.3,showscale=False),
                   name='LiDAR',hoverinfo='skip')]

all_cnrs = []
for _oi, col in zip(sorted(_obb.keys()), _OBJ_COLS):
    ob = _obb[_oi]
    cnrs = ob['cl_v2']; dims = ob['obb_dims_v2']; axes = ob['obb_axes_v2']
    tr.append(_wire(cnrs, dims, _oi, col))
    tr.append(_fwd_arrow(cnrs, axes, dims, col, _oi))
    if ob['n_valid_obj'] > 0:
        p = ob['pts_obj_lidar']
        tr.append(go.Scatter3d(x=p[:,0],y=p[:,1],z=p[:,2],mode='markers',
                               marker=dict(size=3,color=col,opacity=.7),
                               name=f'obj[{_oi}] LiDAR', showlegend=False))
    all_cnrs.append(cnrs)

_ac = np.vstack(all_cnrs)
ctr = _ac.mean(0)
r   = max(float(np.abs(_ac - ctr).max()) * 1.5, 8.)

fig_all = go.Figure(data=tr)
fig_all.update_layout(
    title=f'All VAR2 OBBs — scene {config["NUSCENES_SCENE_IDX"]} / frame {config["FRAME_IDX"]}  '
          f'({len(_obb)} object(s))',
    scene=dict(xaxis=dict(title='X (m)', range=[ctr[0]-r, ctr[0]+r]),
               yaxis=dict(title='Y (m)', range=[ctr[1]-r, ctr[1]+r]),
               zaxis=dict(title='Z (m)', range=[ctr[2]-r, ctr[2]+r]),
               aspectmode='cube',
               camera=dict(eye=dict(x=0, y=-.5, z=1.8))),
    margin=dict(l=0,r=0,b=0,t=60), height=700)
fig_all.show()


In [ ]:
# ── Combined BEV: all VAR2 OBBs over shared LiDAR sweep ─────────────────────
_OBJ_COLS_BEV = ['limegreen','orangered','dodgerblue',
                 'mediumpurple','gold','cyan','hotpink','turquoise']

xyz_lid = frame_cache['xyz_lid']
R_l2e_  = frame_cache['R_l2e']

fig_bev_all, ax_bev_all = plt.subplots(figsize=(10, 10))
ax_bev_all.scatter(xyz_lid[:,0], xyz_lid[:,1],
                   c=xyz_lid[:,2], cmap='Greys', s=1, alpha=.2, zorder=1)

def _bev_box(ax, cnrs, col, lw, lbl):
    zs  = cnrs[:,2]
    bot = cnrs[:4] if zs[:4].mean() <= zs[4:].mean() else cnrs[4:]
    xs  = list(bot[:,0]) + [bot[0,0]]
    ys  = list(bot[:,1]) + [bot[0,1]]
    ax.plot(xs, ys, color=col, lw=lw, label=lbl, zorder=3)
    ax.scatter([cnrs[:,0].mean()], [cnrs[:,1].mean()], color=col, s=50, zorder=5)

def _bev_arrow(ax, cnrs, axes_mat, dims, col):
    ctr  = cnrs.mean(0)[:2]
    fwd  = (R_l2e_.T @ axes_mat[:,0])[:2]
    tip  = ctr + fwd * (dims[0]/2 + 0.5)
    ax.annotate('', xy=tip, xytext=ctr,
                arrowprops=dict(arrowstyle='->', color=col, lw=2))

all_cnrs_bev = []
for _oi, col in zip(sorted(_obb.keys()), _OBJ_COLS_BEV):
    ob   = _obb[_oi]
    cnrs = ob['cl_v2']; dims = ob['obb_dims_v2']; axes = ob['obb_axes_v2']
    _bev_box(ax_bev_all, cnrs, col, 2.5,
             f'obj[{_oi}] {dims[0]:.2f}×{dims[1]:.2f}×{dims[2]:.2f}m  '
             f'gz={ob["ground_z"]:+.2f}m')
    _bev_arrow(ax_bev_all, cnrs, axes, dims, col)
    if ob['n_valid_obj'] > 0:
        p = ob['pts_obj_lidar']
        ax_bev_all.scatter(p[:,0], p[:,1], c=col, s=12, alpha=.7, zorder=4)
    all_cnrs_bev.append(cnrs)

_ac2 = np.vstack(all_cnrs_bev)
ctr2 = _ac2.mean(0)
r2   = max(float(np.abs(_ac2 - ctr2).max()) * 1.8, 10.)

ax_bev_all.set_xlim(ctr2[0]-r2, ctr2[0]+r2)
ax_bev_all.set_ylim(ctr2[1]-r2, ctr2[1]+r2)
ax_bev_all.set_aspect('equal')
ax_bev_all.grid(alpha=.35)
ax_bev_all.set_xlabel('LiDAR X (m)')
ax_bev_all.set_ylabel('LiDAR Y (m)')
ax_bev_all.set_title(f'BEV — All VAR2 OBBs\n'
                     f'Scene {config["NUSCENES_SCENE_IDX"]} / Frame {config["FRAME_IDX"]}  '
                     f'({len(_obb)} object(s))')
ax_bev_all.legend(loc='upper right', fontsize=8,
                  facecolor='k', labelcolor='w', framealpha=.8)
plt.tight_layout()
plt.show()


---
## Appendix: Legacy hybrid post-hoc anchor (optional baseline)

The original pipeline used a post-hoc anchor that applies:
1. A **scale correction** derived from visible-face radius matching (v3)
2. A **translation correction** from near-surface LiDAR alignment (v2)

This is kept here as a reference baseline. It is **not the recommended output** --
VAR2 supersedes it by using proper z-buffer visibility and tighter constraints.

Run the cells below **after** §9 (SAM3D inference) if you want to include the
legacy hybrid in the comparison.


### Appendix A.1: Smart post-hoc correction (legacy)

Apply hybrid anchor (v3 scale + v2 translation) only if it improves metrics.
This is the original pipeline approach, superseded by VAR2.


### Appendix A.2: MoGe-only baseline

Run SAM3D with the raw MoGe pointmap (no LiDAR calibration) as a baseline.
